# 04 ABLATIONS AND AUDITS

Extracted from the original notebooks. **Outputs preserved.** Originals are unmodified.


## A · Pooling / mask ablations


**`IL2` cell 9** — ══════════════════════════════════════════════════════════════════════  
<sub>4 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════
#  CELL P1 — POOLING ABLATION  (Novelty 1)   ***REAL RUN***
#
#  One class, one switch. Identical crops, masks, folds, schedule, seeds
#  and augmentation across every variant — the pooling rule is the ONLY
#  thing that changes.
#
#  VARIANTS
#    dual       [f_g || f_u]           2048-d   <- YOUR contribution
#    weighted   f_g only               1024-d   <- prior-work formulation
#    global2x   [f_u || f_u]           2048-d   <- capacity-matched control
#    global     f_u only               1024-d   <- mask-blind baseline
#    gated      hard mask, then pool   1024-d   <- discards context
#    dual_l1    dual, lambda = 1       2048-d   <- lambda sensitivity
#    dual_l4    dual, lambda = 4       2048-d   <- lambda sensitivity
#
#  PROTOCOL: patient-grouped 5-fold CV  (NOT the official split)
#  RESUMABLE at (variant, fold, seed) granularity. Safe to interrupt.
# ══════════════════════════════════════════════════════════════════════
import os, gc, time
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score
cv2.setNumThreads(0)

D, DEV = "/root/autodl-tmp/CBIS", torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

QUICK_TEST = False                     # <<< REAL RUN

# --- identical to the published classifier configuration ---------------
S, BATCH = 512, 12
SEEDS, EPOCHS, FREEZE = [11, 22], 22, 3
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 3e-5
WD, GAMMA, AUX_W, MULT, PATIENCE = 1e-4, 2.0, 0.3, 4, 7
FOLDS = [0, 1, 2, 3, 4]

VARIANTS = [("dual", 2.0), ("weighted", 2.0), ("global2x", 0.0), ("global", 0.0),
            ("gated", 0.0), ("dual_l1", 1.0), ("dual_l4", 4.0)]

if QUICK_TEST:
    SEEDS, EPOCHS, FREEZE, MULT, FOLDS = [11], 4, 1, 1, [0]
    VARIANTS = [("dual", 2.0), ("weighted", 2.0)]
    print(">>> QUICK TEST: 1 fold, 1 seed, 4 epochs, 2 variants\n")

WIDE2 = {"dual", "global2x", "dual_l1", "dual_l4"}     # 2048-d descriptor

CKPT = os.path.join(D, "_poolabl_cache")               # resume cache
os.makedirs(CKPT, exist_ok=True)

# ══════════════════════════ data ══════════════════════════
d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv")).reset_index(drop=True)
d["label"] = d["label"].astype(int)
PM = os.path.join(D, "predmasks_mass")
d["predmask"] = d["img"].apply(lambda p: os.path.join(
    PM, os.path.basename(str(p)).replace("_img.png", "") + "_pred.png"))
assert d["predmask"].apply(os.path.exists).all(), "predmasks_mass incomplete"

# --- resolve the role-column naming actually present in the CSV --------
def role_col(k):
    for pat in (f"role_f{k}", f"role_of{k}", f"role_fold{k}", f"role{k}"):
        if pat in d.columns:
            return pat
    raise KeyError(f"no role column for fold {k}; columns are {list(d.columns)}")
ROLE = {k: role_col(k) for k in FOLDS}
print("role columns:", {k: ROLE[k] for k in FOLDS})

print(f"{len(d)} regions | {d.patient_id.nunique()} patients | "
      f"{d.lesion_key.nunique()} lesions | malignant {100*d.label.mean():.1f}%")

_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
CACHE, t0 = {}, time.time()
for _, r in d.iterrows():
    k = str(r["img"])
    if k in CACHE: continue
    im = cv2.imread(k, cv2.IMREAD_GRAYSCALE)
    im = np.zeros((S, S), np.uint8) if im is None else (
        cv2.resize(im, (S, S)) if im.shape != (S, S) else im)
    pm = cv2.imread(str(r["predmask"]), cv2.IMREAD_GRAYSCALE)
    pm = (np.zeros((S, S), np.uint8) if pm is None
          else (cv2.resize(pm, (S, S), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8))
    CACHE[k] = (_clahe.apply(im), pm)
print(f"cached {len(CACHE)} in {time.time()-t0:.0f}s")

def primary(x):
    return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()

aux, meta = {}, {}
for c in ["subtlety", "mass_shape", "mass_margins"]:
    if c not in d.columns or d[c].notna().sum() == 0: continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z: (z >= 1) & (z <= 5))
        codes, n = (v - 1).fillna(-1).astype(int).values, 5
    else:
        pr = d[c].map(primary)
        pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([k for k in pr.unique() if k != "UNK"])
        mp = {k: i for i, k in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z, -1)).astype(int).values, len(cats)
    if n > 1: aux[c], meta[c] = codes, n
AK = sorted(aux)
print("helper heads:", meta)

MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)

class DS(Dataset):
    def __init__(s, idx, augment, mult=1, tta=0):
        s.idx = np.asarray(idx); s.aug = augment
        s.mult = mult if augment else 1; s.tta = tta
    def __len__(s): return len(s.idx) * s.mult
    def __getitem__(s, i):
        j = int(s.idx[i % len(s.idx)])
        img, msk = [a.copy() for a in CACHE[str(d.iloc[j]["img"])]]
        if s.aug:
            if np.random.rand() < .5: img, msk = img[:, ::-1], msk[:, ::-1]
            if np.random.rand() < .5: img, msk = img[::-1, :], msk[::-1, :]
            k = np.random.randint(4)
            if k: img, msk = np.rot90(img, k), np.rot90(msk, k)
            img, msk = np.ascontiguousarray(img), np.ascontiguousarray(msk)
            if np.random.rand() < .7:
                M = cv2.getRotationMatrix2D((S/2, S/2), np.random.uniform(-25, 25),
                                            np.random.uniform(.90, 1.12))
                img = cv2.warpAffine(img, M, (S, S), flags=cv2.INTER_LINEAR,
                                     borderMode=cv2.BORDER_REFLECT)
                msk = cv2.warpAffine(msk, M, (S, S), flags=cv2.INTER_NEAREST,
                                     borderMode=cv2.BORDER_CONSTANT)
            if np.random.rand() < .5:
                img = np.clip(img.astype(np.float32) * np.random.uniform(.85, 1.15)
                              + np.random.uniform(-12, 12), 0, 255).astype(np.uint8)
        else:
            t = s.tta
            if   t == 1: img, msk = img[:, ::-1], msk[:, ::-1]
            elif t == 2: img, msk = img[::-1, :], msk[::-1, :]
            elif t == 3: img, msk = np.rot90(img, 2), np.rot90(msk, 2)
        img, msk = np.ascontiguousarray(img), np.ascontiguousarray(msk)
        g = img.astype(np.float32) / 255.
        x = ((np.stack([g, g, g], 0) - MEAN) / STD).astype(np.float32)
        av = (np.array([aux[c][j] for c in AK], dtype=np.int64) if AK else np.zeros(0, np.int64))
        return (torch.from_numpy(x), torch.from_numpy(msk.astype(np.float32))[None],
                torch.tensor(int(d.iloc[j]["label"])), torch.from_numpy(av))

# ══════════════════════ model: one switch ══════════════════════
class PoolNet(nn.Module):
    """DenseNet-121 backbone. The pooling rule is the only variable."""
    def __init__(s, aux_meta, mode, lam):
        super().__init__()
        try:
            s.b = models.densenet121(
                weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
        except Exception:
            s.b = models.densenet121(weights=None).features
            print("  (ImageNet weights unavailable)")
        s.mode, s.lam = mode, float(lam)
        dim = 2048 if mode in WIDE2 else 1024
        s.head = nn.Sequential(nn.Linear(dim, 512), nn.BatchNorm1d(512), nn.ReLU(True),
                               nn.Dropout(0.4), nn.Linear(512, 2))
        s.keys = sorted(aux_meta)
        s.aux = nn.ModuleList([nn.Sequential(nn.Linear(dim, 128), nn.ReLU(True),
                                             nn.Dropout(0.3), nn.Linear(128, aux_meta[k]))
                               for k in s.keys])

    def forward(s, x, mask):
        f = F.relu(s.b(x))
        m = F.interpolate(mask, size=f.shape[2:], mode="bilinear", align_corners=False)
        fu = f.mean((2, 3))
        if s.mode == "global":
            g = fu
        elif s.mode == "global2x":
            g = torch.cat([fu, fu], 1)
        elif s.mode == "gated":
            hm = (m > 0.5).float()
            den = hm.sum((2, 3))
            g = torch.where(den > 0, (f * hm).sum((2, 3)) / (den + 1e-6), fu)
        else:                                     # weighted, dual, dual_l1, dual_l4
            w = 1.0 + s.lam * m
            fg = (f * w).sum((2, 3)) / (w.sum((2, 3)) + 1e-6)
            g = fg if s.mode == "weighted" else torch.cat([fg, fu], 1)
        return s.head(g), [h(g) for h in s.aux]

def focal(lo, t, alpha):
    ce = F.cross_entropy(lo.float(), t, weight=alpha, reduction="none")
    return ((1 - torch.exp(-ce)) ** GAMMA * ce).mean()

@torch.no_grad()
def predict(net, idx, tta=True):
    net.eval(); tot = None
    for t in ([0, 1, 2, 3] if tta else [0]):
        ps = []
        for x, m, _, _ in DataLoader(DS(idx, False, 1, tta=t), batch_size=20,
                                     shuffle=False, num_workers=0):
            x = x.to(DEV).to(memory_format=torch.channels_last); m = m.to(DEV)
            with torch.amp.autocast(device_type="cuda"):
                o, _ = net(x, m)
            ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot + ps
    return tot / (4 if tta else 1)

def train_one(tr, va, te, seed, mode, lam):
    torch.manual_seed(seed); np.random.seed(seed)
    y = d["label"].values
    n0, n1 = float((y[tr] == 0).sum()), float((y[tr] == 1).sum())
    alpha = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)

    net = PoolNet(meta, mode, lam).to(DEV).to(memory_format=torch.channels_last)
    for p in net.b.parameters(): p.requires_grad = False
    hp = [p for n_, p in net.named_parameters() if not n_.startswith("b.")]
    scaler = torch.amp.GradScaler()
    opt, sch = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD), None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)

    best, bstate, bad = -1.0, None, 0
    for ep in range(1, EPOCHS + 1):
        if ep == FREEZE + 1:
            for p in net.b.parameters(): p.requires_grad = True
            opt = torch.optim.AdamW([{"params": net.b.parameters(), "lr": LR_BACK},
                                     {"params": hp, "lr": LR_HEAD_FT}], weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS-FREEZE))
        net.train()
        if ep <= FREEZE: net.b.eval()
        for x, m, t, a in tl:
            x = x.to(DEV, non_blocking=True).to(memory_format=torch.channels_last)
            m, t, a = m.to(DEV), t.to(DEV), a.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o, ax = net(x, m)
                loss = focal(o, t, alpha)
                if len(ax):
                    loss = loss + AUX_W * sum(
                        F.cross_entropy(g.float(), a[:, h], ignore_index=-1)
                        for h, g in enumerate(ax)) / len(ax)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        if sch: sch.step()
        pv = predict(net, va, tta=False)
        auc = roc_auc_score(y[va], pv) if len(set(y[va])) > 1 else 0.0
        star = ""
        if auc > best:
            best, bad, star = auc, 0, " *"
            bstate = {q: v.detach().cpu().clone() for q, v in net.state_dict().items()}
        else:
            bad += 1
        print(f"      ep {ep:2d}  val-AUC {auc:.4f}{star}")
        if bad >= PATIENCE:
            print("      early stop"); break
    net.load_state_dict({q: v.to(DEV) for q, v in bstate.items()})
    p = predict(net, te, tta=True)
    npar = sum(q.numel() for q in net.parameters())
    del net; gc.collect(); torch.cuda.empty_cache()
    return p, best, npar

# ══════════════════════════ run ══════════════════════════
y = d["label"].values
summary = []
T_ALL = time.time()

for mode, lam in VARIANTS:
    out = os.path.join(D, f"cv_mass_pool_{mode}_oof.csv")
    if os.path.exists(out) and not QUICK_TEST:
        r = pd.read_csv(out)
        L = r.merge(d[["img", "lesion_key"]], on="img").groupby("lesion_key").agg(
            y=("true", "max"), p=("prob", "mean"))
        print(f"[skip] {mode:<10} lesion AUC {roc_auc_score(L.y, L.p):.4f}")
        summary.append(dict(variant=mode, lesion_auc=roc_auc_score(L.y, L.p), params=np.nan))
        continue

    print(f"\n{'='*68}\nVARIANT: {mode}   (lambda={lam}, "
          f"{'2048' if mode in WIDE2 else '1024'}-d descriptor)\n{'='*68}")
    oof, npar, t0 = np.full(len(d), np.nan), np.nan, time.time()
    for k in FOLDS:
        role = d[ROLE[k]].astype(str).str.lower()
        tr, va, te = (np.where(role == "train")[0], np.where(role == "val")[0],
                      np.where(role == "test")[0])
        assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "LEAK"
        preds = []
        print(f"  fold {k}  train {len(tr)} val {len(va)} test {len(te)}")
        for sd in SEEDS:
            cf = os.path.join(CKPT, f"{mode}_f{k}_s{sd}.npz")
            if os.path.exists(cf) and not QUICK_TEST:
                z = np.load(cf)
                if len(z["p"]) == len(te) and np.array_equal(z["te"], te):
                    preds.append(z["p"]); npar = float(z["npar"])
                    print(f"    seed {sd}: [cached] test AUC "
                          f"{roc_auc_score(y[te], z['p']):.4f}")
                    continue
            p, bv, npar = train_one(tr, va, te, sd, mode, lam)
            preds.append(p)
            if not QUICK_TEST:
                np.savez(cf, p=p, te=te, npar=npar)
            print(f"    seed {sd}: best val {bv:.4f} | test AUC "
                  f"{roc_auc_score(y[te], p):.4f}  "
                  f"[{(time.time()-T_ALL)/3600:.2f} h elapsed]")
        oof[te] = np.mean(preds, axis=0)
        print(f"  FOLD {k} test AUC {roc_auc_score(y[te], oof[te]):.4f}")

    done = ~np.isnan(oof)
    res = d.loc[done, ["img", "lesion_key", "label"]].copy()
    res["prob"] = oof[done]
    L = res.groupby("lesion_key").agg(y=("label", "max"), p=("prob", "mean"))
    la = roc_auc_score(L.y, L.p)
    print(f"\n  {mode}: region AUC {roc_auc_score(y[done], oof[done]):.4f} | "
          f"LESION AUC {la:.4f} | {npar/1e6:.2f} M params | {(time.time()-t0)/60:.0f} min")
    if not QUICK_TEST:
        res.rename(columns={"label": "true"})[["img", "true", "prob"]].to_csv(out, index=False)
        print(f"  saved {os.path.basename(out)}")
    summary.append(dict(variant=mode, lesion_auc=la, params=npar))

# ══════════════════════════ table ══════════════════════════
S_ = pd.DataFrame(summary)
if len(S_):
    base = S_.loc[S_.variant == "global", "lesion_auc"]
    b = float(base.iloc[0]) if len(base) else np.nan
    S_["delta_vs_global"] = (S_.lesion_auc - b).round(4)
    S_["params_M"] = (S_.params / 1e6).round(2)
    print("\n" + "=" * 68)
    print("POOLING ABLATION — pooled per-lesion AUC, patient-grouped 5-fold CV")
    print("=" * 68)
    print(S_[["variant", "params_M", "lesion_auc", "delta_vs_global"]].to_string(index=False))
    print(f"\ntotal wall time {(time.time()-T_ALL)/3600:.2f} h")
    if not QUICK_TEST:
        S_.to_csv(os.path.join(D, "pooling_ablation_summary.csv"), index=False)
        print("saved pooling_ablation_summary.csv")

role columns: {0: 'role_f0', 1: 'role_f1', 2: 'role_f2', 3: 'role_f3', 4: 'role_f4'}
1696 regions | 892 patients | 1005 lesions | malignant 46.2%
cached 1696 in 9s
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

VARIANT: dual   (lambda=2.0, 2048-d descriptor)
  fold 0  train 1054 val 264 test 378
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:20<00:00, 1.60MB/s]


      ep  1  val-AUC 0.7481 *
      ep  2  val-AUC 0.6101
      ep  3  val-AUC 0.6225
      ep  4  val-AUC 0.7933 *
      ep  5  val-AUC 0.7940 *
      ep  6  val-AUC 0.8111 *
      ep  7  val-AUC 0.8555 *
      ep  8  val-AUC 0.8422
      ep  9  val-AUC 0.8572 *


KeyboardInterrupt: 

**`IL2` cell 10** — ══════════════════════════════════════════════════════════════════════  
<sub>3 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════
#  CELL NM — NO-MASK BASELINE  (official split)
#
#  Two DenseNet-121 streams, straight from the cropped images:
#     stream T : tight crop  512x512   -> global average pool -> 1024
#     stream W : wide  crop  384x384   -> global average pool -> 1024
#     concat 2048 -> head
#
#  NO segmentation, NO mask, NO mask-weighted pooling. This is the row
#  that answers "what do you get without the segmentation stage?"
#
#  Everything else identical to the published two-stream configuration:
#  same crops, same split, same augmentation, same schedule, same focal
#  loss, same helper heads, same 4x TTA.
#
#  Saves: nomask_twostream_officialsplit_test.csv  (feeds the decision
#         layer cells unchanged)
# ══════════════════════════════════════════════════════════════════════
import os, gc, re, time, glob
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
cv2.setNumThreads(0)

D, DEV = "/root/autodl-tmp/CBIS", torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# --- identical to CELL_D two-stream official configuration -------------
ST, SW, BATCH = 512, 384, 8
SEEDS, EPOCHS, FREEZE = [11], 20, 3
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 3e-5
WD, GAMMA, AUX_W, MULT, PATIENCE = 1e-4, 2.0, 0.3, 3, 7
VAL_FRAC_SEED = 42
WIDE_DIR = os.path.join(D, "crops_wide_mass")

# ══════════════════════════ data ══════════════════════════
d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv")).reset_index(drop=True)
d["label"] = d["label"].astype(int)

# --- resolve the wide-crop filename convention automatically -----------
pool = set(os.path.basename(f) for f in glob.glob(os.path.join(WIDE_DIR, "*")))
assert pool, f"{WIDE_DIR} is empty or missing"

def wide_candidates(p):
    b = os.path.basename(str(p)); s = os.path.splitext(b)[0]
    r = s[:-4] if s.endswith("_img") else s
    return [b, s + ".png", r + ".png", r + "_wide.png", r + "_img.png",
            r + "_full.png", s + ".jpg", r + ".jpg", r + "_wide.jpg"]

hit = {}
for c_i in range(9):
    n = sum(1 for p in d["img"] if wide_candidates(p)[c_i] in pool)
    hit[c_i] = n
best_i = max(hit, key=hit.get)
print("wide-crop name match: pattern %d matched %d / %d files"
      % (best_i, hit[best_i], len(d)))
assert hit[best_i] == len(d), \
    ("could not match every wide crop; match counts per pattern = %s\n"
     "  sample tight name : %s\n  sample wide names : %s"
     % (hit, os.path.basename(str(d['img'].iloc[0])), sorted(pool)[:5]))
d["wide"] = d["img"].apply(lambda p: os.path.join(WIDE_DIR, wide_candidates(p)[best_i]))

print(f"{len(d)} regions | {d.patient_id.nunique()} patients | "
      f"{d.lesion_key.nunique()} lesions | malignant {100*d.label.mean():.1f}%")

# --- official split, with a patient-grouped val carved from train ------
sp = d["official_split"].astype(str).str.lower().str.strip()
te_idx = np.where(sp.str.contains("test"))[0]
tr_all = np.where(~sp.str.contains("test"))[0]
sub = d.iloc[tr_all]
strat = (sub.label.astype(str) + "_" +
         pd.to_numeric(sub.assessment, errors="coerce").fillna(4)
           .astype(int).clip(0, 5).astype(str)).values
sgk = StratifiedGroupKFold(6, shuffle=True, random_state=VAL_FRAC_SEED)
_, va_rel = next(iter(sgk.split(sub, strat, sub.patient_id.values)))
va_idx = tr_all[va_rel]
tr_idx = np.setdiff1d(tr_all, va_idx)
assert not (set(d.patient_id[tr_idx]) & set(d.patient_id[te_idx])), "LEAK train/test"
assert not (set(d.patient_id[tr_idx]) & set(d.patient_id[va_idx])), "LEAK train/val"
assert not (set(d.patient_id[va_idx]) & set(d.patient_id[te_idx])), "LEAK val/test"
print("official split: train %d | val %d | test %d  (patient-disjoint)"
      % (len(tr_idx), len(va_idx), len(te_idx)))

# --- cache both crops --------------------------------------------------
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
CT, CW, t0 = {}, {}, time.time()
for _, r in d.iterrows():
    kt = str(r["img"])
    if kt in CT: continue
    a = cv2.imread(kt, cv2.IMREAD_GRAYSCALE)
    a = np.zeros((ST, ST), np.uint8) if a is None else cv2.resize(a, (ST, ST))
    CT[kt] = _clahe.apply(a)
    b = cv2.imread(str(r["wide"]), cv2.IMREAD_GRAYSCALE)
    b = np.zeros((SW, SW), np.uint8) if b is None else cv2.resize(b, (SW, SW))
    CW[kt] = _clahe.apply(b)
print(f"cached {len(CT)} tight + {len(CW)} wide in {time.time()-t0:.0f}s")

def primary(x):
    return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()

aux, meta = {}, {}
for c in ["subtlety", "mass_shape", "mass_margins"]:
    if c not in d.columns or d[c].notna().sum() == 0: continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z: (z >= 1) & (z <= 5))
        codes, n = (v - 1).fillna(-1).astype(int).values, 5
    else:
        pr = d[c].map(primary)
        pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([k for k in pr.unique() if k != "UNK"])
        mp = {k: i for i, k in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z, -1)).astype(int).values, len(cats)
    if n > 1: aux[c], meta[c] = codes, n
AK = sorted(aux)
print("helper heads:", meta)

MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)

def norm3(g):
    g = g.astype(np.float32) / 255.
    return ((np.stack([g, g, g], 0) - MEAN) / STD).astype(np.float32)

class DS(Dataset):
    """No mask is ever loaded or returned."""
    def __init__(s, idx, augment, mult=1, tta=0):
        s.idx = np.asarray(idx); s.aug = augment
        s.mult = mult if augment else 1; s.tta = tta
    def __len__(s): return len(s.idx) * s.mult
    def __getitem__(s, i):
        j = int(s.idx[i % len(s.idx)])
        k = str(d.iloc[j]["img"])
        a, b = CT[k].copy(), CW[k].copy()
        if s.aug:
            if np.random.rand() < .5: a, b = a[:, ::-1], b[:, ::-1]
            if np.random.rand() < .5: a, b = a[::-1, :], b[::-1, :]
            r = np.random.randint(4)
            if r: a, b = np.rot90(a, r), np.rot90(b, r)
            a, b = np.ascontiguousarray(a), np.ascontiguousarray(b)
            if np.random.rand() < .7:
                ang, sc = np.random.uniform(-25, 25), np.random.uniform(.90, 1.12)
                Ma = cv2.getRotationMatrix2D((ST/2, ST/2), ang, sc)
                Mb = cv2.getRotationMatrix2D((SW/2, SW/2), ang, sc)
                a = cv2.warpAffine(a, Ma, (ST, ST), flags=cv2.INTER_LINEAR,
                                   borderMode=cv2.BORDER_REFLECT)
                b = cv2.warpAffine(b, Mb, (SW, SW), flags=cv2.INTER_LINEAR,
                                   borderMode=cv2.BORDER_REFLECT)
            if np.random.rand() < .5:
                gn, br = np.random.uniform(.85, 1.15), np.random.uniform(-12, 12)
                a = np.clip(a.astype(np.float32) * gn + br, 0, 255).astype(np.uint8)
                b = np.clip(b.astype(np.float32) * gn + br, 0, 255).astype(np.uint8)
        else:
            t = s.tta
            if   t == 1: a, b = a[:, ::-1], b[:, ::-1]
            elif t == 2: a, b = a[::-1, :], b[::-1, :]
            elif t == 3: a, b = np.rot90(a, 2), np.rot90(b, 2)
        a, b = np.ascontiguousarray(a), np.ascontiguousarray(b)
        av = (np.array([aux[c][j] for c in AK], dtype=np.int64) if AK else np.zeros(0, np.int64))
        return (torch.from_numpy(norm3(a)), torch.from_numpy(norm3(b)),
                torch.tensor(int(d.iloc[j]["label"])), torch.from_numpy(av))

# ══════════════════════ model: no mask input ══════════════════════
def backbone():
    try:
        return models.densenet121(
            weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
    except Exception:
        print("  (ImageNet weights unavailable)")
        return models.densenet121(weights=None).features

class NoMaskTwoStream(nn.Module):
    def __init__(s, aux_meta):
        super().__init__()
        s.bt, s.bw = backbone(), backbone()
        dim = 2048                                    # 1024 tight + 1024 wide
        s.head = nn.Sequential(nn.Linear(dim, 512), nn.BatchNorm1d(512), nn.ReLU(True),
                               nn.Dropout(0.4), nn.Linear(512, 2))
        s.keys = sorted(aux_meta)
        s.aux = nn.ModuleList([nn.Sequential(nn.Linear(dim, 128), nn.ReLU(True),
                                             nn.Dropout(0.3), nn.Linear(128, aux_meta[k]))
                               for k in s.keys])
    def forward(s, xt, xw):
        ft = F.relu(s.bt(xt)).mean((2, 3))            # plain GAP, no mask
        fw = F.relu(s.bw(xw)).mean((2, 3))            # plain GAP, no mask
        g  = torch.cat([ft, fw], 1)
        return s.head(g), [h(g) for h in s.aux]

def focal(lo, t, alpha):
    ce = F.cross_entropy(lo.float(), t, weight=alpha, reduction="none")
    return ((1 - torch.exp(-ce)) ** GAMMA * ce).mean()

@torch.no_grad()
def predict(net, idx, tta=True):
    net.eval(); tot = None
    for t in ([0, 1, 2, 3] if tta else [0]):
        ps = []
        for xt, xw, _, _ in DataLoader(DS(idx, False, 1, tta=t), batch_size=16,
                                       shuffle=False, num_workers=0):
            xt = xt.to(DEV).to(memory_format=torch.channels_last)
            xw = xw.to(DEV).to(memory_format=torch.channels_last)
            with torch.amp.autocast(device_type="cuda"):
                o, _ = net(xt, xw)
            ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot + ps
    return tot / (4 if tta else 1)

def train_one(tr, va, te, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    y = d["label"].values
    n0, n1 = float((y[tr] == 0).sum()), float((y[tr] == 1).sum())
    alpha = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)

    net = NoMaskTwoStream(meta).to(DEV).to(memory_format=torch.channels_last)
    bb = list(net.bt.parameters()) + list(net.bw.parameters())
    for p in bb: p.requires_grad = False
    hp = [p for n_, p in net.named_parameters()
          if not (n_.startswith("bt.") or n_.startswith("bw."))]
    scaler = torch.amp.GradScaler()
    opt, sch = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD), None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)

    best, bstate, bad = -1.0, None, 0
    for ep in range(1, EPOCHS + 1):
        if ep == FREEZE + 1:
            for p in bb: p.requires_grad = True
            opt = torch.optim.AdamW([{"params": bb, "lr": LR_BACK},
                                     {"params": hp, "lr": LR_HEAD_FT}], weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS-FREEZE))
        net.train()
        if ep <= FREEZE: net.bt.eval(); net.bw.eval()
        for xt, xw, t, a in tl:
            xt = xt.to(DEV, non_blocking=True).to(memory_format=torch.channels_last)
            xw = xw.to(DEV, non_blocking=True).to(memory_format=torch.channels_last)
            t, a = t.to(DEV), a.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o, ax = net(xt, xw)
                loss = focal(o, t, alpha)
                if len(ax):
                    loss = loss + AUX_W * sum(
                        F.cross_entropy(g.float(), a[:, h], ignore_index=-1)
                        for h, g in enumerate(ax)) / len(ax)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        if sch: sch.step()
        pv = predict(net, va, tta=False)
        auc = roc_auc_score(y[va], pv)
        star = ""
        if auc > best:
            best, bad, star = auc, 0, " *"
            bstate = {q: v.detach().cpu().clone() for q, v in net.state_dict().items()}
        else:
            bad += 1
        print(f"    ep {ep:2d}  val-AUC {auc:.4f}{star}")
        if bad >= PATIENCE:
            print("    early stop"); break
    net.load_state_dict({q: v.to(DEV) for q, v in bstate.items()})
    pte = predict(net, te, tta=True)
    pva = predict(net, va, tta=True)
    npar = sum(q.numel() for q in net.parameters())
    del net; gc.collect(); torch.cuda.empty_cache()
    return pte, pva, best, npar

# ══════════════════════════ run ══════════════════════════
print("\n" + "=" * 70)
print("NO-MASK TWO-STREAM  (tight 512 GAP  ||  wide 384 GAP)  2048-d")
print("=" * 70)
y = d["label"].values
T0, PT, PV, npar = time.time(), [], [], None
for sd in SEEDS:
    print(f"\n  seed {sd}")
    pte, pva, bv, npar = train_one(tr_idx, va_idx, te_idx, sd)
    PT.append(pte); PV.append(pva)
    print(f"  seed {sd}: best val {bv:.4f} | TEST ROI AUC {roc_auc_score(y[te_idx], pte):.4f}")
pte, pva = np.mean(PT, 0), np.mean(PV, 0)
print(f"\n  {npar/1e6:.2f} M params | {(time.time()-T0)/60:.0f} min")

# --- threshold fitted on the held-out val set, never on test ----------
GRID = np.round(np.arange(0.02, 0.99, 0.01), 3)
accs = [( (pva >= t).astype(int) == y[va_idx] ).mean() for t in GRID]
THR = float(GRID[int(np.argmax(accs))])
print(f"  global threshold fitted on val: {THR:.2f}")

# --- report at every unit ---------------------------------------------
t = d.iloc[te_idx][["img", "lesion_key", "patient_id", "label", "assessment"]].copy()
t["prob"] = pte
side = t["img"].astype(str).str.upper().str.extract(r"_(LEFT|RIGHT)_", expand=False)
t["breast_key"] = t["patient_id"].astype(str) + "_" + side.fillna("NA")

def unit(g, name):
    if g is None:
        print(f"  {name:<8} -- not derivable from filenames, skipped"); return
    a  = roc_auc_score(g.y, g.p)
    yh = (g.p >= THR).astype(int)
    tp = int(((yh == 1) & (g.y == 1)).sum()); fn = int(((yh == 0) & (g.y == 1)).sum())
    tn = int(((yh == 0) & (g.y == 0)).sum()); fp = int(((yh == 1) & (g.y == 0)).sum())
    print("  %-8s n=%-4d AUC %.4f   acc %5.1f%%  sens %.3f  spec %.3f  missed %d"
          % (name, len(g), a, 100*(tp+tn)/len(g), tp/max(tp+fn,1), tn/max(tn+fp,1), fn))

print("\n" + "=" * 70)
print("NO-MASK BASELINE — official test set")
print("=" * 70)
roi = pd.DataFrame(dict(y=t.label.values, p=t.prob.values))
unit(roi, "ROI")
for key, nm in (("lesion_key", "lesion"), ("breast_key", "breast"), ("patient_id", "patient")):
    if key == "breast_key" and side.isna().all():
        unit(None, "breast"); continue
    g = t.groupby(key).agg(p=("prob", "mean"), y=("label", "max"))
    unit(g, nm)

out = os.path.join(D, "nomask_twostream_officialsplit_test.csv")
t.rename(columns={"label": "true"})[
    ["img", "lesion_key", "patient_id", "breast_key", "true", "assessment", "prob"]
].to_csv(out, index=False)
print(f"\nsaved {os.path.basename(out)}  ({len(t)} rows)")
print("\nCompare against your headline (mask-weighted, official test):")
print("  ROI 0.8769 | lesion 0.9043 | breast 0.9016 | patient 0.9041")

wide-crop name match: pattern 0 matched 1696 / 1696 files
1696 regions | 892 patients | 1005 lesions | malignant 46.2%
official split: train 1098 | val 220 | test 378  (patient-disjoint)


/root/miniconda3/lib/python3.12/site-packages/sklearn/model_selection/_split.py:1036: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=6.
  warnings.warn(


cached 1696 tight + 1696 wide in 12s
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

NO-MASK TWO-STREAM  (tight 512 GAP  ||  wide 384 GAP)  2048-d

  seed 11
    ep  1  val-AUC 0.7356 *
    ep  2  val-AUC 0.7748 *
    ep  3  val-AUC 0.7559
    ep  4  val-AUC 0.7908 *
    ep  5  val-AUC 0.8179 *
    ep  6  val-AUC 0.8316 *
    ep  7  val-AUC 0.8706 *
    ep  8  val-AUC 0.8715 *
    ep  9  val-AUC 0.8515
    ep 10  val-AUC 0.8894 *
    ep 11  val-AUC 0.8716
    ep 12  val-AUC 0.8754
    ep 13  val-AUC 0.8715
    ep 14  val-AUC 0.8639
    ep 15  val-AUC 0.8687
    ep 16  val-AUC 0.8629
    ep 17  val-AUC 0.8862
    early stop
  seed 11: best val 0.8894 | TEST ROI AUC 0.8571

  15.75 M params | 21 min
  global threshold fitted on val: 0.51

NO-MASK BASELINE — official test set
  ROI      n=378  AUC 0.8571   acc  78.3%  sens 0.789  spec 0.779  missed 31
  lesion   n=223  AUC 0.8816   acc  82.5%  sens 0.793  spec 0.846  missed 18
  breast   -- not derivable from filenames,

## B · Segmentation ablations (U-Net baseline, dueling head)


**`BCF` cell 115** — CELL U-TRAIN — Standard U-Net baseline, IDENTICAL conditions to your  
<sub>1 output block(s) preserved</sub>


In [ ]:

# CELL U-TRAIN — Standard U-Net baseline, IDENTICAL conditions to your
#   Dueling model (cell 84). Only the model head/architecture differs.
#   Self-contained. Plain PyTorch only (no smp, no internet).
# ════════════════════════════════════════════════════════════════════
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2, os
from torch.utils.data import Dataset, DataLoader

DATA_ROOT = "/root/autodl-tmp/CBIS"
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE  = 256

EPOCHS      = 40          # same as cell 84
BATCH_SIZE  = 16          # same
LR          = 1e-3        # same
PREPROCESS  = "rl"        # same
PATIENCE    = 10          # same
torch.backends.cudnn.benchmark = True

# ── Standard U-Net: SAME encoder-decoder backbone as your Dueling model,
#    but a PLAIN segmentation head (single conv -> 2 channels).
#    No value/advantage streams. This is the honest architectural baseline.
def cblock(i, o):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class StandardUNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.e1 = cblock(1, base)
        self.e2 = cblock(base, base*2)
        self.e3 = cblock(base*2, base*4)
        self.e4 = cblock(base*4, base*8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = cblock(base*8, base*16)
        self.u4 = nn.ConvTranspose2d(base*16, base*8, 2, stride=2); self.d4 = cblock(base*16, base*8)
        self.u3 = nn.ConvTranspose2d(base*8,  base*4, 2, stride=2); self.d3 = cblock(base*8,  base*4)
        self.u2 = nn.ConvTranspose2d(base*4,  base*2, 2, stride=2); self.d2 = cblock(base*4,  base*2)
        self.u1 = nn.ConvTranspose2d(base*2,  base,   2, stride=2); self.d1 = cblock(base*2,  base)
        self.head = nn.Conv2d(base, 2, 1)     # PLAIN head (no dueling)
    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b),  e4], 1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.head(d1)

# ── RL preprocessing — IDENTICAL to cell 84 ──────────────────────────
PIPELINES={0:"no_preprocessing",1:"clahe_only",2:"gamma_only",3:"clahe_gamma",4:"clahe_denoise",
           5:"clahe_sharpen",6:"strong_clahe",7:"median_clahe",8:"denoise_clahe_sharpen"}
def apply_clahe(img,clip=2.0,grid=8): return cv2.createCLAHE(clipLimit=clip,tileGridSize=(grid,grid)).apply(img)
def apply_denoise(img,k=3): return cv2.GaussianBlur(img,(k,k),0)
def apply_sharpen(img): bb=cv2.GaussianBlur(img,(5,5),0); return cv2.addWeighted(img,1.5,bb,-0.5,0)
def apply_gamma(img,g=1.2): x=img.astype(np.float32)/255.0; return np.clip(np.power(x,g)*255.0,0,255).astype(np.uint8)
def apply_median(img,k=3): return cv2.medianBlur(img,k)
def apply_pipeline(img,pid):
    n=PIPELINES.get(pid,"no_preprocessing")
    if n=="no_preprocessing":return img
    if n=="clahe_only":return apply_clahe(img)
    if n=="gamma_only":return apply_gamma(img)
    if n=="clahe_gamma":return apply_gamma(apply_clahe(img))
    if n=="clahe_denoise":return apply_denoise(apply_clahe(img))
    if n=="clahe_sharpen":return apply_sharpen(apply_clahe(img))
    if n=="strong_clahe":return apply_clahe(img,clip=4.0,grid=8)
    if n=="median_clahe":return apply_clahe(apply_median(img))
    if n=="denoise_clahe_sharpen":return apply_sharpen(apply_clahe(apply_denoise(img)))
    return img
def extract_features_12d(img,clinical=None):
    f=img.astype(np.float32); tot=f.size
    mean_v,std_v=f.mean()/255.0,f.std()/128.0; min_v,max_v=f.min()/255.0,f.max()/255.0
    hist=cv2.calcHist([img],[0],None,[256],[0,256]); hist=hist/(hist.sum()+1e-8)
    hnz=hist[hist>1e-8]; ent=float(-np.sum(hnz*np.log2(hnz+1e-8)))/8.0
    dark=float((f<50).sum())/tot; bright=float((f>200).sum())/tot
    edges=cv2.Canny(img,50,150); ed=float(edges.sum())/(255.0*tot+1e-8)
    bd=sub=assess=0.5; isc=0.0
    if clinical:
        try:bd=float(clinical.get("breast_density",2))/4.0
        except:bd=0.5
        try:sub=float(clinical.get("subtlety",3))/5.0
        except:sub=0.6
        isc=1.0 if "calc" in str(clinical.get("abn_type","")).lower() else 0.0
        try:assess=float(clinical.get("assessment",3))/5.0
        except:assess=0.6
    return np.clip(np.array([mean_v,std_v,min_v,max_v,ent,dark,bright,ed,bd,sub,isc,assess],dtype=np.float32),0,1)
class DuelingDQN12(nn.Module):
    def __init__(self,state_dim=12,n_actions=9):
        super().__init__()
        self.feature_extractor=nn.Sequential(nn.Linear(state_dim,128),nn.LayerNorm(128),nn.ReLU(),nn.Dropout(0.2),
                                             nn.Linear(128,128),nn.LayerNorm(128),nn.ReLU())
        self.value_stream=nn.Sequential(nn.Linear(128,64),nn.ReLU(),nn.Linear(64,1))
        self.advantage_stream=nn.Sequential(nn.Linear(128,64),nn.ReLU(),nn.Linear(64,n_actions))
    def forward(self,x):
        z=self.feature_extractor(x);V=self.value_stream(z);A=self.advantage_stream(z)
        return V+A-A.mean(dim=1,keepdim=True)
_rl=DuelingDQN12(12,9).to(DEVICE)
_rl.load_state_dict(torch.load(os.path.join(DATA_ROOT,"rl_200_test/dueling_dqn_200_test.pth"),map_location=DEVICE))
_rl.eval()
def rl_preprocess(img,clinical=None):
    t=torch.tensor(extract_features_12d(img,clinical),dtype=torch.float32).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): act=int(torch.argmax(_rl(t),dim=1).item())
    return apply_pipeline(img,act)

# ── Dataset — IDENTICAL to cell 84 (CBISSegDataset) ──────────────────
def crop_mask_to_lesion(mb,pad=0.15):
    ys,xs=np.where(mb>0)
    if len(xs)==0: return None
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mb.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return mb[max(0,y0-py):min(h,y1+py),max(0,x0-px):min(w,x1+px)]
class CBISSegDataset(Dataset):
    def __init__(self, df, preprocess="rl", augment=False, size=IMG_SIZE):
        self.df=df.reset_index(drop=True); self.preprocess=preprocess; self.augment=augment; self.size=size
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((self.size,self.size),np.uint8)
        if mask is None: mask=np.zeros((self.size,self.size),np.uint8)
        mb=(mask>127).astype(np.uint8); mc=crop_mask_to_lesion(mb)
        if mc is None: mc=np.zeros((self.size,self.size),np.uint8)
        if self.preprocess=="rl":
            clinical={"breast_density":row.get("breast_density",2),"subtlety":row.get("subtlety",3),
                      "abn_type":row.get("abn_type","mass"),"assessment":row.get("assessment",3)}
            img=rl_preprocess(img,clinical)
        elif self.preprocess=="fixed":
            img=apply_pipeline(img,6)
        img_r=cv2.resize(img,(self.size,self.size),interpolation=cv2.INTER_LINEAR)
        mask_r=cv2.resize(mc,(self.size,self.size),interpolation=cv2.INTER_NEAREST)
        if self.augment:
            if np.random.rand()<0.5: img_r,mask_r=np.fliplr(img_r).copy(),np.fliplr(mask_r).copy()
            if np.random.rand()<0.5: img_r,mask_r=np.flipud(img_r).copy(),np.flipud(mask_r).copy()
        return (torch.from_numpy(img_r.astype(np.float32)/255.0).unsqueeze(0),
                torch.from_numpy((mask_r>0).astype(np.float32)).unsqueeze(0))

# ── Data loaders — same splits, same settings as cell 84 ─────────────
seg_train_df = pd.read_csv(os.path.join(DATA_ROOT, "seg_train_split.csv"))
seg_val_df   = pd.read_csv(os.path.join(DATA_ROOT, "seg_val_split.csv"))
train_ds = CBISSegDataset(seg_train_df, preprocess=PREPROCESS, augment=True)
val_ds   = CBISSegDataset(seg_val_df,   preprocess=PREPROCESS, augment=False)
train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_ld   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print("Train: "+str(len(train_ds))+" | Val: "+str(len(val_ds))+" | STANDARD U-NET | Preprocess: "+PREPROCESS)

model = StandardUNet().to(DEVICE)
opt   = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", patience=4, factor=0.5)
scaler = torch.amp.GradScaler()
CE_WEIGHT = torch.tensor([1.0, 2.0], device=DEVICE)   # same loss weights

def seg_loss_and_dice(q, mask):
    q = q.float()
    target = mask.squeeze(1).long()
    ce = F.cross_entropy(q, target, weight=CE_WEIGHT)
    prob_lesion = F.softmax(q, dim=1)[:, 1:2, :, :]
    inter = (prob_lesion * mask).sum(dim=(1,2,3))
    denom = prob_lesion.sum(dim=(1,2,3)) + mask.sum(dim=(1,2,3))
    dice_loss = 1 - ((2*inter + 1e-7) / (denom + 1e-7)).mean()
    loss = ce + dice_loss
    pred = q.argmax(dim=1, keepdim=True).float()
    hi   = (pred * mask).sum(dim=(1,2,3))
    hd   = (2*hi + 1e-7) / (pred.sum(dim=(1,2,3)) + mask.sum(dim=(1,2,3)) + 1e-7)
    return loss, hd.mean().item()

@torch.no_grad()
def evaluate(loader):
    model.eval()
    tot = 0.0
    for img, mask in loader:
        img, mask = img.to(DEVICE), mask.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"):
            q = model(img)
        _, d = seg_loss_and_dice(q, mask)
        tot += d
    return tot / len(loader)

history, best_dice, best_state, no_improve = [], 0.0, None, 0
print("\nTraining "+str(EPOCHS)+" epochs\n")
for ep in range(1, EPOCHS+1):
    model.train()
    ep_loss = ep_dice = 0.0
    for img, mask in train_ld:
        img, mask = img.to(DEVICE), mask.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"):
            q = model(img)
            loss, d = seg_loss_and_dice(q, mask)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        ep_loss += loss.item(); ep_dice += d
    tr_loss = ep_loss/len(train_ld); tr_dice = ep_dice/len(train_ld)
    val_dice = evaluate(val_ld)
    sched.step(val_dice)
    history.append({"epoch":ep,"train_loss":tr_loss,"train_dice":tr_dice,"val_dice":val_dice})
    flag = ""
    if val_dice > best_dice:
        best_dice = val_dice
        best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
        no_improve = 0; flag = " *best*"
    else:
        no_improve += 1
    print("Ep "+str(ep)+"/"+str(EPOCHS)+" | loss "+format(tr_loss,".4f")+" | train-Dice "+format(tr_dice,".4f")+" | val-Dice "+format(val_dice,".4f")+flag)
    if no_improve >= PATIENCE:
        print("Early stop at epoch "+str(ep)); break

if best_state is not None:
    model.load_state_dict({k:v.to(DEVICE) for k,v in best_state.items()})
SAVE_PATH = os.path.join(DATA_ROOT, "standard_unet_best.pth")
torch.save(best_state, SAVE_PATH)
pd.DataFrame(history).to_csv(os.path.join(DATA_ROOT, "standard_unet_history.csv"), index=False)
print("\n"+"="*55)
print("DONE | Best val pixel-Dice (U-Net): "+format(best_dice,".4f"))
print("MODEL SAVED: "+SAVE_PATH)
print("=" *55)

Train: 2433 | Val: 430 | STANDARD U-NET | Preprocess: rl

Training 40 epochs

Ep 1/40 | loss 0.4908 | train-Dice 0.8427 | val-Dice 0.8641 *best*
Ep 2/40 | loss 0.4114 | train-Dice 0.8650 | val-Dice 0.8661 *best*
Ep 3/40 | loss 0.3987 | train-Dice 0.8694 | val-Dice 0.8696 *best*
Ep 4/40 | loss 0.3905 | train-Dice 0.8714 | val-Dice 0.8669
Ep 5/40 | loss 0.3850 | train-Dice 0.8725 | val-Dice 0.8614
Ep 6/40 | loss 0.3839 | train-Dice 0.8733 | val-Dice 0.8736 *best*
Ep 7/40 | loss 0.3804 | train-Dice 0.8740 | val-Dice 0.8694
Ep 8/40 | loss 0.3775 | train-Dice 0.8748 | val-Dice 0.8757 *best*
Ep 9/40 | loss 0.3761 | train-Dice 0.8752 | val-Dice 0.8596
Ep 10/40 | loss 0.3745 | train-Dice 0.8751 | val-Dice 0.8754
Ep 11/40 | loss 0.3730 | train-Dice 0.8755 | val-Dice 0.8748
Ep 12/40 | loss 0.3707 | train-Dice 0.8767 | val-Dice 0.8755
Ep 13/40 | loss 0.3669 | train-Dice 0.8776 | val-Dice 0.8741
Ep 14/40 | loss 0.3581 | train-Dice 0.8799 | val-Dice 0.8745
Ep 15/40 | loss 0.3564 | train-Dice 0.8811

**`BCF` cell 116** — CELL U-TEST — Standard U-Net on test set + comparison vs your Dueling.  
<sub>1 output block(s) preserved</sub>


In [ ]:

# CELL U-TEST — Standard U-Net on test set + comparison vs your Dueling.
#   Same TestDS + same per-image Dice/IoU as your cell 85.
#   Run U-TRAIN first. Self-contained.
# ════════════════════════════════════════════════════════════════════
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2, os
from torch.utils.data import Dataset, DataLoader

DATA_ROOT = "/root/autodl-tmp/CBIS"
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE  = 256

def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class StandardUNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.e1=cblock(1,base); self.e2=cblock(base,base*2); self.e3=cblock(base*2,base*4); self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2); self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2); self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);   self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);   self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);     self.d1=cblock(base*2,base)
        self.head=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.pool(e1)); e3=self.e3(self.pool(e2)); e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        d4=self.d4(torch.cat([self.u4(b),e4],1)); d3=self.d3(torch.cat([self.u3(d4),e3],1))
        d2=self.d2(torch.cat([self.u2(d3),e2],1)); d1=self.d1(torch.cat([self.u1(d2),e1],1))
        return self.head(d1)

# Dueling model (to score it in the same run, same method)
class PixelDuelingDQN(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.e1=cblock(1,base); self.e2=cblock(base,base*2); self.e3=cblock(base*2,base*4); self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2); self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2); self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);   self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);   self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);     self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1); self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.pool(e1)); e3=self.e3(self.pool(e2)); e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        d4=self.d4(torch.cat([self.u4(b),e4],1)); d3=self.d3(torch.cat([self.u3(d4),e3],1))
        d2=self.d2(torch.cat([self.u2(d3),e2],1)); d1=self.d1(torch.cat([self.u1(d2),e1],1))
        V=self.value(d1); A=self.advantage(d1)
        return V+A-A.mean(dim=1,keepdim=True)

# RL preprocessing (identical to cell 85)
PIPELINES={0:"no_preprocessing",1:"clahe_only",2:"gamma_only",3:"clahe_gamma",4:"clahe_denoise",
           5:"clahe_sharpen",6:"strong_clahe",7:"median_clahe",8:"denoise_clahe_sharpen"}
def apply_clahe(img,clip=2.0,grid=8): return cv2.createCLAHE(clipLimit=clip,tileGridSize=(grid,grid)).apply(img)
def apply_denoise(img,k=3): return cv2.GaussianBlur(img,(k,k),0)
def apply_sharpen(img): bb=cv2.GaussianBlur(img,(5,5),0); return cv2.addWeighted(img,1.5,bb,-0.5,0)
def apply_gamma(img,g=1.2): x=img.astype(np.float32)/255.0; return np.clip(np.power(x,g)*255.0,0,255).astype(np.uint8)
def apply_median(img,k=3): return cv2.medianBlur(img,k)
def apply_pipeline(img,pid):
    n=PIPELINES.get(pid,"no_preprocessing")
    if n=="no_preprocessing":return img
    if n=="clahe_only":return apply_clahe(img)
    if n=="gamma_only":return apply_gamma(img)
    if n=="clahe_gamma":return apply_gamma(apply_clahe(img))
    if n=="clahe_denoise":return apply_denoise(apply_clahe(img))
    if n=="clahe_sharpen":return apply_sharpen(apply_clahe(img))
    if n=="strong_clahe":return apply_clahe(img,clip=4.0,grid=8)
    if n=="median_clahe":return apply_clahe(apply_median(img))
    if n=="denoise_clahe_sharpen":return apply_sharpen(apply_clahe(apply_denoise(img)))
    return img
def extract_features_12d(img,clinical=None):
    f=img.astype(np.float32); tot=f.size
    mean_v,std_v=f.mean()/255.0,f.std()/128.0; min_v,max_v=f.min()/255.0,f.max()/255.0
    hist=cv2.calcHist([img],[0],None,[256],[0,256]); hist=hist/(hist.sum()+1e-8)
    hnz=hist[hist>1e-8]; ent=float(-np.sum(hnz*np.log2(hnz+1e-8)))/8.0
    dark=float((f<50).sum())/tot; bright=float((f>200).sum())/tot
    edges=cv2.Canny(img,50,150); ed=float(edges.sum())/(255.0*tot+1e-8)
    bd=sub=assess=0.5; isc=0.0
    if clinical:
        try:bd=float(clinical.get("breast_density",2))/4.0
        except:bd=0.5
        try:sub=float(clinical.get("subtlety",3))/5.0
        except:sub=0.6
        isc=1.0 if "calc" in str(clinical.get("abn_type","")).lower() else 0.0
        try:assess=float(clinical.get("assessment",3))/5.0
        except:assess=0.6
    return np.clip(np.array([mean_v,std_v,min_v,max_v,ent,dark,bright,ed,bd,sub,isc,assess],dtype=np.float32),0,1)
class DuelingDQN12(nn.Module):
    def __init__(self,state_dim=12,n_actions=9):
        super().__init__()
        self.feature_extractor=nn.Sequential(nn.Linear(state_dim,128),nn.LayerNorm(128),nn.ReLU(),nn.Dropout(0.2),
                                             nn.Linear(128,128),nn.LayerNorm(128),nn.ReLU())
        self.value_stream=nn.Sequential(nn.Linear(128,64),nn.ReLU(),nn.Linear(64,1))
        self.advantage_stream=nn.Sequential(nn.Linear(128,64),nn.ReLU(),nn.Linear(64,n_actions))
    def forward(self,x):
        z=self.feature_extractor(x);V=self.value_stream(z);A=self.advantage_stream(z)
        return V+A-A.mean(dim=1,keepdim=True)
_rl=DuelingDQN12(12,9).to(DEVICE)
_rl.load_state_dict(torch.load(os.path.join(DATA_ROOT,"rl_200_test/dueling_dqn_200_test.pth"),map_location=DEVICE))
_rl.eval()
def rl_preprocess(img,clinical=None):
    t=torch.tensor(extract_features_12d(img,clinical),dtype=torch.float32).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): act=int(torch.argmax(_rl(t),dim=1).item())
    return apply_pipeline(img,act)

# Test dataset — IDENTICAL to cell 85 TestDS
def crop_mask_to_lesion(mb,pad=0.15):
    ys,xs=np.where(mb>0)
    if len(xs)==0: return None
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mb.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return mb[max(0,y0-py):min(h,y1+py),max(0,x0-px):min(w,x1+px)]
class TestDS(Dataset):
    def __init__(self, df, size=IMG_SIZE):
        self.df=df.reset_index(drop=True); self.size=size
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((self.size,self.size),np.uint8)
        if mask is None: mask=np.zeros((self.size,self.size),np.uint8)
        mb=(mask>127).astype(np.uint8); mc=crop_mask_to_lesion(mb)
        if mc is None: mc=np.zeros((self.size,self.size),np.uint8)
        clinical={"breast_density":row.get("breast_density",2),"subtlety":row.get("subtlety",3),
                  "abn_type":row.get("abn_type","mass"),"assessment":row.get("assessment",3)}
        img=rl_preprocess(img,clinical)
        img_r=cv2.resize(img,(self.size,self.size),interpolation=cv2.INTER_LINEAR)
        mask_r=cv2.resize(mc,(self.size,self.size),interpolation=cv2.INTER_NEAREST)
        return (torch.from_numpy(img_r.astype(np.float32)/255.0).unsqueeze(0),
                torch.from_numpy((mask_r>0).astype(np.float32)).unsqueeze(0))

test_df = pd.read_csv(os.path.join(DATA_ROOT,"seg_test_split.csv"))
test_ld = DataLoader(TestDS(test_df), batch_size=16, shuffle=False, num_workers=0, pin_memory=True)
print("Test images: "+str(len(test_df)))

def score(model):
    model.eval(); dices, ious = [], []
    with torch.no_grad():
        for img, mask in test_ld:
            img, mask = img.to(DEVICE), mask.to(DEVICE)
            with torch.amp.autocast(device_type="cuda"):
                q = model(img)
            pred = q.float().argmax(dim=1, keepdim=True).float()
            for b in range(pred.shape[0]):
                p=pred[b]; m=mask[b]
                inter=(p*m).sum().item(); psum=p.sum().item(); msum=m.sum().item()
                dices.append((2*inter+1e-7)/(psum+msum+1e-7))
                ious.append((inter+1e-7)/(psum+msum-inter+1e-7))
    return np.array(dices), np.array(ious)

# Score Dueling (your model)
mD = PixelDuelingDQN().to(DEVICE)
mD.load_state_dict(torch.load(os.path.join(DATA_ROOT,"pixel_dueling_supervised_best.pth"), map_location=DEVICE))
dD, iD = score(mD)

# Score U-Net baseline
mU = StandardUNet().to(DEVICE)
mU.load_state_dict(torch.load(os.path.join(DATA_ROOT,"standard_unet_best.pth"), map_location=DEVICE))
dU, iU = score(mU)

print("\n"+"="*62)
print("TEST COMPARISON (identical pipeline; only architecture differs)")
print("="*62)
print("  Model                    | Dice mean | Dice std | IoU mean | median")
print("  "+"-"*58)
print("  Dueling U-Net (yours)    |  "+format(dD.mean(),".4f")+"  |  "+format(dD.std(),".4f")+"  |  "+format(iD.mean(),".4f")+"  | "+format(np.median(dD),".4f"))
print("  Standard U-Net (baseline)|  "+format(dU.mean(),".4f")+"  |  "+format(dU.std(),".4f")+"  |  "+format(iU.mean(),".4f")+"  | "+format(np.median(dU),".4f"))
print("  "+"-"*58)
print("  Difference (Dueling - U-Net): "+format(dD.mean()-dU.mean(),"+.4f")+" Dice")

# Paired significance test
diff = dD - dU; n=len(diff); md=diff.mean(); sd=diff.std(ddof=1)
t = md/(sd/np.sqrt(n)+1e-12)
print("  Paired t-statistic: "+format(t,".3f")+"  (|t|<~2 => not significantly different)")
print("="*62)
pd.DataFrame({"dueling_dice":dD,"unet_dice":dU}).to_csv(
    os.path.join(DATA_ROOT,"test_dueling_vs_unet.csv"), index=False)
print("Saved: test_dueling_vs_unet.csv")

Test images: 379

TEST COMPARISON (identical pipeline; only architecture differs)
  Model                    | Dice mean | Dice std | IoU mean | median
  ----------------------------------------------------------
  Dueling U-Net (yours)    |  0.8996  |  0.0424  |  0.8201  | 0.9086
  Standard U-Net (baseline)|  0.9101  |  0.0438  |  0.8378  | 0.9201
  ----------------------------------------------------------
  Difference (Dueling - U-Net): -0.0105 Dice
  Paired t-statistic: -13.663  (|t|<~2 => not significantly different)
Saved: test_dueling_vs_unet.csv


**`BCF` cell 113** — CELL P1 - Plain-head baseline: SAME encoder-decoder as your model,  
<sub>0 output block(s) preserved</sub>


In [ ]:

# CELL P1 - Plain-head baseline: SAME encoder-decoder as your model,
#           dueling head REPLACED by a single 1x1 conv. Only the head differs.
#           Matched training to your 0.90 model. Run ALL-IN-ONE SETUP first.
# ════════════════════════════════════════════════════════════════════
import torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd, os
from torch.utils.data import DataLoader

DATA_ROOT = "/root/autodl-tmp/CBIS"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS, BATCH, LR = 40, 16, 1e-3
torch.backends.cudnn.benchmark = True

# ── Plain-head model: identical to PixelDuelingDQN EXCEPT the output head ──
class PixelPlainHead(nn.Module):
    """
    Same encoder-decoder as PixelDuelingDQN (uses the same cblock).
    The ONLY difference: the dueling value+advantage streams are replaced
    by a single 1x1 conv producing the 2-channel output directly.
    This isolates the effect of the dueling head.
    """
    def __init__(self, base=32):
        super().__init__()
        self.e1=cblock(1,base); self.e2=cblock(base,base*2); self.e3=cblock(base*2,base*4); self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2); self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2); self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);   self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);   self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);     self.d1=cblock(base*2,base)
        self.head = nn.Conv2d(base, 2, 1)      # <-- PLAIN head (no value/advantage)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.pool(e1)); e3=self.e3(self.pool(e2)); e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        d4=self.d4(torch.cat([self.u4(b),e4],1)); d3=self.d3(torch.cat([self.u3(d4),e3],1))
        d2=self.d2(torch.cat([self.u2(d3),e2],1)); d1=self.d1(torch.cat([self.u1(d2),e1],1))
        return self.head(d1)                    # (B,2,256,256) direct output

# ── Same loss/metric as your model (CE weighted + soft Dice) ──
CE_WEIGHT = torch.tensor([1.0, 2.0], device=DEVICE)
def seg_loss(q, mask):
    q=q.float(); target=mask.squeeze(1).long()
    ce=F.cross_entropy(q,target,weight=CE_WEIGHT)
    pl=F.softmax(q,dim=1)[:,1:2,:,:]
    inter=(pl*mask).sum(dim=(1,2,3)); denom=pl.sum(dim=(1,2,3))+mask.sum(dim=(1,2,3))
    dl=1-((2*inter+1e-7)/(denom+1e-7)).mean()
    return ce+dl
def hard_dice(q,mask):
    pred=q.float().argmax(dim=1,keepdim=True).float()
    inter=(pred*mask).sum(dim=(1,2,3)); denom=pred.sum(dim=(1,2,3))+mask.sum(dim=(1,2,3))
    return ((2*inter+1e-7)/(denom+1e-7)).mean().item()
@torch.no_grad()
def evaluate(model,loader):
    model.eval(); tot=0.0
    for img,mask in loader:
        img,mask=img.to(DEVICE),mask.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): q=model(img)
        tot+=hard_dice(q,mask)
    return tot/len(loader)

# ── Train the plain-head model ──
tr=pd.read_csv(os.path.join(DATA_ROOT,"seg_train_split.csv"))
va=pd.read_csv(os.path.join(DATA_ROOT,"seg_val_split.csv"))
train_ld=DataLoader(CroppedPixelDataset(tr,augment=True), batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
val_ld  =DataLoader(CroppedPixelDataset(va,augment=False),batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)
print("Train "+str(len(tr))+" | Val "+str(len(va))+" | PLAIN HEAD (only head differs from your model)")

model=PixelPlainHead().to(DEVICE)
opt=torch.optim.Adam(model.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
scaler=torch.amp.GradScaler()

best,bstate,noimp=0.0,None,0
for ep in range(1,EPOCHS+1):
    model.train()
    for img,mask in train_ld:
        img,mask=img.to(DEVICE),mask.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"):
            q=model(img); loss=seg_loss(q,mask)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    vd=evaluate(model,val_ld); sched.step(vd)
    flag=""
    if vd>best: best=vd; bstate={k:v.cpu().clone() for k,v in model.state_dict().items()}; noimp=0; flag=" *best*"
    else: noimp+=1
    print("Ep "+str(ep)+"/"+str(EPOCHS)+" | val-Dice "+format(vd,".4f")+flag)
    if noimp>=10: print("Early stop at "+str(ep)); break

torch.save(bstate, os.path.join(DATA_ROOT,"pixel_plainhead_best.pth"))
print("")
print("="*55)
print("PLAIN-HEAD RESULT (only the head differs from your model)")
print("  Plain head  val-Dice: "+format(best,".4f"))
print("  Your model  val-Dice: ~0.88 (dueling head, same architecture)")
print("  Saved: pixel_plainhead_best.pth")
print("="*55)

**`BCF` cell 114** — CELL P2-DET - Deterministic test comparison: Dueling vs Plain head  
<sub>2 output block(s) preserved</sub>


In [ ]:

# CELL P2-DET - Deterministic test comparison: Dueling vs Plain head
#   Fixed seeds + deterministic cuDNN so both models see byte-identical input.
#   Run ALL-IN-ONE SETUP + P1 first (needs PixelDuelingDQN, PixelPlainHead, CroppedPixelDataset).
# ════════════════════════════════════════════════════════════════════
import torch, torch.nn.functional as F
import numpy as np, pandas as pd, os, random
from torch.utils.data import DataLoader

DATA_ROOT = "/root/autodl-tmp/CBIS"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Full determinism ──
SEED = 42
def set_determinism(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

test_df = pd.read_csv(os.path.join(DATA_ROOT, "seg_test_split.csv"))

# ── Single per-image Dice function used for BOTH models ──
@torch.no_grad()
def per_image_dice(model):
    model.eval()
    set_determinism()                       # reset seed so input pipeline is identical each call
    loader = DataLoader(CroppedPixelDataset(test_df, augment=False),
                        batch_size=1, shuffle=False, num_workers=0, pin_memory=True)
    dices, ious = [], []
    for img, mask in loader:
        img, mask = img.to(DEVICE), mask.to(DEVICE)
        # NO autocast -> full precision, fully deterministic
        q = model(img).float()
        pred = q.argmax(dim=1, keepdim=True).float()
        inter = (pred*mask).sum().item(); ps, ms = pred.sum().item(), mask.sum().item()
        dices.append((2*inter+1e-7)/(ps+ms+1e-7))
        union = ps+ms-inter
        ious.append((inter+1e-7)/(union+1e-7))
    return np.array(dices), np.array(ious)

print("Test set: " + str(len(test_df)) + " images | deterministic (seed="+str(SEED)+")")
store = {}

# ── Dueling model ──
dueling_path = os.path.join(DATA_ROOT, "pixel_dueling_supervised_best.pth")
if not os.path.exists(dueling_path):
    dueling_path = os.path.join(DATA_ROOT, "pixel_rl_dueling_best.pth")
mD = PixelDuelingDQN().to(DEVICE)
mD.load_state_dict(torch.load(dueling_path, map_location=DEVICE))
dD, iD = per_image_dice(mD)
store["Dueling (yours)"] = dD
print("Dueling    : Dice "+format(dD.mean(),".4f")+" +/- "+format(dD.std(),".4f")+" | "+os.path.basename(dueling_path))

# ── Plain-head model ──
mP = PixelPlainHead().to(DEVICE)
mP.load_state_dict(torch.load(os.path.join(DATA_ROOT, "pixel_plainhead_best.pth"), map_location=DEVICE))
dP, iP = per_image_dice(mP)
store["Plain head"] = dP
print("Plain head : Dice "+format(dP.mean(),".4f")+" +/- "+format(dP.std(),".4f"))

# ── Comparison table ──
print("\n" + "="*62)
print("DETERMINISTIC TEST COMPARISON (byte-identical input, only head differs)")
print("="*62)
print("  Model            | Dice mean | Dice std | IoU mean | median")
print("  " + "-"*58)
print("  Dueling (yours)  |  "+format(dD.mean(),".4f")+"  |  "+format(dD.std(),".4f")+"  |  "+format(iD.mean(),".4f")+"  | "+format(np.median(dD),".4f"))
print("  Plain head       |  "+format(dP.mean(),".4f")+"  |  "+format(dP.std(),".4f")+"  |  "+format(iP.mean(),".4f")+"  | "+format(np.median(dP),".4f"))
print("  " + "-"*58)
print("  Difference (Dueling - Plain): "+format(dD.mean()-dP.mean(),"+.4f")+" Dice")

# ── Paired significance test (are they really different?) ──
diff = dD - dP
n = len(diff); mean_diff = diff.mean(); sd_diff = diff.std(ddof=1)
t_stat = mean_diff / (sd_diff/np.sqrt(n) + 1e-12)
print("\n  Paired difference per image: mean "+format(mean_diff,"+.4f")+" | std "+format(sd_diff,".4f"))
print("  Paired t-statistic: "+format(t_stat,".3f")+"  (|t| < ~2 => NOT significantly different)")
if abs(t_stat) < 2.0:
    print("  -> Dueling and Plain are STATISTICALLY EQUIVALENT on the test set.")
else:
    print("  -> The difference is statistically detectable (report honestly).")
print("="*62)

pd.DataFrame({k: v for k,v in store.items()}).to_csv(
    os.path.join(DATA_ROOT,"per_image_dice_deterministic.csv"), index=False)
print("Saved per-image Dice: per_image_dice_deterministic.csv")

Test set: 379 images | deterministic (seed=42)


error: OpenCV(4.13.0) :-1: error: (-5:Bad argument) in function 'resize'
> Overload resolution failed:
>  - src is not a numerical tuple
>  - Expected Ptr<cv::UMat> for argument 'src'


**`BCF` cell 91** — CELL NOVELTY-2 — Plain DQN segmentation (baseline vs your Dueling DQN)  
<sub>2 output block(s) preserved</sub>


In [ ]:

# CELL NOVELTY-2 — Plain DQN segmentation (baseline vs your Dueling DQN)
#                  Identical pipeline, ONLY the Q-head differs.
# ════════════════════════════════════════════════════════════════════
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2, os
from torch.utils.data import Dataset, DataLoader

DATA_ROOT = "/root/autodl-tmp/CBIS"
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE  = 256
EPOCHS, BATCH_SIZE, LR, PATIENCE = 40, 16, 1e-3, 10
PREPROCESS = "rl"
torch.backends.cudnn.benchmark = True

# ── Backbone block ───────────────────────────────────────────────────
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))

# ── PLAIN DQN: same encoder-decoder, but ONE Q-head (no value/advantage) ──
class PlainDQN(nn.Module):
    """
    Identical backbone to PixelDuelingDQN, but the head outputs Q-values
    DIRECTLY (single conv) instead of splitting into value + advantage.
    This is the base paper's plain DQN. Everything else is the same so the
    comparison isolates exactly the dueling contribution.
    """
    def __init__(self, base=32):
        super().__init__()
        self.e1=cblock(1,base); self.e2=cblock(base,base*2); self.e3=cblock(base*2,base*4); self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2); self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2); self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);   self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);   self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);     self.d1=cblock(base*2,base)
        self.q_head = nn.Conv2d(base, 2, 1)        # <-- single Q-head, NO dueling
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.pool(e1)); e3=self.e3(self.pool(e2)); e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        d4=self.d4(torch.cat([self.u4(b),e4],1)); d3=self.d3(torch.cat([self.u3(d4),e3],1))
        d2=self.d2(torch.cat([self.u2(d3),e2],1)); d1=self.d1(torch.cat([self.u1(d2),e1],1))
        return self.q_head(d1)                     # direct Q-values (B,2,256,256)

# ── Preprocessing (RL) — same as before ──────────────────────────────
PIPELINES={0:"no_preprocessing",1:"clahe_only",2:"gamma_only",3:"clahe_gamma",4:"clahe_denoise",
           5:"clahe_sharpen",6:"strong_clahe",7:"median_clahe",8:"denoise_clahe_sharpen"}
def apply_clahe(img,clip=2.0,grid=8): return cv2.createCLAHE(clipLimit=clip,tileGridSize=(grid,grid)).apply(img)
def apply_denoise(img,k=3): return cv2.GaussianBlur(img,(k,k),0)
def apply_sharpen(img): b=cv2.GaussianBlur(img,(5,5),0); return cv2.addWeighted(img,1.5,b,-0.5,0)
def apply_gamma(img,g=1.2): x=img.astype(np.float32)/255.0; return np.clip(np.power(x,g)*255.0,0,255).astype(np.uint8)
def apply_median(img,k=3): return cv2.medianBlur(img,k)
def apply_pipeline(img,pid):
    n=PIPELINES.get(pid,"no_preprocessing")
    if n=="no_preprocessing":return img
    if n=="clahe_only":return apply_clahe(img)
    if n=="gamma_only":return apply_gamma(img)
    if n=="clahe_gamma":return apply_gamma(apply_clahe(img))
    if n=="clahe_denoise":return apply_denoise(apply_clahe(img))
    if n=="clahe_sharpen":return apply_sharpen(apply_clahe(img))
    if n=="strong_clahe":return apply_clahe(img,clip=4.0,grid=8)
    if n=="median_clahe":return apply_clahe(apply_median(img))
    if n=="denoise_clahe_sharpen":return apply_sharpen(apply_clahe(apply_denoise(img)))
    return img
def extract_features_12d(img,clinical=None):
    f=img.astype(np.float32); tot=f.size
    mean_v,std_v=f.mean()/255.0,f.std()/128.0; min_v,max_v=f.min()/255.0,f.max()/255.0
    hist=cv2.calcHist([img],[0],None,[256],[0,256]); hist=hist/(hist.sum()+1e-8)
    hnz=hist[hist>1e-8]; ent=float(-np.sum(hnz*np.log2(hnz+1e-8)))/8.0
    dark=float((f<50).sum())/tot; bright=float((f>200).sum())/tot
    edges=cv2.Canny(img,50,150); ed=float(edges.sum())/(255.0*tot+1e-8)
    bd=sub=assess=0.5; isc=0.0
    if clinical:
        try:bd=float(clinical.get("breast_density",2))/4.0
        except:bd=0.5
        try:sub=float(clinical.get("subtlety",3))/5.0
        except:sub=0.6
        isc=1.0 if "calc" in str(clinical.get("abn_type","")).lower() else 0.0
        try:assess=float(clinical.get("assessment",3))/5.0
        except:assess=0.6
    return np.clip(np.array([mean_v,std_v,min_v,max_v,ent,dark,bright,ed,bd,sub,isc,assess],dtype=np.float32),0,1)
class DuelingDQN(nn.Module):
    def __init__(self,state_dim=12,n_actions=9):
        super().__init__()
        self.feature_extractor=nn.Sequential(nn.Linear(state_dim,128),nn.LayerNorm(128),nn.ReLU(),nn.Dropout(0.2),
                                             nn.Linear(128,128),nn.LayerNorm(128),nn.ReLU())
        self.value_stream=nn.Sequential(nn.Linear(128,64),nn.ReLU(),nn.Linear(64,1))
        self.advantage_stream=nn.Sequential(nn.Linear(128,64),nn.ReLU(),nn.Linear(64,n_actions))
    def forward(self,x):
        z=self.feature_extractor(x);V=self.value_stream(z);A=self.advantage_stream(z)
        return V+A-A.mean(dim=1,keepdim=True)
rl_preproc=DuelingDQN(12,9).to(DEVICE)
rl_preproc.load_state_dict(torch.load(os.path.join(DATA_ROOT,"rl_200_test/dueling_dqn_200_test.pth"),map_location=DEVICE))
rl_preproc.eval()
def rl_preprocess(img,clinical=None):
    t=torch.tensor(extract_features_12d(img,clinical),dtype=torch.float32).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): act=int(torch.argmax(rl_preproc(t),dim=1).item())
    return apply_pipeline(img,act),act

# ── Dataset — same as Dueling run ────────────────────────────────────
def crop_mask_to_lesion(mb,pad=0.15):
    ys,xs=np.where(mb>0)
    if len(xs)==0:return None
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mb.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return mb[max(0,y0-py):min(h,y1+py),max(0,x0-px):min(w,x1+px)]
class CBISSegDataset(Dataset):
    def __init__(self,df,size=IMG_SIZE,preprocess="rl",augment=False):
        self.df=df.reset_index(drop=True); self.size=size; self.preprocess=preprocess; self.augment=augment
    def __len__(self): return len(self.df)
    def _aug(self,img,mask):
        if np.random.rand()<0.5: img,mask=np.fliplr(img).copy(),np.fliplr(mask).copy()
        if np.random.rand()<0.5: img,mask=np.flipud(img).copy(),np.flipud(mask).copy()
        if np.random.rand()<0.5:
            k=np.random.randint(1,4); img,mask=np.rot90(img,k).copy(),np.rot90(mask,k).copy()
        return img,mask
    def __getitem__(self,idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None:img=np.zeros((self.size,self.size),np.uint8)
        if mask is None:mask=np.zeros((self.size,self.size),np.uint8)
        if self.preprocess=="rl":
            clinical={"breast_density":row.get("breast_density",2),"subtlety":row.get("subtlety",3),
                      "abn_type":row.get("abn_type","mass"),"assessment":row.get("assessment",3)}
            img,_=rl_preprocess(img,clinical)
        elif self.preprocess=="fixed":
            img=apply_pipeline(img,6)
        mb=(mask>127).astype(np.uint8); mc=crop_mask_to_lesion(mb)
        if mc is None:mc=np.zeros((self.size,self.size),np.uint8)
        img_r=cv2.resize(img,(self.size,self.size),interpolation=cv2.INTER_LINEAR)
        mask_r=cv2.resize(mc,(self.size,self.size),interpolation=cv2.INTER_NEAREST)
        if self.augment: img_r,mask_r=self._aug(img_r,mask_r)
        return (torch.from_numpy(img_r.astype(np.float32)/255.0).unsqueeze(0),
                torch.from_numpy((mask_r>0).astype(np.float32)).unsqueeze(0))

# ── Same loss/metric as Dueling run (CE + soft Dice) for fair compare ──
CE_WEIGHT = torch.tensor([1.0, 2.0], device=DEVICE)
def seg_loss_and_dice(q, mask):
    q=q.float(); target=mask.squeeze(1).long()
    ce=F.cross_entropy(q,target,weight=CE_WEIGHT)
    pl=F.softmax(q,dim=1)[:,1:2,:,:]
    inter=(pl*mask).sum(dim=(1,2,3)); denom=pl.sum(dim=(1,2,3))+mask.sum(dim=(1,2,3))
    dl=1-((2*inter+1e-7)/(denom+1e-7)).mean()
    loss=ce+dl
    pred=q.argmax(dim=1,keepdim=True).float()
    hi=(pred*mask).sum(dim=(1,2,3))
    hd=(2*hi+1e-7)/(pred.sum(dim=(1,2,3))+mask.sum(dim=(1,2,3))+1e-7)
    return loss,hd.mean().item()

@torch.no_grad()
def evaluate(model,loader):
    model.eval(); tot=0.0
    for img,mask in loader:
        img,mask=img.to(DEVICE),mask.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): q=model(img)
        _,d=seg_loss_and_dice(q,mask); tot+=d
    return tot/len(loader)

# ── Train the PLAIN DQN ──────────────────────────────────────────────
train_df=pd.read_csv(os.path.join(DATA_ROOT,"seg_train_split.csv"))
val_df  =pd.read_csv(os.path.join(DATA_ROOT,"seg_val_split.csv"))
train_ld=DataLoader(CBISSegDataset(train_df,preprocess=PREPROCESS,augment=True),batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True)
val_ld  =DataLoader(CBISSegDataset(val_df,preprocess=PREPROCESS,augment=False),batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
print(f"Train: {len(train_ld.dataset)} | Val: {len(val_ld.dataset)} | PLAIN DQN | preprocess={PREPROCESS}")

model=PlainDQN().to(DEVICE)
opt=torch.optim.Adam(model.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
scaler=torch.amp.GradScaler()

best_dice,best_state,no_improve,history=0.0,None,0,[]
for ep in range(1,EPOCHS+1):
    model.train(); ep_dice=0.0
    for img,mask in train_ld:
        img,mask=img.to(DEVICE),mask.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"):
            q=model(img); loss,d=seg_loss_and_dice(q,mask)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        ep_dice+=d
    val_dice=evaluate(model,val_ld); sched.step(val_dice)
    history.append({"epoch":ep,"train_dice":ep_dice/len(train_ld),"val_dice":val_dice})
    flag=""
    if val_dice>best_dice:
        best_dice=val_dice; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; no_improve=0; flag=" *best*"
    else: no_improve+=1
    print(f"Ep {ep:2d}/{EPOCHS} | train-Dice {ep_dice/len(train_ld):.4f} | val-Dice {val_dice:.4f}{flag}")
    if no_improve>=PATIENCE: print(f"Early stop at epoch {ep}."); break

torch.save(best_state, os.path.join(DATA_ROOT,"plain_dqn_seg_rl_best.pth"))
pd.DataFrame(history).to_csv(os.path.join(DATA_ROOT,"plain_dqn_seg_rl_history.csv"),index=False)
print("\n"+"="*55)
print("NOVELTY 2 RESULT")
print("="*55)
print(f"  Plain DQN   val-Dice: {best_dice:.4f}")
print(f"  Dueling DQN val-Dice: ~0.8797  (your model)")
print(f"  Difference  : {0.8797 - best_dice:+.4f}  (positive = Dueling wins)")
print("="*55)

Train: 2433 | Val: 430 | PLAIN DQN | preprocess=rl


Ep  1/40 | train-Dice 0.8413 | val-Dice 0.8575 *best*
Ep  2/40 | train-Dice 0.8650 | val-Dice 0.8558
Ep  3/40 | train-Dice 0.8663 | val-Dice 0.8668 *best*
Ep  4/40 | train-Dice 0.8679 | val-Dice 0.8681 *best*
Ep  5/40 | train-Dice 0.8681 | val-Dice 0.8526
Ep  6/40 | train-Dice 0.8683 | val-Dice 0.8663
Ep  7/40 | train-Dice 0.8696 | val-Dice 0.8689 *best*
Ep  8/40 | train-Dice 0.8700 | val-Dice 0.8671
Ep  9/40 | train-Dice 0.8665 | val-Dice 0.8674
Ep 10/40 | train-Dice 0.8708 | val-Dice 0.8716 *best*
Ep 11/40 | train-Dice 0.8699 | val-Dice 0.8677
Ep 12/40 | train-Dice 0.8716 | val-Dice 0.8685
Ep 13/40 | train-Dice 0.8703 | val-Dice 0.8673
Ep 14/40 | train-Dice 0.8703 | val-Dice 0.8719 *best*
Ep 15/40 | train-Dice 0.8732 | val-Dice 0.8697
Ep 16/40 | train-Dice 0.8727 | val-Dice 0.8598
Ep 17/40 | train-Dice 0.8737 | val-Dice 0.8735 *best*
Ep 18/40 | train-Dice 0.8729 | val-Dice 0.8750 *best*
Ep 19/40 | train-Dice 0.8747 | val-Dice 0.8687
Ep 20/40 | train-Dice 0.8749 | val-Dice 0.8736
Ep 2

## C · Classification negative results


**`BCF` cell 341** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [10]:
# ══════════════════════════════════════════════════════════════════════
# HARD-CASE-FOCUSED RETRAINING — weight BI-RADS 3/4 heavily
#   forces the model to learn the uncertain cases, raising their AUC
#   set LESION="calc" or "mass" | evaluates BI-RADS 3+4 specifically
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, balanced_accuracy_score
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==== CHOOSE LESION TYPE ====
LESION="calc"     # "calc" or "mass"
# ============================
CSV = "cbis_calc_fixed.csv" if LESION=="calc" else "cbis_mass_fixed.csv"
AUXC = ["subtlety","calc_type","calc_dist"] if LESION=="calc" else ["subtlety","mass_shape","mass_margins"]
HARD_WEIGHT=3.0   # how much more to sample BI-RADS 3/4

S=512; BATCH=12; MULT=8; EPOCHS=22; LR_HEAD=1e-3; LR_FT=1e-5; FREEZE=3; GAMMA=2.5; AUX_W=0.3; NFOLD=5
torch.backends.cudnn.benchmark=True; torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
MEAN=np.array([0.485,0.456,0.406],np.float32); STD=np.array([0.229,0.224,0.225],np.float32)

sub=pd.read_csv(os.path.join(D,CSV)); sub["label"]=sub["label"].astype(int)
sub["assessment"]=pd.to_numeric(sub["assessment"],errors="coerce")
sub=sub.dropna(subset=["img","label"]).reset_index(drop=True)
is_hard=sub.assessment.isin([3,4]).values
print(LESION+": n="+str(len(sub))+" | hard(BIRADS3/4) "+format(100*is_hard.mean(),".0f")+"% | malignant "+format(100*sub.label.mean(),".1f")+"%\n")

CACHE={}
def build_cache(df):
    t0=time.time()
    for _,r in df.iterrows():
        k=r["img"]
        if k in CACHE: continue
        img=cv2.imread(k,cv2.IMREAD_GRAYSCALE)
        img=cv2.resize(img if img is not None else np.zeros((S,S),np.uint8),(S,S))
        mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        mask=(cv2.resize(mk,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.float32) if mk is not None else np.zeros((S,S),np.float32)
        CACHE[k]=(_clahe.apply(img),mask)
    print("  cached "+str(len(CACHE))+" in "+format(time.time()-t0,".1f")+"s")

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
def build_aux(df,cols,topn=6):
    out={};meta={}
    for c in cols:
        if c not in df.columns or df[c].notna().sum()==0: continue
        if c=="subtlety":
            v=pd.to_numeric(df[c],errors="coerce").where(lambda z:(z>=1)&(z<=5))
            codes=(v-1).fillna(-1).astype(int); n=int(v.max()) if v.notna().any() else 0
        else:
            pr=df[c].map(primary); keep=pr.value_counts().head(topn).index.tolist()
            pr=pr.where(pr.isin(keep),"OTHER")
            cats=sorted([k for k in pr.unique() if k!="UNK"]); mp={k:i for i,k in enumerate(cats)}
            codes=pr.map(lambda z:mp.get(z,-1)).astype(int); n=len(cats)
        if n>1: out[c]=codes.values; meta[c]=n
    return out,meta

class DS(Dataset):
    def __init__(s,df,aux,aug,mult=1,tta=0):
        s.keys=df["img"].values; s.lab=df["label"].astype(int).values
        s.aux=aux; s.aug=aug; s.mult=mult if aug else 1; s.tta=tta; s.ak=sorted(aux.keys())
    def __len__(s): return len(s.keys)*s.mult
    def __getitem__(s,i):
        j=i%len(s.keys); k=i//len(s.keys); img,mask=CACHE[s.keys[j]]; img=img.copy(); mask=mask.copy()
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); mask=np.fliplr(mask)
            elif k%8==2: img=np.flipud(img); mask=np.flipud(mask)
            elif k%8==3: img=np.rot90(img,1); mask=np.rot90(mask,1)
            elif k%8==4: img=np.rot90(img,2); mask=np.rot90(mask,2)
            elif k%8==5: img=np.rot90(img,3); mask=np.rot90(mask,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((S/2,S/2),np.random.uniform(-20,20),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(S,S),borderMode=cv2.BORDER_REFLECT)
                mask=cv2.warpAffine(mask,M,(S,S),flags=cv2.INTER_NEAREST)
            elif k%8==7: img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
        if s.tta==1: img=np.fliplr(img); mask=np.fliplr(mask)
        elif s.tta==2: img=np.flipud(img); mask=np.flipud(mask)
        elif s.tta==3: img=np.rot90(img,2); mask=np.rot90(mask,2)
        im=np.ascontiguousarray(img).astype(np.float32)/255.
        x=np.stack([im,im,im],0); x=((x.transpose(1,2,0)-MEAN)/STD).transpose(2,0,1).astype(np.float32)
        av=np.array([s.aux[k2][j] for k2 in s.ak],dtype=np.int64) if s.ak else np.zeros(0,np.int64)
        return (torch.from_numpy(np.ascontiguousarray(x)),torch.from_numpy(np.ascontiguousarray(mask))[None],
                torch.tensor(int(s.lab[j])),torch.from_numpy(av))

class Net(nn.Module):
    def __init__(s,meta):
        super().__init__()
        try: dn=models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception: dn=models.densenet121(weights=None)
        s.b=dn.features; s.pool=nn.AdaptiveAvgPool2d(1)
        s.head=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Dropout(0.5),nn.Linear(256,2))
        s.keys=sorted(meta.keys())
        s.aux=nn.ModuleList([nn.Sequential(nn.Linear(1024,128),nn.ReLU(),nn.Dropout(0.3),nn.Linear(128,meta[k])) for k in s.keys])
    def forward(s,x,mask):
        f=F.relu(s.b(x))
        m=F.interpolate(mask,size=f.shape[2:],mode="bilinear",align_corners=False)
        w=1.0+2.0*m; f=f*w; g=(f.sum((2,3))/(w.sum((2,3))+1e-6))
        return s.head(g),[h(g) for h in s.aux]

build_cache(sub)
aux,meta=build_aux(sub,AUXC); print("aux heads:",meta,"\n")
y=sub.label.values; gr=sub.patient_id.values
strat=sub["label"].astype(str)+"_"+sub.assessment.isin([3,4]).astype(int).astype(str)
oofp=np.zeros(len(sub)); fa=[]

for fold,(tri,tei) in enumerate(StratifiedGroupKFold(NFOLD,shuffle=True,random_state=42).split(sub,strat,gr),1):
    t0=time.time()
    g2=gr[tri]; uq=np.array(sorted(set(g2))); rs=np.random.RandomState(fold)
    vg=set(rs.permutation(uq)[:max(1,int(0.12*len(uq)))]); vm=np.array([g in vg for g in g2])
    tr_i=tri[~vm]; va_i=tri[vm]; assert len(set(gr[tr_i])&set(gr[tei]))==0
    tr=sub.iloc[tr_i]; va=sub.iloc[va_i]; te=sub.iloc[tei]; sl=lambda ix:{k:v[ix] for k,v in aux.items()}
    n0=float((tr.label==0).sum()); n1=float((tr.label==1).sum()); al=torch.tensor([n1/(n0+n1),n0/(n0+n1)],device=DEV)
    # HARD-CASE WEIGHTING: BI-RADS 3/4 sampled HARD_WEIGHT x more
    sw=np.ones(len(tr_i)); sw[is_hard[tr_i]]=HARD_WEIGHT
    sampler=WeightedRandomSampler(torch.DoubleTensor(np.tile(sw,MULT)),num_samples=len(tr_i)*MULT,replacement=True)
    def focal(lo,t):
        ce=F.cross_entropy(lo.float(),t,weight=al,reduction="none"); pt=torch.exp(-ce); return ((1-pt)**GAMMA*ce).mean()
    torch.manual_seed(fold); np.random.seed(fold)
    net=Net(meta).to(DEV).to(memory_format=torch.channels_last)
    for p_ in net.b.parameters(): p_.requires_grad=False
    sc=torch.amp.GradScaler(); opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=1e-3)
    tl=DataLoader(DS(tr,sl(tr_i),True,MULT),batch_size=BATCH,sampler=sampler,num_workers=0,pin_memory=True)
    @torch.no_grad()
    def col(df_,ix,tta=True):
        net.eval(); reps=[0,1,2,3] if tta else [0]; tot=None; lb=None
        for t in reps:
            ld=DataLoader(DS(df_,sl(ix),False,tta=t),batch_size=20,shuffle=False,num_workers=0,pin_memory=True); ps=[];ll=[]
            for x,m,t2,a in ld:
                x=x.to(DEV,non_blocking=True).to(memory_format=torch.channels_last); m=m.to(DEV,non_blocking=True)
                with torch.amp.autocast(device_type="cuda"): o,_=net(x,m)
                ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy()); ll+=list(t2.numpy())
            ps=np.array(ps); lb=np.array(ll); tot=ps if tot is None else tot+ps
        return lb,tot/len(reps)
    best=0.;bs=None;ni=0
    for ep in range(1,EPOCHS+1):
        if ep==FREEZE+1:
            for p_ in net.b.parameters(): p_.requires_grad=True
            opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=1e-3)
        net.train()
        if ep<=FREEZE: net.b.eval()
        for x,m,t2,a in tl:
            x=x.to(DEV,non_blocking=True).to(memory_format=torch.channels_last); m=m.to(DEV,non_blocking=True)
            t2=t2.to(DEV,non_blocking=True); a=a.to(DEV,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o,ax=net(x,m); lm=focal(o,t2); la=torch.zeros((),device=DEV)
                for h,lg in enumerate(ax): la=la+F.cross_entropy(lg.float(),a[:,h],ignore_index=-1)
                if len(ax): la=la/len(ax)
                loss=lm+AUX_W*la
            if not torch.isfinite(loss): continue
            sc.scale(loss).backward(); sc.unscale_(opt); torch.nn.utils.clip_grad_norm_(net.parameters(),5.0)
            sc.step(opt); sc.update()
        yv,pv=col(va,va_i,tta=False); au=roc_auc_score(yv,pv) if len(set(yv))>1 else 0
        if au>best: best=au; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        if ni>=5: break
    net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
    yt,pt=col(te,tei); oofp[tei]=pt; a_=roc_auc_score(yt,pt); fa.append(a_)
    print("  fold "+str(fold)+"  AUC "+format(a_,".4f")+"  ("+format(time.time()-t0,".0f")+"s)")

sub["prob"]=oofp
sub.to_csv(os.path.join(D,"cv_"+LESION+"_hardfocus_oof.csv"),index=False)
y=sub.label.values; hard=sub.assessment.isin([3,4]).values
def acc_at_best(mask):
    yy=y[mask]; pp=oofp[mask]
    if len(set(yy))<2: return 0,0
    a=max([(accuracy_score(yy,(pp>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[0]
    return a, roc_auc_score(yy,pp)
print("\n"+"="*60)
print(LESION.upper()+" HARD-FOCUSED — did BI-RADS 3/4 improve?")
print("="*60)
print("  overall AUC "+format(np.mean(fa),".4f")+" +/- "+format(np.std(fa),".4f"))
a_all,auc_all=acc_at_best(np.ones(len(y),bool))
a_hard,auc_hard=acc_at_best(hard)
a_easy,auc_easy=acc_at_best(sub.assessment.isin([2,5]).values)
print("  FULL data:        acc "+format(100*a_all,".1f")+"%  AUC "+format(auc_all,".3f"))
print("  BI-RADS 3+4:      acc "+format(100*a_hard,".1f")+"%  AUC "+format(auc_hard,".3f")+"   (was: calc 62.6%/0.645, mass 74.4%/0.777)")
print("  BI-RADS 2+5:      acc "+format(100*a_easy,".1f")+"%  AUC "+format(auc_easy,".3f"))
print("="*60)

calc: n=1866 | hard(BIRADS3/4) 56% | malignant 36.0%

  cached 1866 in 9.8s
aux heads: {'subtlety': 5, 'calc_type': 7, 'calc_dist': 5} 

  fold 1  AUC 0.8046  (723s)
  fold 2  AUC 0.7845  (579s)
  fold 3  AUC 0.7481  (638s)
  fold 4  AUC 0.7644  (625s)
  fold 5  AUC 0.8276  (1275s)

CALC HARD-FOCUSED — did BI-RADS 3/4 improve?
  overall AUC 0.7858 +/- 0.0282
  FULL data:        acc 71.8%  AUC 0.780
  BI-RADS 3+4:      acc 62.9%  AUC 0.650   (was: calc 62.6%/0.645, mass 74.4%/0.777)
  BI-RADS 2+5:      acc 86.9%  AUC 0.924


**`BCF` cell 345** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [14]:
# ══════════════════════════════════════════════════════════════════════
# MASS — DUAL-PATHWAY: margin pathway + INTERNAL-TEXTURE pathway
#   targets the smooth-cancer FN problem: forces model to read texture
#   INSIDE the lesion, not just the margin shape
#   pathway A: full crop (margin/shape)  pathway B: texture-enhanced lesion interior
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
S=512; BATCH=10; MULT=8; EPOCHS=20; LR_HEAD=1e-3; LR_FT=1e-5; FREEZE=3; GAMMA=2.0; NFOLD=5
torch.backends.cudnn.benchmark=True; torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
_clahe_strong=cv2.createCLAHE(clipLimit=4.0,tileGridSize=(4,4))   # stronger for texture
MEAN=np.array([0.485,0.456,0.406],np.float32); STD=np.array([0.229,0.224,0.225],np.float32)

sub=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); sub["label"]=sub["label"].astype(int)
sub["assessment"]=pd.to_numeric(sub["assessment"],errors="coerce")
sub=sub.dropna(subset=["img","label"]).reset_index(drop=True)
print("mass n="+str(len(sub))+" | malignant "+format(100*sub.label.mean(),".1f")+"%\n")

def texture_map(img):
    """enhance INTERNAL texture: local standard deviation + strong CLAHE"""
    g=_clahe_strong.apply(img)
    # local variance (texture heterogeneity) via difference from local mean
    blur=cv2.GaussianBlur(g.astype(np.float32),(9,9),0)
    var=cv2.GaussianBlur((g.astype(np.float32)-blur)**2,(9,9),0)
    tex=np.sqrt(var); tex=np.clip(tex/ (tex.max()+1e-6)*255,0,255).astype(np.uint8)
    return tex

CACHE={}
def build_cache(df):
    t0=time.time()
    for _,r in df.iterrows():
        k=r["img"]
        if k in CACHE: continue
        img=cv2.imread(k,cv2.IMREAD_GRAYSCALE)
        img=cv2.resize(img if img is not None else np.zeros((S,S),np.uint8),(S,S))
        mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        mask=(cv2.resize(mk,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8) if mk is not None else np.zeros((S,S),np.uint8)
        full=_clahe.apply(img)                          # pathway A: margin/shape
        tex=texture_map(img)                            # pathway B: texture
        # texture pathway focuses INSIDE lesion: keep texture in mask, dim outside
        tex_in=(tex.astype(np.float32)*(0.3+0.7*mask)).astype(np.uint8)
        CACHE[k]=(full,tex_in,mask)
    print("  cached "+str(len(CACHE))+" in "+format(time.time()-t0,".1f")+"s")

class DS(Dataset):
    def __init__(s,idx,aug,mult=1,tta=0):
        s.idx=np.array(idx); s.aug=aug; s.mult=mult if aug else 1; s.tta=tta
    def __len__(s): return len(s.idx)*s.mult
    def __getitem__(s,i):
        j=s.idx[i%len(s.idx)]; k=i//len(s.idx); r=sub.iloc[j]
        full,tex,mask=CACHE[r["img"]]; full=full.copy(); tex=tex.copy(); mask=mask.copy()
        if s.aug and k>0:
            if   k%8==1: full,tex,mask=np.fliplr(full),np.fliplr(tex),np.fliplr(mask)
            elif k%8==2: full,tex,mask=np.flipud(full),np.flipud(tex),np.flipud(mask)
            elif k%8==3: full,tex,mask=np.rot90(full,1),np.rot90(tex,1),np.rot90(mask,1)
            elif k%8==4: full,tex,mask=np.rot90(full,2),np.rot90(tex,2),np.rot90(mask,2)
            elif k%8==5: full,tex,mask=np.rot90(full,3),np.rot90(tex,3),np.rot90(mask,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((S/2,S/2),np.random.uniform(-20,20),np.random.uniform(.9,1.1))
                full=cv2.warpAffine(full,M,(S,S),borderMode=cv2.BORDER_REFLECT)
                tex=cv2.warpAffine(tex,M,(S,S),borderMode=cv2.BORDER_REFLECT)
                mask=cv2.warpAffine(mask,M,(S,S),flags=cv2.INTER_NEAREST)
            elif k%8==7: full=np.clip(full.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
        if s.tta==1: full,tex,mask=np.fliplr(full),np.fliplr(tex),np.fliplr(mask)
        elif s.tta==2: full,tex,mask=np.rot90(full,2),np.rot90(tex,2),np.rot90(mask,2)
        def norm(a):
            im=np.ascontiguousarray(a).astype(np.float32)/255.
            x=np.stack([im,im,im],0); return ((x.transpose(1,2,0)-MEAN)/STD).transpose(2,0,1).astype(np.float32)
        return (torch.from_numpy(np.ascontiguousarray(norm(full))),
                torch.from_numpy(np.ascontiguousarray(norm(tex))),
                torch.from_numpy(np.ascontiguousarray(mask.astype(np.float32)))[None],
                torch.tensor(int(r["label"])))

class DualNet(nn.Module):
    """two DenseNet pathways: margin (full) + texture (interior), fused"""
    def __init__(s):
        super().__init__()
        def backbone():
            try: return models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
            except Exception: return models.densenet121(weights=None).features
        s.bm=backbone()   # margin pathway
        s.bt=backbone()   # texture pathway
        s.pool=nn.AdaptiveAvgPool2d(1)
        s.head=nn.Sequential(nn.Linear(2048,512),nn.ReLU(),nn.Dropout(0.5),
                             nn.Linear(512,128),nn.ReLU(),nn.Dropout(0.3),nn.Linear(128,2))
    def forward(s,full,tex,mask):
        fm=F.relu(s.bm(full))
        m=F.interpolate(mask,size=fm.shape[2:],mode="bilinear",align_corners=False)
        w=1.0+2.0*m; fm=fm*w; gm=(fm.sum((2,3))/(w.sum((2,3))+1e-6))   # margin, lesion-weighted
        gt=s.pool(F.relu(s.bt(tex))).flatten(1)                          # texture
        return s.head(torch.cat([gm,gt],1))

build_cache(sub)
y=sub.label.values; gr=sub.patient_id.values
strat=sub["label"].astype(str)+"_"+sub.assessment.isin([3,4]).astype(int).astype(str)
oofp=np.zeros(len(sub)); fa=[]
for fold,(tri,tei) in enumerate(StratifiedGroupKFold(NFOLD,shuffle=True,random_state=42).split(sub,strat,gr),1):
    t0=time.time()
    g2=gr[tri]; uq=np.array(sorted(set(g2))); rs=np.random.RandomState(fold)
    vg=set(rs.permutation(uq)[:max(1,int(0.12*len(uq)))]); vm=np.array([g in vg for g in g2])
    tr_i=tri[~vm]; va_i=tri[vm]; assert len(set(gr[tr_i])&set(gr[tei]))==0
    n0=float((y[tr_i]==0).sum()); n1=float((y[tr_i]==1).sum()); al=torch.tensor([n1/(n0+n1),n0/(n0+n1)],device=DEV)
    def focal(lo,t):
        ce=F.cross_entropy(lo.float(),t,weight=al,reduction="none"); pt=torch.exp(-ce); return ((1-pt)**GAMMA*ce).mean()
    torch.manual_seed(fold); np.random.seed(fold)
    net=DualNet().to(DEV).to(memory_format=torch.channels_last)
    for b in [net.bm,net.bt]:
        for p_ in b.parameters(): p_.requires_grad=False
    sc=torch.amp.GradScaler(); opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=1e-3)
    tl=DataLoader(DS(tr_i,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
    @torch.no_grad()
    def col(idx,tta=True):
        net.eval(); reps=[0,1,2] if tta else [0]; tot=None; lb=None
        for t in reps:
            ld=DataLoader(DS(idx,False,tta=t),batch_size=16,shuffle=False,num_workers=0,pin_memory=True); ps=[];ll=[]
            for fu,tx,m,t2 in ld:
                fu=fu.to(DEV,non_blocking=True).to(memory_format=torch.channels_last)
                tx=tx.to(DEV,non_blocking=True).to(memory_format=torch.channels_last); m=m.to(DEV,non_blocking=True)
                with torch.amp.autocast(device_type="cuda"): o=net(fu,tx,m)
                ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy()); ll+=list(t2.numpy())
            ps=np.array(ps); lb=np.array(ll); tot=ps if tot is None else tot+ps
        return lb,tot/len(reps)
    best=0.;bs=None;ni=0
    for ep in range(1,EPOCHS+1):
        if ep==FREEZE+1:
            for b in [net.bm,net.bt]:
                for p_ in b.parameters(): p_.requires_grad=True
            opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=1e-3)
        net.train()
        if ep<=FREEZE: net.bm.eval(); net.bt.eval()
        for fu,tx,m,t2 in tl:
            fu=fu.to(DEV,non_blocking=True).to(memory_format=torch.channels_last)
            tx=tx.to(DEV,non_blocking=True).to(memory_format=torch.channels_last)
            m=m.to(DEV,non_blocking=True); t2=t2.to(DEV,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(fu,tx,m); loss=focal(o,t2)
            if not torch.isfinite(loss): continue
            sc.scale(loss).backward(); sc.unscale_(opt); torch.nn.utils.clip_grad_norm_(net.parameters(),5.0)
            sc.step(opt); sc.update()
        yv,pv=col(va_i,tta=False); au=roc_auc_score(yv,pv) if len(set(yv))>1 else 0
        if au>best: best=au; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        if ni>=5: break
    net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
    yt,pt=col(tei); oofp[tei]=pt; fa.append(roc_auc_score(yt,pt))
    print("  fold "+str(fold)+"  AUC "+format(fa[-1],".4f")+"  ("+format(time.time()-t0,".0f")+"s)")

sub["prob"]=oofp; sub.to_csv(os.path.join(D,"cv_mass_dualpath_oof.csv"),index=False)
thr=max([(balanced_accuracy_score(y,(oofp>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
pred=(oofp>thr).astype(int); cm=confusion_matrix(y,pred,labels=[0,1]); tn,fp,fn,tp=cm.ravel()
# did we catch more smooth cancers?
circ=sub.mass_margins.astype(str).str.contains("CIRCUMSCRIBED",case=False).values
smooth_cancer=(y==1)&circ
caught=((pred==1)&smooth_cancer).sum(); total_sc=smooth_cancer.sum()
print("\n"+"="*62)
print("MASS DUAL-PATHWAY (margin + internal texture)")
print("="*62)
print("  MEAN AUC "+format(np.mean(fa),".4f")+" +/- "+format(np.std(fa),".4f")+"  (single-path was 0.8325)")
print("  acc "+format(accuracy_score(y,pred),".3f")+"  sens "+format(tp/max(tp+fn,1),".3f")+
      "  spec "+format(tn/max(tn+fp,1),".3f"))
print("  FP "+str(fp)+" (was 168)   FN "+str(fn)+" (was 221)")
print("  SMOOTH cancers caught: "+str(caught)+"/"+str(total_sc)+"  (the FN problem we targeted)")
print("="*62)

mass n=1696 | malignant 46.2%

  cached 1696 in 13.3s
  fold 1  AUC 0.7487  (3119s)
  fold 2  AUC 0.8116  (3242s)
  fold 3  AUC 0.8123  (3241s)
  fold 4  AUC 0.8294  (2854s)
  fold 5  AUC 0.8060  (1490s)

MASS DUAL-PATHWAY (margin + internal texture)
  MEAN AUC 0.8016 +/- 0.0276  (single-path was 0.8325)
  acc 0.735  sens 0.704  spec 0.761
  FP 218 (was 168)   FN 232 (was 221)
  SMOOTH cancers caught: 30/62  (the FN problem we targeted)


**`BCF` cell 352** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════
# DIAGNOSTIC #6: overfitting or ceiling? — train acc vs val acc gap
#   trains ONE fold quickly, prints train AND val accuracy each epoch
#   if train>>val = overfitting (augmentation/regularization can help)
#   if train≈val  = ceiling (tweaks won't help, it's the data)
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
LESION="mass"   # start with mass
CSV="cbis_mass_fixed.csv" if LESION=="mass" else "cbis_calc_fixed.csv"
S=512; BATCH=12; MULT=8; EPOCHS=18; LR_HEAD=1e-3; LR_FT=1e-5; FREEZE=3; GAMMA=2.0
torch.backends.cudnn.benchmark=True; torch.backends.cuda.matmul.allow_tf32=True
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
MEAN=np.array([0.485,0.456,0.406],np.float32); STD=np.array([0.229,0.224,0.225],np.float32)

sub=pd.read_csv(os.path.join(D,CSV)); sub["label"]=sub["label"].astype(int)
sub["assessment"]=pd.to_numeric(sub["assessment"],errors="coerce")
sub=sub.dropna(subset=["img","label"]).reset_index(drop=True)
print(LESION+": n="+str(len(sub))+"\n")

CACHE={}
for _,r in sub.iterrows():
    k=r["img"]
    if k in CACHE: continue
    img=cv2.imread(k,cv2.IMREAD_GRAYSCALE); img=cv2.resize(img if img is not None else np.zeros((S,S),np.uint8),(S,S))
    mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    mask=(cv2.resize(mk,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.float32) if mk is not None else np.zeros((S,S),np.float32)
    CACHE[k]=(_clahe.apply(img),mask)
print("cached "+str(len(CACHE)))

class DS(Dataset):
    def __init__(s,idx,aug,mult=1):
        s.idx=np.array(idx); s.aug=aug; s.mult=mult if aug else 1
    def __len__(s): return len(s.idx)*s.mult
    def __getitem__(s,i):
        j=s.idx[i%len(s.idx)]; k=i//len(s.idx); r=sub.iloc[j]
        img,mask=CACHE[r["img"]]; img=img.copy(); mask=mask.copy()
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); mask=np.fliplr(mask)
            elif k%8==2: img=np.flipud(img); mask=np.flipud(mask)
            elif k%8==3: img=np.rot90(img,1); mask=np.rot90(mask,1)
            elif k%8==4: img=np.rot90(img,2); mask=np.rot90(mask,2)
            elif k%8==5: img=np.rot90(img,3); mask=np.rot90(mask,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((S/2,S/2),np.random.uniform(-15,15),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(S,S),borderMode=cv2.BORDER_REFLECT); mask=cv2.warpAffine(mask,M,(S,S),flags=cv2.INTER_NEAREST)
            elif k%8==7: img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
        im=np.ascontiguousarray(img).astype(np.float32)/255.
        x=np.stack([im,im,im],0); x=((x.transpose(1,2,0)-MEAN)/STD).transpose(2,0,1).astype(np.float32)
        return (torch.from_numpy(np.ascontiguousarray(x)),torch.from_numpy(np.ascontiguousarray(mask))[None],torch.tensor(int(r["label"])))

class Net(nn.Module):
    def __init__(s):
        super().__init__()
        try: dn=models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception: dn=models.densenet121(weights=None)
        s.b=dn.features
        s.head=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Dropout(0.5),nn.Linear(256,2))
    def forward(s,x,mask):
        f=F.relu(s.b(x))
        m=F.interpolate(mask,size=f.shape[2:],mode="bilinear",align_corners=False)
        w=1.0+2.0*m; f=f*w; g=(f.sum((2,3))/(w.sum((2,3))+1e-6))
        return s.head(g)

y=sub.label.values; gr=sub.patient_id.values
strat=sub["label"].astype(str)+"_"+sub.assessment.isin([3,4]).astype(int).astype(str)
tri,tei=next(iter(StratifiedGroupKFold(5,shuffle=True,random_state=42).split(sub,strat,gr)))
n0=float((y[tri]==0).sum()); n1=float((y[tri]==1).sum()); al=torch.tensor([n1/(n0+n1),n0/(n0+n1)],device=DEV)
def focal(lo,t):
    ce=F.cross_entropy(lo.float(),t,weight=al,reduction="none"); pt=torch.exp(-ce); return ((1-pt)**GAMMA*ce).mean()
torch.manual_seed(1); np.random.seed(1)
net=Net().to(DEV).to(memory_format=torch.channels_last)
for p_ in net.b.parameters(): p_.requires_grad=False
sc=torch.amp.GradScaler(); opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=1e-3)
tl=DataLoader(DS(tri,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
@torch.no_grad()
def evalacc(idx):
    net.eval(); ld=DataLoader(DS(idx,False),batch_size=20,shuffle=False,num_workers=0); ps=[];ys=[]
    for x,m,t in ld:
        x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x,m)
        ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy()); ys+=list(t.numpy())
    ps=np.array(ps); ys=np.array(ys); return accuracy_score(ys,(ps>0.5).astype(int)), roc_auc_score(ys,ps)

print("\nepoch | train_acc | test_acc | test_auc  | gap")
print("-"*50)
for ep in range(1,EPOCHS+1):
    if ep==FREEZE+1:
        for p_ in net.b.parameters(): p_.requires_grad=True
        opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=1e-3)
    net.train()
    if ep<=FREEZE: net.b.eval()
    for x,m,t in tl:
        x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV); t=t.to(DEV)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): o=net(x,m); loss=focal(o,t)
        if not torch.isfinite(loss): continue
        sc.scale(loss).backward(); sc.unscale_(opt); torch.nn.utils.clip_grad_norm_(net.parameters(),5.0); sc.step(opt); sc.update()
    tr_acc,_=evalacc(tri[:600]); te_acc,te_auc=evalacc(tei)
    print("  "+str(ep).rjust(2)+"   |   "+format(100*tr_acc,".1f")+"%  |  "+format(100*te_acc,".1f")+
          "%  |  "+format(te_auc,".3f")+"  | "+format(100*(tr_acc-te_acc),"+.1f")+"%")
print("\nif gap (train-test) > 15% = OVERFITTING (aug/regularization can help)")
print("if gap < 8% = at CAPACITY/DATA ceiling (tweaks won't move it much)")

mass: n=1696

cached 1696

epoch | train_acc | test_acc | test_auc  | gap
--------------------------------------------------
   1   |   58.0%  |  60.2%  |  0.703  | -2.2%
   2   |   72.3%  |  64.0%  |  0.696  | +8.3%
   3   |   70.3%  |  64.0%  |  0.711  | +6.3%
   4   |   72.7%  |  62.8%  |  0.699  | +9.8%
   5   |   81.0%  |  67.0%  |  0.742  | +14.0%
   6   |   89.5%  |  65.2%  |  0.724  | +24.3%
   7   |   96.0%  |  67.8%  |  0.732  | +28.2%
   8   |   97.5%  |  69.9%  |  0.764  | +27.6%
   9   |   99.7%  |  70.2%  |  0.769  | +29.5%
  10   |   99.2%  |  64.9%  |  0.747  | +34.3%
  11   |   100.0%  |  72.0%  |  0.766  | +28.0%
  12   |   100.0%  |  64.9%  |  0.727  | +35.1%
  13   |   100.0%  |  69.0%  |  0.757  | +31.0%
  14   |   100.0%  |  66.1%  |  0.744  | +33.9%
  15   |   100.0%  |  67.3%  |  0.754  | +32.7%
  16   |   100.0%  |  67.3%  |  0.742  | +32.7%
  17   |   100.0%  |  68.1%  |  0.749  | +31.9%
  18   |   100.0%  |  69.3%  |  0.756  | +30.7%

if gap (train-test) > 15

**`BCF` cell 353** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════
# ANTI-OVERFIT DIAGNOSTIC — does stronger reg + gentle aug close the gap?
#   ONE fold, prints train_acc vs test_acc each epoch
#   compare the gap to the +32% we saw before
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
LESION="mass"; CSV="cbis_mass_fixed.csv"
S=512; BATCH=12; MULT=6; EPOCHS=18; LR_HEAD=1e-3; LR_FT=1e-5; FREEZE=3; GAMMA=2.0
WD=5e-3; DROP=0.6; LABEL_SMOOTH=0.1   # stronger regularization
torch.backends.cudnn.benchmark=True; torch.backends.cuda.matmul.allow_tf32=True
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
MEAN=np.array([0.485,0.456,0.406],np.float32); STD=np.array([0.229,0.224,0.225],np.float32)

sub=pd.read_csv(os.path.join(D,CSV)); sub["label"]=sub["label"].astype(int)
sub["assessment"]=pd.to_numeric(sub["assessment"],errors="coerce")
sub=sub.dropna(subset=["img","label"]).reset_index(drop=True)
print(LESION+": stronger reg (dropout "+str(DROP)+", wd "+str(WD)+", smooth "+str(LABEL_SMOOTH)+") + GENTLE aug\n")

CACHE={}
for _,r in sub.iterrows():
    k=r["img"]
    if k in CACHE: continue
    img=cv2.imread(k,cv2.IMREAD_GRAYSCALE); img=cv2.resize(img if img is not None else np.zeros((S,S),np.uint8),(S,S))
    mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    mask=(cv2.resize(mk,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.float32) if mk is not None else np.zeros((S,S),np.float32)
    CACHE[k]=(_clahe.apply(img),mask)
print("cached "+str(len(CACHE)))

class DS(Dataset):
    def __init__(s,idx,aug,mult=1):
        s.idx=np.array(idx); s.aug=aug; s.mult=mult if aug else 1
    def __len__(s): return len(s.idx)*s.mult
    def __getitem__(s,i):
        j=s.idx[i%len(s.idx)]; k=i//len(s.idx); r=sub.iloc[j]
        img,mask=CACHE[r["img"]]; img=img.copy(); mask=mask.copy()
        # GENTLE aug: horizontal flip, small rotation, brightness, translation ONLY
        if s.aug and k>0:
            if   k%6==1: img=np.fliplr(img); mask=np.fliplr(mask)
            elif k%6==2:
                M=cv2.getRotationMatrix2D((S/2,S/2),np.random.uniform(-15,15),1.0)
                img=cv2.warpAffine(img,M,(S,S),borderMode=cv2.BORDER_REFLECT); mask=cv2.warpAffine(mask,M,(S,S),flags=cv2.INTER_NEAREST)
            elif k%6==3:
                M=cv2.getRotationMatrix2D((S/2,S/2),np.random.uniform(-10,10),np.random.uniform(.95,1.05))
                img=cv2.warpAffine(img,M,(S,S),borderMode=cv2.BORDER_REFLECT); mask=cv2.warpAffine(mask,M,(S,S),flags=cv2.INTER_NEAREST)
            elif k%6==4: img=np.clip(img.astype(np.float32)*np.random.uniform(.9,1.1),0,255).astype(np.uint8)
            elif k%6==5:
                tx,ty=np.random.randint(-20,20,2); M=np.float32([[1,0,tx],[0,1,ty]])
                img=cv2.warpAffine(img,M,(S,S),borderMode=cv2.BORDER_REFLECT); mask=cv2.warpAffine(mask,M,(S,S),flags=cv2.INTER_NEAREST)
        im=np.ascontiguousarray(img).astype(np.float32)/255.
        x=np.stack([im,im,im],0); x=((x.transpose(1,2,0)-MEAN)/STD).transpose(2,0,1).astype(np.float32)
        return (torch.from_numpy(np.ascontiguousarray(x)),torch.from_numpy(np.ascontiguousarray(mask))[None],torch.tensor(int(r["label"])))

class Net(nn.Module):
    def __init__(s):
        super().__init__()
        try: dn=models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception: dn=models.densenet121(weights=None)
        s.b=dn.features
        s.head=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Dropout(DROP),nn.Linear(256,2))
    def forward(s,x,mask):
        f=F.relu(s.b(x))
        m=F.interpolate(mask,size=f.shape[2:],mode="bilinear",align_corners=False)
        w=1.0+2.0*m; f=f*w; g=(f.sum((2,3))/(w.sum((2,3))+1e-6))
        return s.head(g)

y=sub.label.values; gr=sub.patient_id.values
strat=sub["label"].astype(str)+"_"+sub.assessment.isin([3,4]).astype(int).astype(str)
tri,tei=next(iter(StratifiedGroupKFold(5,shuffle=True,random_state=42).split(sub,strat,gr)))
n0=float((y[tri]==0).sum()); n1=float((y[tri]==1).sum()); al=torch.tensor([n1/(n0+n1),n0/(n0+n1)],device=DEV)
def focal(lo,t):
    ce=F.cross_entropy(lo.float(),t,weight=al,reduction="none",label_smoothing=LABEL_SMOOTH)
    pt=torch.exp(-ce); return ((1-pt)**GAMMA*ce).mean()
torch.manual_seed(1); np.random.seed(1)
net=Net().to(DEV).to(memory_format=torch.channels_last)
for p_ in net.b.parameters(): p_.requires_grad=False
sc=torch.amp.GradScaler(); opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=WD)
tl=DataLoader(DS(tri,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
@torch.no_grad()
def evalacc(idx):
    net.eval(); ld=DataLoader(DS(idx,False),batch_size=20,shuffle=False,num_workers=0); ps=[];ys=[]
    for x,m,t in ld:
        x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x,m)
        ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy()); ys+=list(t.numpy())
    ps=np.array(ps); ys=np.array(ys); return accuracy_score(ys,(ps>0.5).astype(int)), roc_auc_score(ys,ps)

print("\nepoch | train_acc | test_acc | test_auc | gap   (was +32% before)")
print("-"*58)
best_auc=0
for ep in range(1,EPOCHS+1):
    if ep==FREEZE+1:
        for p_ in net.b.parameters(): p_.requires_grad=True
        opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=WD)
    net.train()
    if ep<=FREEZE: net.b.eval()
    for x,m,t in tl:
        x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV); t=t.to(DEV)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): o=net(x,m); loss=focal(o,t)
        if not torch.isfinite(loss): continue
        sc.scale(loss).backward(); sc.unscale_(opt); torch.nn.utils.clip_grad_norm_(net.parameters(),5.0); sc.step(opt); sc.update()
    tr_acc,_=evalacc(tri[:600]); te_acc,te_auc=evalacc(tei); best_auc=max(best_auc,te_auc)
    print("  "+str(ep).rjust(2)+"   |   "+format(100*tr_acc,".1f")+"%  |  "+format(100*te_acc,".1f")+
          "%  |  "+format(te_auc,".3f")+"  | "+format(100*(tr_acc-te_acc),"+.1f")+"%")
print("\nbest test AUC this fold: "+format(best_auc,".3f"))
print("compare gap to the +32% from before. if gap is now <20%, reg is working.")
print("if best AUC still ~0.77 on this fold, the ceiling holds despite less overfit.")

mass: stronger reg (dropout 0.6, wd 0.005, smooth 0.1) + GENTLE aug

cached 1696

epoch | train_acc | test_acc | test_auc | gap   (was +32% before)
----------------------------------------------------------
   1   |   68.3%  |  65.2%  |  0.691  | +3.1%
   2   |   63.3%  |  63.1%  |  0.705  | +0.2%
   3   |   70.8%  |  64.6%  |  0.699  | +6.2%
   4   |   72.0%  |  63.1%  |  0.705  | +8.9%
   5   |   83.7%  |  64.6%  |  0.713  | +19.1%
   6   |   92.5%  |  66.4%  |  0.738  | +26.1%
   7   |   88.3%  |  62.2%  |  0.777  | +26.1%
   8   |   99.3%  |  69.0%  |  0.749  | +30.3%
   9   |   100.0%  |  68.1%  |  0.761  | +31.9%
  10   |   100.0%  |  68.7%  |  0.753  | +31.3%
  11   |   100.0%  |  67.3%  |  0.745  | +32.7%
  12   |   100.0%  |  66.4%  |  0.760  | +33.6%
  13   |   100.0%  |  69.0%  |  0.749  | +31.0%
  14   |   100.0%  |  67.6%  |  0.750  | +32.4%
  15   |   100.0%  |  67.0%  |  0.732  | +33.0%
  16   |   100.0%  |  67.8%  |  0.741  | +32.2%
  17   |   100.0%  |  67.8%  |  0.776

**`BCF` cell 354** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [3]:
# ══════════════════════════════════════════════════════════════════════
# UNTESTED LEVERS: input size (384 vs 512) x attention weight (1x vs 2x)
#   one fold each, prints train/test gap + best test AUC
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"; DEV=torch.device("cuda")
CSV="cbis_mass_fixed.csv"
BATCH=12; MULT=6; EPOCHS=14; LR_HEAD=1e-3; LR_FT=1e-5; FREEZE=3; GAMMA=2.0; WD=1e-3; DROP=0.5
torch.backends.cudnn.benchmark=True; torch.backends.cuda.matmul.allow_tf32=True
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
MEAN=np.array([0.485,0.456,0.406],np.float32); STD=np.array([0.229,0.224,0.225],np.float32)

sub=pd.read_csv(os.path.join(D,CSV)); sub["label"]=sub["label"].astype(int)
sub["assessment"]=pd.to_numeric(sub["assessment"],errors="coerce")
sub=sub.dropna(subset=["img","label"]).reset_index(drop=True)
y=sub.label.values; gr=sub.patient_id.values
strat=sub["label"].astype(str)+"_"+sub.assessment.isin([3,4]).astype(int).astype(str)
tri,tei=next(iter(StratifiedGroupKFold(5,shuffle=True,random_state=42).split(sub,strat,gr)))

def run(S, ATT):
    CACHE={}
    for _,r in sub.iterrows():
        k=r["img"]
        if k in CACHE: continue
        img=cv2.imread(k,cv2.IMREAD_GRAYSCALE); img=cv2.resize(img if img is not None else np.zeros((S,S),np.uint8),(S,S))
        mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        mask=(cv2.resize(mk,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.float32) if mk is not None else np.zeros((S,S),np.float32)
        CACHE[k]=(_clahe.apply(img),mask)

    class DS(Dataset):
        def __init__(s,idx,aug,mult=1):
            s.idx=np.array(idx); s.aug=aug; s.mult=mult if aug else 1
        def __len__(s): return len(s.idx)*s.mult
        def __getitem__(s,i):
            j=s.idx[i%len(s.idx)]; k=i//len(s.idx); r=sub.iloc[j]
            img,mask=CACHE[r["img"]]; img=img.copy(); mask=mask.copy()
            if s.aug and k>0:
                if   k%6==1: img=np.fliplr(img); mask=np.fliplr(mask)
                elif k%6==2:
                    M=cv2.getRotationMatrix2D((S/2,S/2),np.random.uniform(-15,15),1.0)
                    img=cv2.warpAffine(img,M,(S,S),borderMode=cv2.BORDER_REFLECT); mask=cv2.warpAffine(mask,M,(S,S),flags=cv2.INTER_NEAREST)
                elif k%6==3:
                    M=cv2.getRotationMatrix2D((S/2,S/2),np.random.uniform(-10,10),np.random.uniform(.95,1.05))
                    img=cv2.warpAffine(img,M,(S,S),borderMode=cv2.BORDER_REFLECT); mask=cv2.warpAffine(mask,M,(S,S),flags=cv2.INTER_NEAREST)
                elif k%6==4: img=np.clip(img.astype(np.float32)*np.random.uniform(.9,1.1),0,255).astype(np.uint8)
                elif k%6==5:
                    tx,ty=np.random.randint(-int(S*0.04),int(S*0.04),2); M=np.float32([[1,0,tx],[0,1,ty]])
                    img=cv2.warpAffine(img,M,(S,S),borderMode=cv2.BORDER_REFLECT); mask=cv2.warpAffine(mask,M,(S,S),flags=cv2.INTER_NEAREST)
            im=np.ascontiguousarray(img).astype(np.float32)/255.
            x=np.stack([im,im,im],0); x=((x.transpose(1,2,0)-MEAN)/STD).transpose(2,0,1).astype(np.float32)
            return (torch.from_numpy(np.ascontiguousarray(x)),torch.from_numpy(np.ascontiguousarray(mask))[None],torch.tensor(int(r["label"])))

    class Net(nn.Module):
        def __init__(s):
            super().__init__()
            try: dn=models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
            except Exception: dn=models.densenet121(weights=None)
            s.b=dn.features
            s.head=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Dropout(DROP),nn.Linear(256,2))
        def forward(s,x,mask):
            f=F.relu(s.b(x))
            m=F.interpolate(mask,size=f.shape[2:],mode="bilinear",align_corners=False)
            w=1.0+ATT*m; f=f*w; g=(f.sum((2,3))/(w.sum((2,3))+1e-6))
            return s.head(g)

    n0=float((y[tri]==0).sum()); n1=float((y[tri]==1).sum()); al=torch.tensor([n1/(n0+n1),n0/(n0+n1)],device=DEV)
    def focal(lo,t):
        ce=F.cross_entropy(lo.float(),t,weight=al,reduction="none"); pt=torch.exp(-ce); return ((1-pt)**GAMMA*ce).mean()
    torch.manual_seed(1); np.random.seed(1)
    net=Net().to(DEV).to(memory_format=torch.channels_last)
    for p_ in net.b.parameters(): p_.requires_grad=False
    sc=torch.amp.GradScaler(); opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=WD)
    tl=DataLoader(DS(tri,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
    @torch.no_grad()
    def ev(idx):
        net.eval(); ld=DataLoader(DS(idx,False),batch_size=20,shuffle=False,num_workers=0); ps=[];ys=[]
        for x,m,t in ld:
            x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
            with torch.amp.autocast(device_type="cuda"): o=net(x,m)
            ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy()); ys+=list(t.numpy())
        ps=np.array(ps); ys=np.array(ys); return accuracy_score(ys,(ps>0.5).astype(int)), roc_auc_score(ys,ps)
    best=0; bgap=0
    for ep in range(1,EPOCHS+1):
        if ep==FREEZE+1:
            for p_ in net.b.parameters(): p_.requires_grad=True
            opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=WD)
        net.train()
        if ep<=FREEZE: net.b.eval()
        for x,m,t in tl:
            x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV); t=t.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(x,m); loss=focal(o,t)
            if not torch.isfinite(loss): continue
            sc.scale(loss).backward(); sc.unscale_(opt); torch.nn.utils.clip_grad_norm_(net.parameters(),5.0); sc.step(opt); sc.update()
        tra,_=ev(tri[:500]); tea,teauc=ev(tei)
        if teauc>best: best=teauc; bgap=tra-tea
    return best,bgap

print("config              best_test_AUC   gap_at_best")
print("-"*50)
for S,ATT in [(512,2.0),(384,2.0),(384,1.0),(512,1.0)]:
    t0=time.time(); auc,gap=run(S,ATT)
    print(str(S)+"px, w=1+"+str(ATT)+"m   "+format(auc,".4f")+"        "+format(100*gap,"+.1f")+"%   ("+format(time.time()-t0,".0f")+"s)")
print("\nbaseline fold-1 AUC at 512px/2.0 was ~0.77-0.83")

config              best_test_AUC   gap_at_best
--------------------------------------------------
512px, w=1+2.0m   0.7810        +31.2%   (1321s)
384px, w=1+2.0m   0.7788        +32.0%   (951s)
384px, w=1+1.0m   0.7818        +30.7%   (825s)
512px, w=1+1.0m   0.7814        +30.5%   (1192s)

baseline fold-1 AUC at 512px/2.0 was ~0.77-0.83


## D · Decision-layer analyses


**`SEG` cell 3** — ══════════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [3]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 13 — WHERE DOES THE DECISION-LAYER GAIN COME FROM?
#   A  per-category decomposition of the decision changes
#   B  evaluation on non-BI-RADS-5 cases (thresholds fitted on everything)
#   C  BI-RADS 5 removed from train AND test, both systems refitted
#   D  paired bootstrap CI on the accuracy GAIN
#   E  how many malignant BI-RADS 5 cases the 0.01 threshold rescued
#   No GPU. ~2 min.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]
FLOOR=0.90; GRID=np.round(np.arange(0.01,0.995,0.005),3)
MIN_N,MIN_POS,PASSES=20,3,15; RNG=np.random.default_rng(7); NB=3000

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
d["y"]=d.label.astype(int)
d["a"]=pd.to_numeric(d.assessment,errors="coerce").fillna(4).astype(int).clip(0,5)
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")
def load(fn):
    m=pd.read_csv(os.path.join(D,fn)); m["_k"]=m["img"].map(stem)
    return m[m["_k"].isin(set(d._k))].drop_duplicates("_k").set_index("_k")["prob"]
PTR,PTE=load("cv_mass_twostream_officialtrain_oof.csv"),load("cv_mass_twostream_officialsplit_oof.csv")
TR=d[d.sp.eq("train")&d._k.isin(PTR.index)].copy(); TR["p"]=PTR.loc[TR._k].values
TE=d[d.sp.eq("test") &d._k.isin(PTE.index)].copy(); TE["p"]=PTE.loc[TE._k].values

def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_global(y,p,fl):
    P,N=int(y.sum()),len(y); tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N,tp/max(P,1); ok=se>=fl-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])
def _asc(y,p,a,init,fl):
    N,P=len(y),int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),
                 np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    TPc=int(((yh==1)&(y==1)).sum()); TNc=int(((yh==0)&(y==0)).sum())
    if TPc/max(P,1)<fl-1e-12: return tau,-1.0
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bTP=TPc-int((cur&(y[i]==1)).sum()); bTN=TNc-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bTP+tg+bTN+ng)/N; se=(bTP+tg)/max(P,1); fe=se>=fl-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j]>(TPc+TNc)/N+1e-12:
                tau[g]=float(GRID[j]); TPc,TNc=int(bTP+tg[j]),int(bTN+ng[j]); mv=True
        if not mv: break
    return tau,(TPc+TNc)/N
def fit_bir(y,p,a,fl):
    g0=fit_global(y,p,fl); cats=[int(c) for c in np.unique(a)]
    st=[{g:v for g in cats} for v in (g0,.05,.15,.25,.35,.45,.50,.55,.65,max(.01,g0-.1),min(.99,g0+.1))]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.05,.8)) for g in cats} for _ in range(6)]
    best,ba=None,-np.inf
    for s in st:
        t,ac=_asc(y,p,a,s,fl)
        if ac>ba+1e-12: best,ba=t,ac
    return (best if best is not None and ba>=0 else {g:g0 for g in cats}), g0
ap=lambda p,a,t,g0:(p>=np.array([t.get(int(g),g0) for g in a])).astype(int)

def agg(t,key):
    if key is None: return t.reset_index(drop=True)
    g=t.groupby(key,sort=True)
    o=g.agg(p=("p","mean"),y=("y","max"),a=("a","max")).reset_index()
    o["patient_id"]=g["patient_id"].first().values; return o

for LVL,KEY in (("ROI",None),("LESION","lesion_key")):
    tr,te=agg(TR,KEY),agg(TE,KEY)
    ytr,ptr,atr=tr.y.values.astype(int),tr.p.values,tr.a.values.astype(int)
    yte,pte,ate=te.y.values.astype(int),te.p.values,te.a.values.astype(int)
    pat=te.patient_id.values
    g1=fit_global(ytr,ptr,FLOOR); tau,g0=fit_bir(ytr,ptr,atr,FLOOR)
    yh1=(pte>=g1).astype(int); yh2=ap(pte,ate,tau,g0)
    print("\n"+"#"*84); print("# %s LEVEL — n=%d | global threshold %.3f | per-category %s"
          % (LVL,len(te),g1,{int(k):round(v,3) for k,v in sorted(tau.items())})); print("#"*84)

    # ── A : per-category decomposition ────────────────────────────────────
    print("\nA. WHERE THE DECISION CHANGES HAPPEN")
    print("  %-8s %-5s %-6s %-8s %-9s %-9s %-9s %s"
          % ("BI-RADS","n","malig","thresh","flipped","newly OK","newly bad","net correct"))
    tot=0
    for g in sorted(set(ate)):
        m=ate==g; n=int(m.sum())
        fl_=int((yh1[m]!=yh2[m]).sum())
        ok1=(yh1[m]==yte[m]); ok2=(yh2[m]==yte[m])
        gain=int((~ok1&ok2).sum()); loss=int((ok1&~ok2).sum()); tot+=gain-loss
        print("  %-8d %-5d %-6d %-8.3f %-9d %-9d %-9d %+d"
              % (g,n,int(yte[m].sum()),tau.get(int(g),g0),fl_,gain,loss,gain-loss))
    print("  %-8s %-5d %-6d %-8s %-9d %-9s %-9s %+d  -> %+.2f accuracy points"
          % ("TOTAL",len(te),int(yte.sum()),"",int((yh1!=yh2).sum()),"","",tot,100*tot/len(te)))

    # ── E : the BI-RADS 5 rescue count ────────────────────────────────────
    m5=ate==5
    resc=int(((yh1==0)&(yh2==1)&(yte==1)&m5).sum())
    hurt=int(((yh1==0)&(yh2==1)&(yte==0)&m5).sum())
    print("\nE. BI-RADS 5 SPECIFICALLY")
    print("   n=%d, malignant %d (%.0f%%)" % (int(m5.sum()),int(yte[m5].sum()),100*yte[m5].mean()))
    print("   malignant cases the 0.01 threshold RESCUED from the global cut : %d" % resc)
    print("   benign cases it newly called malignant                         : %d" % hurt)
    print("   net correct from BI-RADS 5 alone                               : %+d" % (resc-hurt))

    # ── B : evaluate on non-BI-RADS-5 only, thresholds fitted on everything
    keep=~m5
    a1=(yh1[keep]==yte[keep]).mean(); a2=(yh2[keep]==yte[keep]).mean()
    print("\nB. EVALUATED ON NON-BI-RADS-5 ONLY (thresholds fitted on all data)")
    print("   n=%d   global %.4f   per-category %.4f   gain %+.2f points"
          % (int(keep.sum()),a1,a2,100*(a2-a1)))

    # ── C : BI-RADS 5 removed from train AND test, both refitted ──────────
    ktr=atr!=5
    if ktr.sum()>50 and len(np.unique(ytr[ktr]))>1:
        g1c=fit_global(ytr[ktr],ptr[ktr],FLOOR)
        tauc,g0c=fit_bir(ytr[ktr],ptr[ktr],atr[ktr],FLOOR)
        h1=(pte[keep]>=g1c).astype(int); h2=ap(pte[keep],ate[keep],tauc,g0c)
        c1=(h1==yte[keep]).mean(); c2=(h2==yte[keep]).mean()
        print("\nC. BI-RADS 5 REMOVED FROM TRAIN AND TEST, BOTH SYSTEMS REFITTED")
        print("   thresholds %s" % {int(k):round(v,3) for k,v in sorted(tauc.items())})
        print("   n=%d   global %.4f   per-category %.4f   gain %+.2f points"
              % (int(keep.sum()),c1,c2,100*(c2-c1)))

    # ── D : paired bootstrap CI on the GAIN ───────────────────────────────
    print("\nD. PAIRED BOOTSTRAP CI ON THE ACCURACY GAIN (clustered by patient)")
    ps=np.unique(pat)
    for lab,sel in (("all cases",np.ones(len(te),bool)),("excluding BI-RADS 5",keep)):
        o=[]
        for _ in range(NB):
            ix=np.concatenate([np.where(pat==q)[0] for q in RNG.choice(ps,len(ps),True)])
            ix=ix[sel[ix]]
            if len(ix)<20: continue
            o.append((yh2[ix]==yte[ix]).mean()-(yh1[ix]==yte[ix]).mean())
        o=np.array(o)
        print("   %-22s gain %+.2f pts  95%% CI [%+.2f, %+.2f]  P(gain>0) = %.3f"
              % (lab,100*o.mean(),100*np.percentile(o,2.5),100*np.percentile(o,97.5),(o>0).mean()))


####################################################################################
# ROI LEVEL — n=378 | global threshold 0.440 | per-category {0: 0.515, 1: 0.44, 2: 0.44, 3: 0.495, 4: 0.455, 5: 0.01}
####################################################################################

A. WHERE THE DECISION CHANGES HAPPEN
  BI-RADS  n     malig  thresh   flipped   newly OK  newly bad net correct
  0        33    3      0.515    5         5         0         +5
  1        2     2      0.440    0         0         0         +0
  2        14    1      0.440    0         0         0         +0
  3        85    4      0.495    13        12        1         +11
  4        169   67     0.455    7         5         2         +3
  5        75    70     0.010    3         2         1         +1
  TOTAL    378   147             28                            +20  -> +5.29 accuracy points

E. BI-RADS 5 SPECIFICALLY
   n=75, malignant 70 (93%)
   malignant cases the 0.01 threshold RESCUED from th

**`SEG` cell 4** — ══════════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [4]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 14 — PAIRED CI: your layer (S2) versus the fusion baseline (S3)
#   Same 0.90 sensitivity constraint. Clustered bootstrap by patient.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]
FLOOR=0.90; GRID=np.round(np.arange(0.01,0.995,0.005),3)
MIN_N,MIN_POS,PASSES=20,3,15; RNG=np.random.default_rng(7); NB=3000
LOGIT=lambda p: np.log(np.clip(p,1e-6,1-1e-6)/(1-np.clip(p,1e-6,1-1e-6)))

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
for c in ("density","subtlety","side"):
    if c not in d.columns:
        fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
        d=d.merge(fx[["_k",c]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d.label.astype(int)
d["a"]=pd.to_numeric(d.assessment,errors="coerce").fillna(4).astype(int).clip(0,5)
d["ds"]=pd.to_numeric(d.density,errors="coerce");  d["ds"]=d.ds.fillna(d.ds.median())
d["sb"]=pd.to_numeric(d.subtlety,errors="coerce"); d["sb"]=d.sb.fillna(d.sb.median())
d["breast_key"]=d.patient_id.astype(str)+"_"+d["side"].astype(str)
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")
def load(fn):
    m=pd.read_csv(os.path.join(D,fn)); m["_k"]=m["img"].map(stem)
    return m[m["_k"].isin(set(d._k))].drop_duplicates("_k").set_index("_k")["prob"]
TR=d[d.sp.eq("train")].copy(); TR["p"]=load("cv_mass_twostream_officialtrain_oof.csv").loc[TR._k].values
TE=d[d.sp.eq("test")].copy();  TE["p"]=load("cv_mass_twostream_officialsplit_oof.csv").loc[TE._k].values

def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_global(y,p,fl):
    P,N=int(y.sum()),len(y); tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N,tp/max(P,1); ok=se>=fl-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])
def _asc(y,p,a,init,fl):
    N,P=len(y),int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),
                 np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    A=int(((yh==1)&(y==1)).sum()); B=int(((yh==0)&(y==0)).sum())
    if A/max(P,1)<fl-1e-12: return tau,-1.0
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bA=A-int((cur&(y[i]==1)).sum()); bB=B-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bA+tg+bB+ng)/N; se=(bA+tg)/max(P,1); fe=se>=fl-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j]>(A+B)/N+1e-12:
                tau[g]=float(GRID[j]); A,B=int(bA+tg[j]),int(bB+ng[j]); mv=True
        if not mv: break
    return tau,(A+B)/N
def fit_bir(y,p,a,fl):
    g0=fit_global(y,p,fl); cats=[int(c) for c in np.unique(a)]
    st=[{g:v for g in cats} for v in (g0,.05,.15,.25,.35,.45,.50,.55,.65,max(.01,g0-.1),min(.99,g0+.1))]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.05,.8)) for g in cats} for _ in range(6)]
    best,ba=None,-np.inf
    for s in st:
        t,ac=_asc(y,p,a,s,fl)
        if ac>ba+1e-12: best,ba=t,ac
    return (best if best is not None and ba>=0 else {g:g0 for g in cats}), g0
ap=lambda p,a,t,g0:(p>=np.array([t.get(int(g),g0) for g in a])).astype(int)
CATS=[0,1,2,3,4,5]
def fuse(tr,te):
    X=lambda t: np.column_stack([LOGIT(t.p.values)]+
        [(t.a.values==g).astype(float) for g in CATS]+[t.ds.values,t.sb.values])
    Xtr,Xte=X(tr),X(te); mu,sd=Xtr.mean(0),Xtr.std(0)+1e-9
    lr=LogisticRegression(C=0.05,max_iter=5000).fit((Xtr-mu)/sd,tr.y.values)
    return lr.predict_proba((Xtr-mu)/sd)[:,1],lr.predict_proba((Xte-mu)/sd)[:,1]
def roll(t,key):
    if key is None: return t.reset_index(drop=True)
    sp=dict(p=("p","mean"),y=("y","max"),a=("a","max"),ds=("ds","first"),sb=("sb","first"))
    if key!="patient_id": sp["patient_id"]=("patient_id","first")
    o=t.groupby(key,sort=True).agg(**sp).reset_index()
    if "patient_id" not in o.columns: o["patient_id"]=o[key].values
    return o

print("="*88); print("S2 (image-only + per-BI-RADS) vs S3 (fusion + one threshold), floor %.2f" % FLOOR)
print("="*88)
print("  %-8s %-5s %-9s %-9s %-9s %-24s %s" % ("unit","n","S2 acc","S3 acc","diff","95% CI on diff","P(S2>S3)"))
for nm,key in (("ROI",None),("LESION","lesion_key"),("BREAST","breast_key"),("PATIENT","patient_id")):
    tr,te=roll(TR,key),roll(TE,key)
    ytr,ptr,atr=tr.y.values.astype(int),tr.p.values,tr.a.values.astype(int)
    yte,pte,ate=te.y.values.astype(int),te.p.values,te.a.values.astype(int)
    pat=te.patient_id.values
    ztr,zte=fuse(tr,te)
    tau,g0=fit_bir(ytr,ptr,atr,FLOOR); g3=fit_global(ytr,ztr,FLOOR)
    h2=ap(pte,ate,tau,g0); h3=(zte>=g3).astype(int)
    a2,a3=(h2==yte).mean(),(h3==yte).mean()
    ps=np.unique(pat); o=[]
    for _ in range(NB):
        ix=np.concatenate([np.where(pat==q)[0] for q in RNG.choice(ps,len(ps),True)])
        o.append((h2[ix]==yte[ix]).mean()-(h3[ix]==yte[ix]).mean())
    o=np.array(o)
    print("  %-8s %-5d %-9.4f %-9.4f %+-9.4f [%+.4f, %+.4f]        %.3f"
          % (nm,len(te),a2,a3,a2-a3,np.percentile(o,2.5),np.percentile(o,97.5),(o>0).mean()))

S2 (image-only + per-BI-RADS) vs S3 (fusion + one threshold), floor 0.90
  unit     n     S2 acc    S3 acc    diff      95% CI on diff           P(S2>S3)
  ROI      378   0.8095    0.7910    +0.0185   [-0.0099, +0.0477]        0.881
  LESION   223   0.8565    0.8386    +0.0179   [-0.0046, +0.0444]        0.901
  BREAST   210   0.8476    0.8381    +0.0095   [-0.0141, +0.0333]        0.722
  PATIENT  201   0.8507    0.8358    +0.0149   [-0.0100, +0.0448]        0.832


**`SEG` cell 15** — ══════════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 17 — HOW MUCH BI-RADS ACCURACY DOES THE DECISION LAYER NEED?
#
#   Thresholds are fitted on TRAIN using the true BI-RADS categories
#   (available offline, from the dataset). At TEST time the category is
#   deliberately corrupted, simulating a model-predicted category.
#   This answers whether an autonomous version of the layer could work.
#
#   Two corruption models:
#     RANDOM   - a wrong prediction lands on any other category (pessimistic)
#     ADJACENT - a wrong prediction lands on a neighbouring category
#                (realistic: predictors confuse 3 with 4, not 1 with 5)
#
#   No GPU, no training. ~1 min.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]
FLOOR=0.90; GRID=np.round(np.arange(0.01,0.995,0.005),3)
MIN_N,MIN_POS,PASSES=20,3,15
ACCS=[1.00,0.95,0.90,0.85,0.80,0.70,0.60,0.50]; REPS=300
RNG=np.random.default_rng(7)

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
d["y"]=d.label.astype(int)
d["a"]=pd.to_numeric(d.assessment,errors="coerce").fillna(4).astype(int).clip(0,5)
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")
def load(fn):
    m=pd.read_csv(os.path.join(D,fn)); m["_k"]=m["img"].map(stem)
    return m[m["_k"].isin(set(d._k))].drop_duplicates("_k").set_index("_k")["prob"]
TR=d[d.sp.eq("train")].copy(); TR["p"]=load("cv_mass_twostream_officialtrain_oof.csv").loc[TR._k].values
TE=d[d.sp.eq("test")].copy();  TE["p"]=load("cv_mass_twostream_officialsplit_oof.csv").loc[TE._k].values

# ── threshold machinery ───────────────────────────────────────────────────
def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_global(y,p,fl):
    P,N=int(y.sum()),len(y); tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N,tp/max(P,1); ok=se>=fl-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])
def _asc(y,p,a,init,fl):
    N,P=len(y),int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),
                 np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    A=int(((yh==1)&(y==1)).sum()); B=int(((yh==0)&(y==0)).sum())
    if A/max(P,1)<fl-1e-12: return tau,-1.0
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bA=A-int((cur&(y[i]==1)).sum()); bB=B-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bA+tg+bB+ng)/N; se=(bA+tg)/max(P,1); fe=se>=fl-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j]>(A+B)/N+1e-12:
                tau[g]=float(GRID[j]); A,B=int(bA+tg[j]),int(bB+ng[j]); mv=True
        if not mv: break
    return tau,(A+B)/N
def fit_bir(y,p,a,fl):
    g0=fit_global(y,p,fl); cats=[int(c) for c in np.unique(a)]
    st=[{g:v for g in cats} for v in (g0,.05,.15,.25,.35,.45,.50,.55,.65,max(.01,g0-.1),min(.99,g0+.1))]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.05,.8)) for g in cats} for _ in range(6)]
    best,ba=None,-np.inf
    for s in st:
        t,ac=_asc(y,p,a,s,fl)
        if ac>ba+1e-12: best,ba=t,ac
    return (best if best is not None and ba>=0 else {g:g0 for g in cats}), g0
ap=lambda p,a,t,g0:(p>=np.array([t.get(int(g),g0) for g in a])).astype(int)

def roll(t,key):
    if key is None: return t.reset_index(drop=True)
    return t.groupby(key,sort=True).agg(p=("p","mean"),y=("y","max"),a=("a","max")).reset_index()

# ── corruption models ─────────────────────────────────────────────────────
def corrupt(a, acc, cats, mode, rng):
    """with probability (1-acc) replace the category"""
    out=a.copy()
    wrong=rng.random(len(a)) > acc
    idx=np.where(wrong)[0]
    for i in idx:
        if mode=="random":
            opts=[c for c in cats if c!=a[i]]
        else:                                   # adjacent
            opts=[c for c in (a[i]-1,a[i]+1) if c in cats]
            if not opts: opts=[c for c in cats if c!=a[i]]
        if opts: out[i]=rng.choice(opts)
    return out

# ── run ───────────────────────────────────────────────────────────────────
for LVL,KEY in (("ROI",None),("LESION","lesion_key")):
    tr,te=roll(TR,KEY),roll(TE,KEY)
    ytr,ptr,atr=tr.y.values.astype(int),tr.p.values,tr.a.values.astype(int)
    yte,pte,ate=te.y.values.astype(int),te.p.values,te.a.values.astype(int)
    cats=sorted(set(atr)|set(ate))

    g1=fit_global(ytr,ptr,FLOOR)                       # no layer at all
    tau,g0=fit_bir(ytr,ptr,atr,FLOOR)                  # thresholds from TRUE train categories
    acc_global=((pte>=g1).astype(int)==yte).mean()
    acc_perfect=(ap(pte,ate,tau,g0)==yte).mean()

    print("\n"+"#"*86)
    print("# %s LEVEL — n=%d | thresholds %s"
          % (LVL,len(te),{int(k):round(v,3) for k,v in sorted(tau.items())}))
    print("#"*86)
    print("  no decision layer (one global threshold) : %.4f" % acc_global)
    print("  decision layer, TRUE categories          : %.4f   (+%.2f points)"
          % (acc_perfect,100*(acc_perfect-acc_global)))
    print("\n  now corrupting the TEST categories, as a predicted category would be:")
    print("  %-10s %-26s %s" % ("category","RANDOM errors","ADJACENT errors"))
    print("  %-10s %-26s %s" % ("accuracy","accuracy [5th-95th]  gain","accuracy [5th-95th]  gain"))
    breakeven={}
    for acc in ACCS:
        row=["%-10.2f" % acc]
        for mode in ("random","adjacent"):
            rng=np.random.default_rng(11)
            v=[]
            for _ in range(REPS):
                an=corrupt(ate,acc,cats,mode,rng)
                v.append((ap(pte,an,tau,g0)==yte).mean())
            v=np.array(v); m=v.mean()
            row.append("%.4f [%.3f-%.3f] %+5.2f " %
                       (m,np.percentile(v,5),np.percentile(v,95),100*(m-acc_global)))
            if mode not in breakeven and m<=acc_global: breakeven[mode]=acc
        print("  %s %s %s" % (row[0],row[1],row[2]))
    print("\n  break-even (category accuracy below which the layer stops helping):")
    for mode in ("random","adjacent"):
        b=breakeven.get(mode)
        print("    %-10s %s" % (mode, ("~%.2f" % b) if b else "never in the tested range — layer still helps at 0.50"))


######################################################################################
# ROI LEVEL — n=378 | thresholds {0: 0.515, 1: 0.44, 2: 0.44, 3: 0.495, 4: 0.455, 5: 0.01}
######################################################################################
  no decision layer (one global threshold) : 0.7566
  decision layer, TRUE categories          : 0.8095   (+5.29 points)

  now corrupting the TEST categories, as a predicted category would be:
  category   RANDOM errors              ADJACENT errors
  accuracy   accuracy [5th-95th]  gain  accuracy [5th-95th]  gain
  1.00       0.8095 [0.810-0.810] +5.29  0.8095 [0.810-0.810] +5.29 
  0.95       0.8043 [0.796-0.812] +4.77  0.8039 [0.796-0.812] +4.73 
  0.90       0.7985 [0.788-0.807] +4.19  0.7983 [0.788-0.810] +4.17 
  0.85       0.7925 [0.778-0.804] +3.59  0.7932 [0.780-0.807] +3.66 
  0.80       0.7870 [0.772-0.802] +3.04  0.7871 [0.772-0.802] +3.05 
  0.70       0.7755 [0.759-0.794] +1.89  0.7773 [0.759-0.794] +2.07 
  0.

## E · Audits


**`IL2` cell 5** — ══════════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 9 — AUDIT OF THE AGGREGATION CHAIN.  No results, only checks.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
for c in ("side","view"):
    if c not in d.columns:
        fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
        d=d.merge(fx[["_k",c]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d.label.astype(int)
d["a"]=pd.to_numeric(d.assessment,errors="coerce").fillna(4).astype(int).clip(0,5)
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")
d["breast_key"]=d.patient_id.astype(str)+"_"+d["side"].astype(str)

print("="*80); print("1. KEY HYGIENE"); print("="*80)
print("  side values   :", dict(d["side"].value_counts(dropna=False)))
print("  view values   :", dict(d["view"].value_counts(dropna=False)))
print("  side NaN %d | view NaN %d | assessment NaN %d"
      % (d["side"].isna().sum(), d["view"].isna().sum(),
         pd.to_numeric(d.assessment,errors='coerce').isna().sum()))
print("  crop key unique:", d["_k"].is_unique)

print("\n"+"="*80); print("2. UNIT COUNTS"); print("="*80)
for sp in ("train","test"):
    t=d[d.sp==sp]
    print("  %-5s  ROI %4d | lesion %4d | breast %4d | patient %4d | mammograms %4d"
          % (sp,len(t),t.lesion_key.nunique(),t.breast_key.nunique(),
             t.patient_id.nunique(),t.full_path.nunique() if "full_path" in t else -1))
tr,te=set(d[d.sp=="train"].patient_id),set(d[d.sp=="test"].patient_id)
print("  patient overlap %d | lesion_key overlap %d | breast overlap %d"
      % (len(tr&te),
         len(set(d[d.sp=='train'].lesion_key)&set(d[d.sp=='test'].lesion_key)),
         len(set(d[d.sp=='train'].breast_key)&set(d[d.sp=='test'].breast_key))))

print("\n"+"="*80); print("3. VIEWS PER LESION"); print("="*80)
g=d.groupby("lesion_key")
vc=g["view"].apply(lambda s: tuple(sorted(set(str(x).upper()[:3] for x in s))))
print("  view combinations per lesion:", dict(pd.Series(vc).value_counts()))
print("  lesions with >2 crops:", int((g.size()>2).sum()))
print("  lesions whose crops disagree on LABEL   :", int((g.y.nunique()>1).sum()))
print("  lesions whose crops disagree on BI-RADS :", int((g.a.nunique()>1).sum()))
print("  breasts containing >1 lesion :", int((d.groupby('breast_key').lesion_key.nunique()>1).sum()))
print("  patients with both breasts   :", int((d.groupby('patient_id').breast_key.nunique()>1).sum()))
print("  breasts mixing benign+malignant lesions:",
      int((d.groupby('breast_key').y.nunique()>1).sum()))
print("  patients mixing benign+malignant       :",
      int((d.groupby('patient_id').y.nunique()>1).sum()))

print("\n"+"="*80); print("4. HOW MUCH IS DECIDED BY THE BI-RADS 5 RULE ALONE"); print("="*80)
p=pd.read_csv(os.path.join(D,"cv_mass_twostream_officialsplit_oof.csv"))
p["_k"]=p["img"].map(stem)
m=d.merge(p[["_k","prob"]].drop_duplicates("_k"),on="_k").query("sp=='test'")
TAU={"ROI":{0:.515,1:.44,2:.44,3:.495,4:.455,5:.01},
     "LESION":{0:.52,1:.455,2:.455,3:.49,4:.465,5:.01}}
for nm,key in (("ROI",None),("LESION","lesion_key"),("BREAST","breast_key"),("PATIENT","patient_id")):
    if key is None:
        u=m.rename(columns={"prob":"pp"})[["pp","y","a"]]
    else:
        gg=m.groupby(key); u=gg.agg(pp=("prob","mean"),y=("y","max"),a=("a","max"))
    tau=TAU["ROI"] if nm=="ROI" else TAU["LESION"]
    forced=(u.a==5)
    yh=(u.pp>=u.a.map(lambda z: tau.get(int(z),0.45))).astype(int)
    acc_all=(yh==u.y).mean()
    acc_rest=(yh[~forced]==u.y[~forced]).mean() if (~forced).sum() else float("nan")
    print("  %-8s n=%-5d  BI-RADS 5 units %3d (%4.1f%%)  of which truly malignant %3d (%.0f%%)"
          % (nm,len(u),int(forced.sum()),100*forced.mean(),
             int(u.y[forced].sum()), 100*u.y[forced].mean() if forced.sum() else 0))
    print("           accuracy all %.1f%%   |   accuracy excluding BI-RADS 5 units %.1f%%"
          % (100*acc_all,100*acc_rest))
print("\n  If BI-RADS 5 units are a small share AND mostly truly malignant, the rule is")
print("  clinically correct rather than a shortcut. If accuracy collapses once they are")
print("  removed, most of the gain is deference to the radiologist and must be disclosed.")

1. KEY HYGIENE
  side values   : {'RIGHT': np.int64(879), 'LEFT': np.int64(817)}
  view values   : {'MLO': np.int64(912), 'CC': np.int64(784)}
  side NaN 0 | view NaN 0 | assessment NaN 0
  crop key unique: True

2. UNIT COUNTS
  train  ROI 1318 | lesion  782 | breast  722 | patient  691 | mammograms 1231
  test   ROI  378 | lesion  223 | breast  210 | patient  201 | mammograms  361
  patient overlap 0 | lesion_key overlap 0 | breast overlap 0

3. VIEWS PER LESION
  view combinations per lesion: {('CC', 'MLO'): np.int64(691), ('MLO',): np.int64(221), ('CC',): np.int64(93)}
  lesions with >2 crops: 0
  lesions whose crops disagree on LABEL   : 3
  lesions whose crops disagree on BI-RADS : 36
  breasts containing >1 lesion : 48
  patients with both breasts   : 40
  breasts mixing benign+malignant lesions: 5
  patients mixing benign+malignant       : 18

4. HOW MUCH IS DECIDED BY THE BI-RADS 5 RULE ALONE
  ROI      n=378    BI-RADS 5 units  75 (19.8%)  of which truly malignant  70 (93%)
 

**`IL2` cell 8** — ══════════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 10 — IS THE TRAINING-SIDE PREDICTION FILE GENUINELY OUT-OF-FOLD?
#   Four independent diagnostics. No GPU, ~5 seconds.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, brier_score_loss
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
d["y"]=d.label.astype(int)
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")
B=d.set_index("_k")

def load(fn):
    m=pd.read_csv(os.path.join(D,fn)); m["_k"]=m["img"].map(stem)
    m=m[m["_k"].isin(B.index)].drop_duplicates("_k")
    return m.set_index("_k")["prob"]

TR=load("cv_mass_twostream_officialtrain_oof.csv")
TE=load("cv_mass_twostream_officialsplit_oof.csv")

print("="*80); print("DIAGNOSTIC 1 — AUC LEVEL"); print("="*80)
for nm,s in (("train-side",TR),("test-side",TE)):
    y=B.loc[s.index,"y"].values
    print("  %-12s n=%4d   AUC %.4f" % (nm,len(s),roc_auc_score(y,s.values)))
print("  reference: an in-sample DenseNet fit typically scores AUC 0.98-0.999")
print("             an out-of-fold fit scores close to the test AUC")

print("\n"+"="*80); print("DIAGNOSTIC 2 — HOW EXTREME ARE THE PROBABILITIES?"); print("="*80)
print("  %-12s %-10s %-10s %-10s %s" % ("","<0.05","<0.10","| >0.90",">0.95"))
for nm,s in (("train-side",TR),("test-side",TE)):
    v=s.values
    print("  %-12s %-10.1f%% %-10.1f%% %-10.1f%% %.1f%%"
          % (nm,100*(v<.05).mean(),100*(v<.10).mean(),100*(v>.90).mean(),100*(v>.95).mean()))
print("  in-sample predictions collapse to the extremes (typically >70% outside 0.1-0.9).")
print("  if the two rows look alike, the training file is out-of-fold.")

print("\n"+"="*80); print("DIAGNOSTIC 3 — ACCURACY AND CALIBRATION AT 0.5"); print("="*80)
for nm,s in (("train-side",TR),("test-side",TE)):
    y=B.loc[s.index,"y"].values; v=s.values
    print("  %-12s acc@0.5 %.3f   Brier %.4f   mean prob: malignant %.3f | benign %.3f"
          % (nm,((v>=.5).astype(int)==y).mean(),brier_score_loss(y,v),
             v[y==1].mean(),v[y==0].mean()))
print("  in-sample accuracy@0.5 is normally 0.95-1.00 and Brier below 0.05")

print("\n"+"="*80); print("DIAGNOSTIC 4 — DOES A FOLD STRUCTURE EXIST INSIDE OFFICIAL TRAIN?"); print("="*80)
rc=[c for c in d.columns if c.startswith("role_of")]
if not rc:
    print("  no role_of* columns found — cannot confirm the fold structure from the csv")
else:
    trmask=d.sp.eq("train")
    vals=[set(d.loc[trmask & d[c].eq("val"),"_k"]) for c in sorted(rc)]
    union=set().union(*vals); overlap=sum(len(a&b) for i,a in enumerate(vals) for b in vals[i+1:])
    print("  role columns      : %s" % sorted(rc))
    print("  val-set sizes     : %s   (sum %d)" % ([len(v) for v in vals],sum(len(v) for v in vals)))
    print("  official train n  : %d" % int(trmask.sum()))
    print("  val sets overlap  : %d   (0 means they partition cleanly)" % overlap)
    print("  union covers train: %s" % ("YES" if union==set(d.loc[trmask,'_k']) else "NO"))
    if union==set(d.loc[trmask,"_k"]) and overlap==0:
        print("\n  per-fold AUC of the train-side predictions (each fold scored by the model")
        print("  that held it out — these should all be similar, none near 0.99):")
        for c in sorted(rc):
            k=set(d.loc[trmask & d[c].eq("val"),"_k"]) & set(TR.index)
            k=list(k); y=B.loc[k,"y"].values
            if len(set(y))>1:
                print("     %-10s n=%3d  AUC %.4f" % (c,len(k),roc_auc_score(y,TR.loc[k].values)))

print("\n"+"="*80); print("VERDICT"); print("="*80)
ytr=B.loc[TR.index,"y"].values; yte=B.loc[TE.index,"y"].values
a_tr,a_te=roc_auc_score(ytr,TR.values),roc_auc_score(yte,TE.values)
acc_tr=((TR.values>=.5).astype(int)==ytr).mean()
if a_tr>0.97 or acc_tr>0.94:
    print("  *** LOOKS IN-SAMPLE *** train AUC %.4f, acc@0.5 %.3f" % (a_tr,acc_tr))
    print("  Regenerate out-of-fold training predictions before fitting thresholds.")
else:
    print("  OUT-OF-FOLD CONFIRMED.  train AUC %.4f vs test %.4f (gap %+.4f), acc@0.5 %.3f"
          % (a_tr,a_te,a_tr-a_te,acc_tr))
    print("  Too low to be in-sample. The thresholds were fitted on honest held-out")
    print("  predictions, and your procedure is sound as described.")

DIAGNOSTIC 1 — AUC LEVEL
  train-side   n=1318   AUC 0.8974
  test-side    n= 378   AUC 0.8769
  reference: an in-sample DenseNet fit typically scores AUC 0.98-0.999
             an out-of-fold fit scores close to the test AUC

DIAGNOSTIC 2 — HOW EXTREME ARE THE PROBABILITIES?
               <0.05      <0.10      | >0.90    >0.95
  train-side   0.0       % 0.7       % 1.0       % 0.1%
  test-side    0.0       % 0.0       % 1.1       % 0.0%
  in-sample predictions collapse to the extremes (typically >70% outside 0.1-0.9).
  if the two rows look alike, the training file is out-of-fold.

DIAGNOSTIC 3 — ACCURACY AND CALIBRATION AT 0.5
  train-side   acc@0.5 0.827   Brier 0.1521   mean prob: malignant 0.644 | benign 0.363
  test-side    acc@0.5 0.807   Brier 0.1636   mean prob: malignant 0.629 | benign 0.382
  in-sample accuracy@0.5 is normally 0.95-1.00 and Brier below 0.05

DIAGNOSTIC 4 — DOES A FOLD STRUCTURE EXIST INSIDE OFFICIAL TRAIN?
  role columns      : ['role_of0', 'role_of1', 'ro

**`IL2` cell 11** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════
#  DATA INTEGRITY AUDIT  (hardened)
#  ROIs per mammogram | split safety | crop sanity | missing abnormalities
#  CPU, ~30 s. Writes nothing.
# ══════════════════════════════════════════════════════════════════════
import os, re, numpy as np, pandas as pd, cv2
cv2.setNumThreads(0)
D = "/root/autodl-tmp/CBIS"

d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv")).reset_index(drop=True)
d = d.drop(columns=[c for c in ["side", "view", "abn", "pid_f", "mammo", "breast"]
                    if c in d.columns])
b = d["img"].astype(str).map(os.path.basename)
print("sample filenames:")
for x in b.head(5): print("   ", x)

# --- parse patient / side / view / abnormality id ----------------------
PAT = r"(P[_-]\d+)[_-](LEFT|RIGHT)[_-](CC|MLO)(?:[_-](\d+))?"
ex = b.str.extract(PAT, flags=re.I, expand=True)
ex.columns = ["pid_f", "side", "view", "abn"]
for col in ["pid_f", "side", "view"]:
    ex[col] = ex[col].str.upper()
d = pd.concat([d, ex], axis=1)

print("\nparsed  side %d/%d | view %d/%d | abnormality-id %d/%d"
      % (d.side.notna().sum(), len(d), d.view.notna().sum(), len(d),
         d.abn.notna().sum(), len(d)))
if d.view.isna().any():
    print("  UNPARSED examples:", list(b[d.view.isna()].head(5)))
    print("  -> the regex does not match these; paste them to me")

d["mammo"]  = d.pid_f.fillna("?") + "_" + d.side.fillna("?") + "_" + d.view.fillna("?")
d["breast"] = d.pid_f.fillna("?") + "_" + d.side.fillna("?")

# --- Q1: ROIs per full mammogram --------------------------------------
r_per_m = d.groupby("mammo").size()
print("\n" + "=" * 66)
print("Q1  ROIs per full mammogram")
print("=" * 66)
print("  distinct mammograms   :", len(r_per_m))
print("  ROIs per mammogram    :", r_per_m.value_counts().sort_index().to_dict())
multi = r_per_m[r_per_m > 1]
print("  mammograms with >1 ROI: %d (%.1f%%)"
      % (len(multi), 100.0 * len(multi) / max(len(r_per_m), 1)))
print("\n  lesions per breast    :",
      d.groupby("breast")["lesion_key"].nunique().value_counts().sort_index().to_dict())
print("  lesions per patient   :",
      d.groupby("patient_id")["lesion_key"].nunique().value_counts().sort_index().to_dict())
print("  views per lesion      :",
      d.groupby("lesion_key")["view"]
       .apply(lambda s: "+".join(sorted(set(s.dropna()))) or "none")
       .value_counts().to_dict())

# --- Q2: does anything straddle the official split? -------------------
sp = d["official_split"].astype(str).str.lower().str.strip()
d["_split"] = np.where(sp.str.contains("test"), "test", "train")
print("\n" + "=" * 66)
print("Q2  does anything straddle the official train/test line?")
print("=" * 66)
for key in ["mammo", "breast", "lesion_key", "patient_id"]:
    n = int((d.groupby(key)["_split"].nunique() > 1).sum())
    print("  %-12s straddling : %d   %s" % (key, n, "OK" if n == 0 else "<<< PROBLEM"))

# --- Q3: is any "crop" actually a binary mask? ------------------------
print("\n" + "=" * 66)
print("Q3  crop sanity — is any 'crop' actually a mask?")
print("=" * 66)
rows = []
for p in d["img"].astype(str):
    im = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if im is None:
        rows.append((p, -1, -1, -1, -1.0))
    else:
        rows.append((p, im.shape[0], im.shape[1], int(np.unique(im).size),
                     float(((im == 0) | (im == 255)).mean())))
c = pd.DataFrame(rows, columns=["img", "h", "w", "nuniq", "extreme"])
good = c[c.h > 0]
print("  unreadable files          : %d" % int((c.h < 0).sum()))
if len(good):
    lm = good[(good.nuniq <= 4) | (good.extreme > 0.95)]
    print("  crops that look like masks: %d  %s"
          % (len(lm), "OK" if len(lm) == 0 else "<<< INSPECT"))
    if len(lm):
        print(lm.assign(img=lm.img.map(os.path.basename)).head(10).to_string(index=False))
    print("  intensity levels per crop : median %d   (real crop = many, mask = 2)"
          % int(good.nuniq.median()))
    print("  crop size   h %d-%d   w %d-%d"
          % (good.h.min(), good.h.max(), good.w.min(), good.w.max()))
else:
    print("  <<< NO readable images at all — the 'img' paths are wrong")
    print("      sample path:", d['img'].iloc[0])

# --- Q4: any missing abnormalities? -----------------------------------
print("\n" + "=" * 66)
print("Q4  abnormality numbering — any gaps?")
print("=" * 66)
a = d.dropna(subset=["abn"]).copy()
if len(a):
    a["abn"] = pd.to_numeric(a["abn"], errors="coerce")
    a = a.dropna(subset=["abn"])
    g = a.groupby("mammo")["abn"].agg(mx="max", nu="nunique")
    gaps = g[g.mx != g.nu]
    print("  mammograms whose ids are not 1..N : %d  %s"
          % (len(gaps), "OK — no missing ROIs" if len(gaps) == 0 else "<<< ROIs absent"))
    if len(gaps): print(gaps.head(10).to_string())
else:
    print("  no abnormality id in filenames — cannot check")

sample filenames:
    3f87f40c5b125dcdee76d4d579f26b59_img.png
    93ece87e3fe98bd44f544b48f211a02f_img.png
    e6f2f4c7eda2376602e48cd1fbb109be_img.png
    4764a2b76b326c83c8fa12fdf7b2bf7b_img.png
    431825da8e5a6298c05fb42a44812cf9_img.png

parsed  side 0/1696 | view 0/1696 | abnormality-id 0/1696
  UNPARSED examples: ['3f87f40c5b125dcdee76d4d579f26b59_img.png', '93ece87e3fe98bd44f544b48f211a02f_img.png', 'e6f2f4c7eda2376602e48cd1fbb109be_img.png', '4764a2b76b326c83c8fa12fdf7b2bf7b_img.png', '431825da8e5a6298c05fb42a44812cf9_img.png']
  -> the regex does not match these; paste them to me

Q1  ROIs per full mammogram
  distinct mammograms   : 1
  ROIs per mammogram    : {1696: 1}
  mammograms with >1 ROI: 1 (100.0%)

  lesions per breast    : {1005: 1}
  lesions per patient   : {1: 821, 2: 44, 3: 21, 4: 3, 5: 1, 7: 1, 9: 1}
  views per lesion      : {'none': 1005}

Q2  does anything straddle the official train/test line?
  mammo        straddling : 1   <<< PROBLEM
  breast       stra

**`SEG` cell 1** — ══════════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 11 — DOES CLASSIFIER PERFORMANCE DEPEND ON MASK QUALITY?
#   Stratifies the 378 official test regions by their ACTUAL segmentation
#   Dice and reports AUC / accuracy inside each stratum.
#   Uses real segmentation failures, not synthetic degradation.
#   No GPU, no inference. ~30 s.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]
FLOOR=0.90; GRID=np.round(np.arange(0.01,0.995,0.005),3)
MIN_N,MIN_POS,PASSES=20,3,15

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
d["y"]=d.label.astype(int)
d["a"]=pd.to_numeric(d.assessment,errors="coerce").fillna(4).astype(int).clip(0,5)
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")

# ── PART 1 : find the per-region Dice column ──────────────────────────────
print("="*82); print("PART 1 — PER-REGION MASK QUALITY"); print("="*82)
dc=[c for c in d.columns if "dice" in c.lower()]
print("  dice-like columns in unified_folds_mass.csv: %s" % dc)
assert dc, "no per-region Dice column found — was PHASE 2 run and the csv written back?"
DC=dc[0] if "oof_dice_official" not in dc else "oof_dice_official"
d["dice"]=pd.to_numeric(d[DC],errors="coerce")
print("  using column: %s" % DC)
print("  coverage: %d/%d rows have a Dice value" % (d.dice.notna().sum(),len(d)))

def load(fn):
    m=pd.read_csv(os.path.join(D,fn)); m["_k"]=m["img"].map(stem)
    return m[m["_k"].isin(set(d._k))].drop_duplicates("_k").set_index("_k")["prob"]
PTR=load("cv_mass_twostream_officialtrain_oof.csv")
PTE=load("cv_mass_twostream_officialsplit_oof.csv")
B=d.set_index("_k")

te=d[d.sp.eq("test") & d._k.isin(PTE.index) & d.dice.notna()].copy()
te["p"]=PTE.loc[te._k].values
tr=d[d.sp.eq("train") & d._k.isin(PTR.index)].copy(); tr["p"]=PTR.loc[tr._k].values
print("  test regions with both a mask Dice and a prediction: %d" % len(te))
q=te.dice.describe(percentiles=[.05,.10,.25,.5,.75,.90])
print("  test mask Dice: min %.3f | 5%% %.3f | 25%% %.3f | median %.3f | 75%% %.3f | max %.3f | mean %.4f"
      % (q["min"],q["5%"],q["25%"],q["50%"],q["75%"],q["max"],te.dice.mean()))

# ── PART 2 : is segmentation worse on malignant lesions? ──────────────────
print("\n"+"="*82); print("PART 2 — IS MASK QUALITY RELATED TO THE LABEL?"); print("="*82)
print("  mean Dice  malignant %.4f (n=%d)   benign %.4f (n=%d)   difference %+.4f"
      % (te.dice[te.y==1].mean(),int((te.y==1).sum()),
         te.dice[te.y==0].mean(),int((te.y==0).sum()),
         te.dice[te.y==1].mean()-te.dice[te.y==0].mean()))
print("  (a large negative difference would mean segmentation fails more on cancers,")
print("   which would make low-Dice cases systematically harder, not just noisier)")

# ── threshold machinery ───────────────────────────────────────────────────
def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_global(y,p,fl):
    P,N=int(y.sum()),len(y); tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N,tp/max(P,1); ok=se>=fl-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])
def _asc(y,p,a,init,fl):
    N,P=len(y),int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),
                 np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    TP=int(((yh==1)&(y==1)).sum()); TN=int(((yh==0)&(y==0)).sum())
    if TP/max(P,1)<fl-1e-12: return tau,-1.0
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bTP=TP-int((cur&(y[i]==1)).sum()); bTN=TN-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bTP+tg+bTN+ng)/N; se=(bTP+tg)/max(P,1); fe=se>=fl-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j]>(TP+TN)/N+1e-12:
                tau[g]=float(GRID[j]); TP,TN=int(bTP+tg[j]),int(bTN+ng[j]); mv=True
        if not mv: break
    return tau,(TP+TN)/N
def fit_bir(y,p,a,fl):
    g0=fit_global(y,p,fl); cats=[int(c) for c in np.unique(a)]
    st=[{g:v for g in cats} for v in (g0,.05,.15,.25,.35,.45,.50,.55,.65,max(.01,g0-.1),min(.99,g0+.1))]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.05,.8)) for g in cats} for _ in range(6)]
    best,ba=None,-np.inf
    for s in st:
        t,ac=_asc(y,p,a,s,fl)
        if ac>ba+1e-12: best,ba=t,ac
    return (best if best is not None and ba>=0 else {g:g0 for g in cats}), g0
ap=lambda p,a,t,g0:(p>=np.array([t.get(int(g),g0) for g in a])).astype(int)

# thresholds frozen from TRAIN, exactly as in the main result
TAU,G0=fit_bir(tr.y.values.astype(int),tr.p.values,tr.a.values.astype(int),FLOOR)
print("\n  frozen thresholds from TRAIN: %s" % {int(k):round(v,3) for k,v in sorted(TAU.items())})

def block(t,name,dcol="dice"):
    y=t.y.values.astype(int); p=t.p.values; a=t.a.values.astype(int)
    if len(np.unique(y))<2: return None
    yh=ap(p,a,TAU,G0)
    tp=int(((yh==1)&(y==1)).sum()); fn=int(((yh==0)&(y==1)).sum())
    tn=int(((yh==0)&(y==0)).sum()); fp=int(((yh==1)&(y==0)).sum())
    return dict(n=len(y),mal=int(y.sum()),dice=t[dcol].mean(),
                auc=roc_auc_score(y,p),acc=(tp+tn)/len(y),
                sens=tp/max(tp+fn,1),spec=tn/max(tn+fp,1))

def report(t,bins,labels,name,dcol="dice"):
    print("\n"+"="*82); print("%s — stratified by ACTUAL mask Dice" % name); print("="*82)
    print("  %-16s %-5s %-6s %-8s %-8s %-8s %-6s %s"
          % ("stratum","n","malig","meanDice","AUC","acc","sens","spec"))
    t=t.copy(); t["bin"]=pd.cut(t[dcol],bins=bins,labels=labels,include_lowest=True)
    for lb in labels:
        s=t[t.bin==lb]
        if len(s)<8: print("  %-16s %-5d  (too few to evaluate)" % (lb,len(s))); continue
        r=block(s,lb,dcol)
        if r is None: print("  %-16s %-5d  (single class)" % (lb,len(s))); continue
        print("  %-16s %-5d %-6d %-8.4f %-8.4f %-8.4f %-6.3f %.3f"
              % (lb,r["n"],r["mal"],r["dice"],r["auc"],r["acc"],r["sens"],r["spec"]))
    r=block(t,"ALL",dcol)
    print("  %-16s %-5d %-6d %-8.4f %-8.4f %-8.4f %-6.3f %.3f"
          % ("ALL",r["n"],r["mal"],r["dice"],r["auc"],r["acc"],r["sens"],r["spec"]))

# ── PART 3 : ROI level ────────────────────────────────────────────────────
report(te,[0,.70,.80,.90,1.01],["Dice <0.70","0.70-0.80","0.80-0.90","0.90+"],
       "PART 3 — ROI LEVEL (n=%d)" % len(te))
qs=te.dice.quantile([0,.25,.5,.75,1.0]).values
report(te,list(np.unique(qs)),["Q1 worst","Q2","Q3","Q4 best"][:len(np.unique(qs))-1],
       "PART 3b — ROI LEVEL, Dice quartiles")

# ── PART 4 : lesion level (mean Dice of the lesion's regions) ─────────────
les=(te.groupby("lesion_key")
       .agg(p=("p","mean"),y=("y","max"),a=("a","max"),dice=("dice","mean")).reset_index())
report(les,[0,.70,.80,.90,1.01],["Dice <0.70","0.70-0.80","0.80-0.90","0.90+"],
       "PART 4 — LESION LEVEL (n=%d)" % len(les))

# ── PART 5 : does poor segmentation predict classifier error? ─────────────
print("\n"+"="*82); print("PART 5 — CORRELATION BETWEEN MASK QUALITY AND ERROR"); print("="*82)
y=te.y.values.astype(int); err=np.abs(te.p.values-y)
c=np.corrcoef(te.dice.values,err)[0,1]
yh=ap(te.p.values,te.a.values.astype(int),TAU,G0); wrong=(yh!=y)
print("  correlation( mask Dice , |prob - label| )        %+.4f" % c)
print("  mean Dice on CORRECT predictions   %.4f  (n=%d)" % (te.dice[~wrong].mean(),int((~wrong).sum())))
print("  mean Dice on WRONG   predictions   %.4f  (n=%d)" % (te.dice[wrong].mean(),int(wrong.sum())))
print("  difference %+.4f" % (te.dice[~wrong].mean()-te.dice[wrong].mean()))
print("\n  a correlation near zero and a small difference mean classifier errors are NOT")
print("  driven by segmentation failure — which is the answer the reviewer is asking for.")

PART 1 — PER-REGION MASK QUALITY
  dice-like columns in unified_folds_mass.csv: ['oof_dice', 'oof_dice_tta', 'oof_dice_official']
  using column: oof_dice_official
  coverage: 1696/1696 rows have a Dice value
  test regions with both a mask Dice and a prediction: 378
  test mask Dice: min 0.413 | 5% 0.784 | 25% 0.888 | median 0.925 | 75% 0.942 | max 0.978 | mean 0.9065

PART 2 — IS MASK QUALITY RELATED TO THE LABEL?
  mean Dice  malignant 0.9050 (n=147)   benign 0.9075 (n=231)   difference -0.0025
  (a large negative difference would mean segmentation fails more on cancers,
   which would make low-Dice cases systematically harder, not just noisier)

  frozen thresholds from TRAIN: {0: 0.515, 1: 0.44, 2: 0.44, 3: 0.495, 4: 0.455, 5: 0.01}

PART 3 — ROI LEVEL (n=378) — stratified by ACTUAL mask Dice
  stratum          n     malig  meanDice AUC      acc      sens   spec
  Dice <0.70       7      (too few to evaluate)
  0.70-0.80        15    7      0.7614   0.8393   0.8000   0.714  0.875


**`SEG` cell 2** — ══════════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 12 — CONFIDENCE INTERVALS ON THE STRATIFIED ANALYSIS
#   Bootstrap CI per Dice quartile, exact binomial CI on the sub-0.80 tail,
#   and a CI on the Dice-vs-error correlation.  ~20 s.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import beta, pearsonr
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]
RNG=np.random.default_rng(7); NB=4000
TAU={0:.515,1:.44,2:.44,3:.495,4:.455,5:.01}; G0=.44

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
d["y"]=d.label.astype(int)
d["a"]=pd.to_numeric(d.assessment,errors="coerce").fillna(4).astype(int).clip(0,5)
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")
d["dice"]=pd.to_numeric(d["oof_dice_official"],errors="coerce")
m=pd.read_csv(os.path.join(D,"cv_mass_twostream_officialsplit_oof.csv")); m["_k"]=m["img"].map(stem)
t=d[d.sp.eq("test")].merge(m[["_k","prob"]].drop_duplicates("_k"),on="_k").dropna(subset=["dice"])
print("test regions: %d" % len(t))

def boot_auc(y,p,nb=NB):
    o=[]
    for _ in range(nb):
        i=RNG.integers(0,len(y),len(y))
        if len(np.unique(y[i]))<2: continue
        o.append(roc_auc_score(y[i],p[i]))
    return np.percentile(o,2.5),np.percentile(o,97.5)
def wilson(k,n):
    if n==0: return (np.nan,np.nan)
    lo=beta.ppf(.025,k,n-k+1) if k>0 else 0.0
    hi=beta.ppf(.975,k+1,n-k) if k<n else 1.0
    return lo,hi
pred=lambda p,a:(p>=np.array([TAU.get(int(g),G0) for g in a])).astype(int)

print("\n"+"="*84); print("PER-QUARTILE CONFIDENCE INTERVALS"); print("="*84)
t=t.copy(); t["q"]=pd.qcut(t.dice,4,labels=["Q1 worst","Q2","Q3","Q4 best"])
print("  %-10s %-5s %-13s %-22s %s" % ("stratum","n","Dice range","AUC [95% CI]","accuracy [95% CI]"))
for lb in ["Q1 worst","Q2","Q3","Q4 best"]:
    s=t[t.q==lb]; y=s.y.values.astype(int); p=s.prob.values
    lo,hi=boot_auc(y,p); k=int((pred(p,s.a.values)==y).sum()); n=len(s)
    al,ah=wilson(k,n)
    print("  %-10s %-5d %.3f-%.3f  %.4f [%.3f-%.3f]    %.3f [%.3f-%.3f]"
          % (lb,n,s.dice.min(),s.dice.max(),roc_auc_score(y,p),lo,hi,k/n,al,ah))
y=t.y.values.astype(int); p=t.prob.values
lo,hi=boot_auc(y,p); k=int((pred(p,t.a.values)==y).sum())
al,ah=wilson(k,len(t))
print("  %-10s %-5d %.3f-%.3f  %.4f [%.3f-%.3f]    %.3f [%.3f-%.3f]"
      % ("ALL",len(t),t.dice.min(),t.dice.max(),roc_auc_score(y,p),lo,hi,k/len(t),al,ah))

print("\n"+"="*84); print("THE SUB-0.80 TAIL, POOLED"); print("="*84)
for lab,sel in (("Dice < 0.80",t.dice<0.80),("Dice < 0.70",t.dice<0.70),
                ("Dice >= 0.80",t.dice>=0.80)):
    s=t[sel]; n=len(s)
    if n==0: continue
    yy=s.y.values.astype(int); k=int((pred(s.prob.values,s.a.values)==yy).sum())
    al,ah=wilson(k,n)
    au=roc_auc_score(yy,s.prob.values) if len(np.unique(yy))>1 else float("nan")
    print("  %-14s n=%-4d malig %-3d  meanDice %.4f  acc %.3f [%.3f-%.3f]  AUC %.4f"
          % (lab,n,int(yy.sum()),s.dice.mean(),k/n,al,ah,au))
print("  (a wide interval here IS the answer: it states the limit of what this cohort can show)")

print("\n"+"="*84); print("CORRELATION, FULL SAMPLE"); print("="*84)
err=np.abs(p-y); r,pv=pearsonr(t.dice.values,err)
z=np.arctanh(r); se=1/np.sqrt(len(t)-3)
print("  corr(Dice, |prob-label|)  r = %+.4f   95%% CI [%+.4f, %+.4f]   p = %.3f"
      % (r,np.tanh(z-1.96*se),np.tanh(z+1.96*se),pv))
print("  n = %d — this is the properly powered statistic, unlike the quartile split" % len(t))

test regions: 378

PER-QUARTILE CONFIDENCE INTERVALS
  stratum    n     Dice range    AUC [95% CI]           accuracy [95% CI]
  Q1 worst   95    0.413-0.888  0.8672 [0.788-0.935]    0.811 [0.717-0.884]
  Q2         94    0.888-0.925  0.8903 [0.817-0.948]    0.809 [0.714-0.882]
  Q3         94    0.926-0.942  0.8717 [0.785-0.947]    0.819 [0.726-0.891]
  Q4 best    95    0.942-0.978  0.8855 [0.805-0.951]    0.800 [0.705-0.875]
  ALL        378   0.413-0.978  0.8769 [0.839-0.911]    0.810 [0.766-0.848]

THE SUB-0.80 TAIL, POOLED
  Dice < 0.80    n=22   malig 10   meanDice 0.7095  acc 0.864 [0.651-0.971]  AUC 0.8917
  Dice < 0.70    n=7    malig 3    meanDice 0.5983  acc 1.000 [0.590-1.000]  AUC 1.0000
  Dice >= 0.80   n=356  malig 137  meanDice 0.9187  acc 0.806 [0.761-0.846]  AUC 0.8818
  (a wide interval here IS the answer: it states the limit of what this cohort can show)

CORRELATION, FULL SAMPLE
  corr(Dice, |prob-label|)  r = +0.0062   95% CI [-0.0947, +0.1070]   p = 0.904
  n = 3

**`SEG` cell 5** — ══════════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [5]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 15 — IS THE OFFICIAL TEST SET EASIER THAN THE TRAINING PARTITION?
#   PART 1  case-mix comparison, official train vs official test
#   PART 2  statistical tests on each difference
#   PART 3  the clean test: same CV models, AUC on official-test patients
#           vs official-train patients
#   No GPU. ~20 s.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency
from sklearn.metrics import roc_auc_score
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
for c in ("density","subtlety"):
    if c not in d.columns:
        fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
        d=d.merge(fx[["_k",c]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d.label.astype(int)
d["a"]=pd.to_numeric(d.assessment,errors="coerce").fillna(4).astype(int).clip(0,5)
d["ds"]=pd.to_numeric(d.density,errors="coerce")
d["sb"]=pd.to_numeric(d.subtlety,errors="coerce")
d["dice"]=pd.to_numeric(d.get("oof_dice_official"),errors="coerce")
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")
TR,TE=d[d.sp.eq("train")],d[d.sp.eq("test")]

print("="*84); print("PART 1 — CASE MIX: OFFICIAL TRAIN vs OFFICIAL TEST"); print("="*84)
print("  %-26s %-14s %-14s %s" % ("variable","train","test","difference"))
print("  %-26s %-14s %-14s" % ("n regions",len(TR),len(TE)))
print("  %-26s %-14.1f %-14.1f %+.1f pts" % ("malignant %",100*TR.y.mean(),100*TE.y.mean(),
                                             100*(TE.y.mean()-TR.y.mean())))
for lab,col in (("mean subtlety (1-5)","sb"),("mean density (1-4)","ds"),
                ("mean mask Dice","dice")):
    a,b=TR[col].dropna(),TE[col].dropna()
    if len(a)==0 or len(b)==0: continue
    print("  %-26s %-14.3f %-14.3f %+.3f" % (lab,a.mean(),b.mean(),b.mean()-a.mean()))

print("\n  BI-RADS distribution (%% of partition)")
print("  %-10s %-10s %-10s %s" % ("BI-RADS","train","test","difference"))
for g in sorted(set(d.a)):
    pa,pb=100*(TR.a==g).mean(),100*(TE.a==g).mean()
    print("  %-10d %-10.1f %-10.1f %+.1f" % (g,pa,pb,pb-pa))

print("\n"+"="*84); print("PART 2 — ARE THE DIFFERENCES SIGNIFICANT?"); print("="*84)
for lab,col in (("subtlety","sb"),("density","ds"),("mask Dice","dice")):
    a,b=TR[col].dropna(),TE[col].dropna()
    if len(a)<10 or len(b)<10: continue
    u,p=mannwhitneyu(a,b,alternative="two-sided")
    print("  %-14s Mann-Whitney p = %.4f   %s" % (lab,p,"DIFFERENT" if p<0.05 else "no difference"))
ct=pd.crosstab(d.sp,d.a)
chi,p,_,_=chi2_contingency(ct)
print("  %-14s chi-square   p = %.4f   %s" % ("BI-RADS",p,"DIFFERENT" if p<0.05 else "no difference"))
ct2=pd.crosstab(d.sp,d.y); chi2_,p2,_,_=chi2_contingency(ct2)
print("  %-14s chi-square   p = %.4f   %s" % ("malignancy",p2,"DIFFERENT" if p2<0.05 else "no difference"))

print("\n"+"="*84); print("PART 3 — THE CLEAN TEST: SAME MODELS, BOTH GROUPS OF PATIENTS"); print("="*84)
m=pd.read_csv(os.path.join(D,"cv_mass_twostream_oof.csv")); m["_k"]=m["img"].map(stem)
cv=d.merge(m[["_k","prob"]].drop_duplicates("_k"),on="_k")
print("  Using the SAME cross-validation models, scored out-of-fold:")
for nm,s in (("official-TRAIN patients",cv[cv.sp.eq("train")]),
             ("official-TEST  patients",cv[cv.sp.eq("test")])):
    print("    %-26s n=%4d  malig %4.1f%%  AUC %.4f"
          % (nm,len(s),100*s.y.mean(),roc_auc_score(s.y,s.prob)))
a_tr=roc_auc_score(cv[cv.sp.eq("train")].y,cv[cv.sp.eq("train")].prob)
a_te=roc_auc_score(cv[cv.sp.eq("test")].y, cv[cv.sp.eq("test")].prob)
print("    difference (test minus train) %+.4f" % (a_te-a_tr))
print("\n  If the official-TEST patients score HIGHER under identical models, the official")
print("  test set is genuinely an easier draw and part of the official-vs-CV gap is case mix.")
print("  If they score the same or lower, the official test set is not anomalously easy.")

print("\n  per-fold CV AUC (is the official test AUC of 0.8769 unusual?)")
for k in sorted(cv.fold.dropna().unique()):
    s=cv[cv.fold==k]
    print("    fold %d  n=%4d  AUC %.4f" % (int(k),len(s),roc_auc_score(s.y,s.prob)))

PART 1 — CASE MIX: OFFICIAL TRAIN vs OFFICIAL TEST
  variable                   train          test           difference
  n regions                  1318           378           
  malignant %                48.3           38.9           -9.4 pts
  mean subtlety (1-5)        3.966          3.786          -0.180
  mean density (1-4)         2.203          2.397          +0.193
  mean mask Dice             0.895          0.906          +0.012

  BI-RADS distribution (%% of partition)
  BI-RADS    train      test       difference
  0          9.8        8.7        -1.1
  1          0.1        0.5        +0.5
  2          5.8        3.7        -2.1
  3          21.2       22.5       +1.3
  4          40.4       44.7       +4.3
  5          22.7       19.8       -2.8

PART 2 — ARE THE DIFFERENCES SIGNIFICANT?
  subtlety       Mann-Whitney p = 0.0084   DIFFERENT
  density        Mann-Whitney p = 0.0002   DIFFERENT
  mask Dice      Mann-Whitney p = 0.0003   DIFFERENT
  BI-RADS        chi-squ

**`SEG` cell 16** — ══════════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 18 — ARE THE CROPS LESION-CENTRED?
#   If the mask fills a large, central fraction of the crop, the crop was
#   generated from the lesion annotation and the lesion is already located.
# ══════════════════════════════════════════════════════════════════════════
import os, pandas as pd, numpy as np
D=r"/root/autodl-tmp/CBIS"

f=os.path.join(D,"mask_geometry_audit.csv")
if os.path.exists(f):
    g=pd.read_csv(f)
    print("="*76); print("MASK GEOMETRY INSIDE THE CROP"); print("="*76)
    print("  rows: %d   columns: %s\n" % (len(g),list(g.columns)))
    for c in ("frac","bbox_frac","cx_off"):
        if c in g.columns:
            v=pd.to_numeric(g[c],errors="coerce").dropna()
            print("  %-12s mean %.4f | median %.4f | 5%% %.4f | 95%% %.4f"
                  % (c,v.mean(),v.median(),v.quantile(.05),v.quantile(.95)))
    print("\n  frac      = fraction of the crop occupied by the mask")
    print("  bbox_frac = fraction occupied by the mask's bounding box")
    print("  cx_off    = how far the mask centroid sits from the crop centre")
else:
    print("mask_geometry_audit.csv not found")

for f2 in ("crop_coverage_check.csv","crop_edge_check.csv"):
    p=os.path.join(D,f2)
    if os.path.exists(p):
        t=pd.read_csv(p)
        print("\n"+"="*76); print(f2); print("="*76)
        print("  columns: %s" % list(t.columns))
        print(t.describe().T.to_string())

print("\n"+"="*76); print("HOW TO READ IT"); print("="*76)
print("  mask fills a LARGE fraction (say >15-20%) and sits NEAR THE CENTRE")
print("     -> the crop was made from the lesion annotation.")
print("        Your network delineates a boundary; it does not detect the lesion.")
print("  mask is SMALL and OFF-CENTRE")
print("     -> the crop is loosely defined and the network is doing real localisation.")

MASK GEOMETRY INSIDE THE CROP
  rows: 3562   columns: ['img', 'abn', 'label', 'frac', 'bbox_frac', 'cx_off', 'cy_off', 'bright_in', 'bright_out', 'contrast']

  frac         mean 0.4030 | median 0.4053 | 5% 0.3091 | 95% 0.4920
  bbox_frac    mean 0.5991 | median 0.5922 | 5% 0.5892 | 95% 0.6553
  cx_off       mean 0.0021 | median 0.0012 | 5% -0.0419 | 95% 0.0493

  frac      = fraction of the crop occupied by the mask
  bbox_frac = fraction occupied by the mask's bounding box
  cx_off    = how far the mask centroid sits from the crop centre

crop_coverage_check.csv
  columns: ['lesion_key', 'pathology', 'area_full', 'area_crop', 'retained']
            count          mean           std           min           25%           50%          75%           max
area_full  1696.0  78978.133255  79427.752567   4020.000000  36535.500000  57087.500000  90774.75000  1.256482e+06
area_crop  1696.0  60132.683962  10311.112076  17007.000000  53943.750000  60379.500000  65632.75000  1.558230e+05
retaine

## F · Other


**`BCF` cell 331** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [11]:
# ══════════════════════════════════════════════════════════════════════
# NOVELTY-2 TEST: segmentation-guided attention — FULL comparison
#   PLAIN vs GUIDED, each evaluated on:
#     (a) full data (all BI-RADS, honest)
#     (b) BI-RADS-4-excluded subset (Shia protocol)
#     (c) BI-RADS 4 alone (the hard core)
#   => complete 2x2 table so you see the novelty's effect everywhere
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, balanced_accuracy_score
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
S=512; BATCH=12; MULT=6; EPOCHS=18; LR_HEAD=1e-3; LR_FT=1e-5; FREEZE=3; GAMMA=2.0; NFOLD=5
torch.backends.cudnn.benchmark=True; torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
MEAN=np.array([0.485,0.456,0.406],np.float32); STD=np.array([0.229,0.224,0.225],np.float32)

sub=pd.read_csv(os.path.join(D,"cbis_calc_fixed.csv"))
sub["label"]=sub["label"].astype(int)
sub["assessment"]=pd.to_numeric(sub["assessment"],errors="coerce")
sub=sub.dropna(subset=["img","label"]).reset_index(drop=True)
print("n="+str(len(sub))+" | BI-RADS4 "+format(100*(sub.assessment==4).mean(),".0f")+"% | malignant "+format(100*sub.label.mean(),".1f")+"%\n")

CACHE={}
def build_cache(df):
    t0=time.time()
    for _,r in df.iterrows():
        k=r["img"]
        if k in CACHE: continue
        img=cv2.imread(k,cv2.IMREAD_GRAYSCALE)
        img=cv2.resize(img if img is not None else np.zeros((S,S),np.uint8),(S,S))
        mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        mask=(cv2.resize(mk,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.float32) if mk is not None else np.zeros((S,S),np.float32)
        CACHE[k]=(_clahe.apply(img),mask)
    print("  cached "+str(len(CACHE))+" in "+format(time.time()-t0,".1f")+"s")

class DS(Dataset):
    def __init__(s,df,aug,mult=1,tta=0):
        s.keys=df["img"].values; s.lab=df["label"].astype(int).values
        s.aug=aug; s.mult=mult if aug else 1; s.tta=tta
    def __len__(s): return len(s.keys)*s.mult
    def __getitem__(s,i):
        j=i%len(s.keys); k=i//len(s.keys); img,mask=CACHE[s.keys[j]]; img=img.copy(); mask=mask.copy()
        if s.aug and k>0:
            if   k%6==1: img=np.fliplr(img); mask=np.fliplr(mask)
            elif k%6==2: img=np.flipud(img); mask=np.flipud(mask)
            elif k%6==3: img=np.rot90(img,2); mask=np.rot90(mask,2)
            elif k%6==4:
                M=cv2.getRotationMatrix2D((S/2,S/2),np.random.uniform(-15,15),np.random.uniform(.92,1.08))
                img=cv2.warpAffine(img,M,(S,S),borderMode=cv2.BORDER_REFLECT)
                mask=cv2.warpAffine(mask,M,(S,S),flags=cv2.INTER_NEAREST)
            elif k%6==5: img=np.clip(img.astype(np.float32)*np.random.uniform(.88,1.12),0,255).astype(np.uint8)
        if s.tta==1: img=np.fliplr(img); mask=np.fliplr(mask)
        elif s.tta==2: img=np.rot90(img,2); mask=np.rot90(mask,2)
        im=np.ascontiguousarray(img).astype(np.float32)/255.
        x=np.stack([im,im,im],0); x=((x.transpose(1,2,0)-MEAN)/STD).transpose(2,0,1).astype(np.float32)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(mask))[None],
                torch.tensor(int(s.lab[j])))

class GuidedNet(nn.Module):
    def __init__(s,guided=True):
        super().__init__()
        try: dn=models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception: dn=models.densenet121(weights=None)
        s.b=dn.features; s.guided=guided; s.pool=nn.AdaptiveAvgPool2d(1)
        s.head=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Dropout(0.5),nn.Linear(256,2))
    def forward(s,x,mask):
        f=F.relu(s.b(x))
        if s.guided:
            m=F.interpolate(mask,size=f.shape[2:],mode="bilinear",align_corners=False)
            w=1.0+2.0*m
            f=f*w; g=(f.sum((2,3))/(w.sum((2,3))+1e-6))
        else:
            g=s.pool(f).flatten(1)
        return s.head(g)

build_cache(sub)
y=sub.label.values; gr=sub.patient_id.values; b4=(sub.assessment==4).values

def run(guided):
    oofp=np.zeros(len(sub)); fa=[]
    for fold,(tri,tei) in enumerate(StratifiedGroupKFold(NFOLD,shuffle=True,random_state=42).split(sub,y,gr),1):
        g2=gr[tri]; uq=np.array(sorted(set(g2))); rs=np.random.RandomState(fold)
        vg=set(rs.permutation(uq)[:max(1,int(0.12*len(uq)))]); vm=np.array([g in vg for g in g2])
        tr_i=tri[~vm]; va_i=tri[vm]; assert len(set(gr[tr_i])&set(gr[tei]))==0
        tr=sub.iloc[tr_i]; va=sub.iloc[va_i]; te=sub.iloc[tei]
        n0=float((tr.label==0).sum()); n1=float((tr.label==1).sum()); al=torch.tensor([n1/(n0+n1),n0/(n0+n1)],device=DEV)
        def focal(lo,t):
            ce=F.cross_entropy(lo.float(),t,weight=al,reduction="none"); pt=torch.exp(-ce); return ((1-pt)**GAMMA*ce).mean()
        torch.manual_seed(fold); np.random.seed(fold)
        net=GuidedNet(guided).to(DEV).to(memory_format=torch.channels_last)
        for p_ in net.b.parameters(): p_.requires_grad=False
        sc=torch.amp.GradScaler(); opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=1e-3)
        tl=DataLoader(DS(tr,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
        @torch.no_grad()
        def col(df_,tta=True):
            net.eval(); reps=[0,1,2] if tta else [0]; tot=None; lb=None
            for t in reps:
                ld=DataLoader(DS(df_,False,tta=t),batch_size=20,shuffle=False,num_workers=0,pin_memory=True); ps=[];ll=[]
                for x,m,t2 in ld:
                    x=x.to(DEV,non_blocking=True).to(memory_format=torch.channels_last); m=m.to(DEV,non_blocking=True)
                    with torch.amp.autocast(device_type="cuda"): o=net(x,m)
                    ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy()); ll+=list(t2.numpy())
                ps=np.array(ps); lb=np.array(ll); tot=ps if tot is None else tot+ps
            return lb,tot/len(reps)
        best=0.;bs=None;ni=0
        for ep in range(1,EPOCHS+1):
            if ep==FREEZE+1:
                for p_ in net.b.parameters(): p_.requires_grad=True
                opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=1e-3)
            net.train()
            if ep<=FREEZE: net.b.eval()
            for x,m,t2 in tl:
                x=x.to(DEV,non_blocking=True).to(memory_format=torch.channels_last); m=m.to(DEV,non_blocking=True); t2=t2.to(DEV,non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type="cuda"): o=net(x,m); loss=focal(o,t2)
                if not torch.isfinite(loss): continue
                sc.scale(loss).backward(); sc.unscale_(opt); torch.nn.utils.clip_grad_norm_(net.parameters(),5.0)
                sc.step(opt); sc.update()
            yv,pv=col(va,tta=False); au=roc_auc_score(yv,pv) if len(set(yv))>1 else 0
            if au>best: best=au; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
            else: ni+=1
            if ni>=5: break
        net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
        yt,pt=col(te); oofp[tei]=pt; fa.append(roc_auc_score(yt,pt))
        print("    fold "+str(fold)+" AUC "+format(fa[-1],".4f"))
    return np.array(fa),oofp

def evaluate(oof,tag):
    def blk(mask,name):
        yy=y[mask]; pp=oof[mask]
        if len(set(yy))<2: print("    "+name+": one class only"); return
        best=max([(accuracy_score(yy,(pp>t).astype(int)),t) for t in np.linspace(0.05,0.95,181)])
        acc,thr=best; pred=(pp>thr).astype(int); cm=confusion_matrix(yy,pred,labels=[0,1]); tn,fp,fn,tp=cm.ravel()
        print("    "+name.ljust(26)+"AUC "+format(roc_auc_score(yy,pp),".4f")+
              "  max-acc "+format(100*acc,".1f")+"%  sens "+format(tp/max(tp+fn,1),".3f")+
              "  spec "+format(tn/max(tn+fp,1),".3f"))
    print("  ["+tag+"]")
    blk(np.ones(len(y),bool),                 "(a) FULL data (all BIRADS)")
    blk(np.isin(sub.assessment.values,[1,2,5,6]), "(b) BIRADS-4-excluded")
    blk(b4,                                    "(c) BIRADS-4 only")

print("=== PLAIN classifier (WITHOUT novelty) ===")
p_fa,p_oof=run(guided=False)
print("=== GUIDED classifier (WITH novelty) ===")
g_fa,g_oof=run(guided=True)

print("\n"+"="*72)
print("COMPLETE COMPARISON: with vs without segmentation-guidance novelty")
print("="*72)
print("PLAIN  mean AUC (full CV): "+format(p_fa.mean(),".4f")+" +/- "+format(p_fa.std(),".4f"))
print("GUIDED mean AUC (full CV): "+format(g_fa.mean(),".4f")+" +/- "+format(g_fa.std(),".4f"))
print("gain from novelty (full data): "+format(g_fa.mean()-p_fa.mean(),"+.4f")+"\n")
evaluate(p_oof,"WITHOUT novelty (PLAIN)")
print()
evaluate(g_oof,"WITH novelty (GUIDED)")
print("\n"+"="*72)
gain=g_fa.mean()-p_fa.mean()
if gain>0.010: print("VERDICT: GUIDED wins clearly (+"+format(gain,".4f")+"). Novelty 2 confirmed.")
elif gain>0.003: print("VERDICT: small gain (+"+format(gain,".4f")+"). Reportable but modest.")
else: print("VERDICT: no real gain ("+format(gain,"+.4f")+"). Honest negative — use multi-task as novelty 2 instead.")
print("="*72)
np.save(os.path.join(D,"guided_vs_plain_full.npy"),{"plain":p_oof,"guided":g_oof,"y":y,"b4":b4})

n=1866 | BI-RADS4 50% | malignant 36.0%

  cached 1866 in 9.2s
=== PLAIN classifier (WITHOUT novelty) ===
    fold 1 AUC 0.7848
    fold 2 AUC 0.7983
    fold 3 AUC 0.7515
    fold 4 AUC 0.7797
    fold 5 AUC 0.8074
=== GUIDED classifier (WITH novelty) ===
    fold 1 AUC 0.7830
    fold 2 AUC 0.8021
    fold 3 AUC 0.7531
    fold 4 AUC 0.7877
    fold 5 AUC 0.8213

COMPLETE COMPARISON: with vs without segmentation-guidance novelty
PLAIN  mean AUC (full CV): 0.7843 +/- 0.0191
GUIDED mean AUC (full CV): 0.7894 +/- 0.0225
gain from novelty (full data): +0.0051

  [WITHOUT novelty (PLAIN)]
    (a) FULL data (all BIRADS)AUC 0.7724  max-acc 70.8%  sens 0.637  spec 0.748
    (b) BIRADS-4-excluded     AUC 0.9039  max-acc 84.9%  sens 0.807  spec 0.863
    (c) BIRADS-4 only         AUC 0.6152  max-acc 60.0%  sens 0.127  spec 0.934

  [WITH novelty (GUIDED)]
    (a) FULL data (all BIRADS)AUC 0.7765  max-acc 71.2%  sens 0.475  spec 0.846
    (b) BIRADS-4-excluded     AUC 0.9156  max-acc 84.9%  sen

**`BCF` cell 371** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [7]:
# ══════════════════════════════════════════════════════════════════════
# END-TO-END: classification using U-Net PREDICTED masks (no retraining)
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["HF_HUB_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"; DEV=torch.device("cuda"); S=512
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
MEAN=np.array([0.485,0.456,0.406],np.float32); STD=np.array([0.229,0.224,0.225],np.float32)

def blk(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class Att(nn.Module):
    def __init__(s,Fg,Fx,Fi):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(Fg,Fi,1),nn.BatchNorm2d(Fi))
        s.Wx=nn.Sequential(nn.Conv2d(Fx,Fi,1),nn.BatchNorm2d(Fi))
        s.psi=nn.Sequential(nn.Conv2d(Fi,1,1),nn.BatchNorm2d(1))
    def forward(s,g,x): return x*torch.sigmoid(s.psi(F.relu(s.Wg(g)+s.Wx(x))))
class AttUNet(nn.Module):
    def __init__(s):
        super().__init__()
        s.e1=blk(1,32); s.e2=blk(32,64); s.e3=blk(64,128); s.e4=blk(128,256); s.bn=blk(256,512)
        s.u4=nn.ConvTranspose2d(512,256,2,2); s.a4=Att(256,256,128); s.d4=blk(512,256)
        s.u3=nn.ConvTranspose2d(256,128,2,2); s.a3=Att(128,128,64);  s.d3=blk(256,128)
        s.u2=nn.ConvTranspose2d(128,64,2,2);  s.a2=Att(64,64,32);    s.d2=blk(128,64)
        s.u1=nn.ConvTranspose2d(64,32,2,2);   s.a1=Att(32,32,16);    s.d1=blk(64,32)
        s.v=nn.Conv2d(32,1,1); s.adv=nn.Conv2d(32,2,1)
    def forward(s,x):
        p=F.max_pool2d
        c1=s.e1(x); c2=s.e2(p(c1,2)); c3=s.e3(p(c2,2)); c4=s.e4(p(c3,2)); b=s.bn(p(c4,2))
        d=s.u4(b); d=s.d4(torch.cat([s.a4(d,c4),d],1))
        d=s.u3(d); d=s.d3(torch.cat([s.a3(d,c3),d],1))
        d=s.u2(d); d=s.d2(torch.cat([s.a2(d,c2),d],1))
        d=s.u1(d); d=s.d1(torch.cat([s.a1(d,c1),d],1))
        a=s.adv(d); return s.v(d)+a-a.mean(1,keepdim=True)

seg=AttUNet().to(DEV)
sd=torch.load(os.path.join(D,"seg_cbis_mass.pth"),map_location="cpu")
if isinstance(sd,dict) and "state_dict" in sd: sd=sd["state_dict"]
mi,un=seg.load_state_dict(sd,strict=False)
print("SEG load -> missing "+str(len(mi))+", unexpected "+str(len(un)))
if mi: print("   missing:",mi[:4])
if un: print("   unexpected:",un[:4])
seg.eval()

sub=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv"))
sub["label"]=sub["label"].astype(int)
sub["assessment"]=pd.to_numeric(sub["assessment"],errors="coerce")
sub=sub.dropna(subset=["img","label"]).reset_index(drop=True)

CACHE={}; PRED={}; dice=[]
print("\ngenerating predicted masks..."); t0=time.time()
with torch.no_grad():
    for i0 in range(0,len(sub),16):
        ch=sub.iloc[i0:i0+16]; ims=[];gts=[];ks=[]
        for _,r in ch.iterrows():
            im=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
            im=cv2.resize(im if im is not None else np.zeros((S,S),np.uint8),(S,S))
            g=_clahe.apply(im); mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
            gt=(cv2.resize(mk,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8) if mk is not None else np.zeros((S,S),np.uint8)
            CACHE[r["img"]]=(g,gt.astype(np.float32))
            ims.append(g.astype(np.float32)/255.); gts.append(gt); ks.append(r["img"])
        x=torch.from_numpy(np.stack(ims)[:,None]).to(DEV)
        with torch.amp.autocast(device_type="cuda"): lo=seg(x)
        pr=torch.softmax(lo.float(),1)[:,1].cpu().numpy()
        for k,p_,gt in zip(ks,pr,gts):
            m=(p_>0.5).astype(np.uint8); PRED[k]=m
            t=m.sum()+gt.sum(); dice.append(2*(m&gt).sum()/t if t>0 else 1.0)
print("  "+format(time.time()-t0,".0f")+"s   Dice(pred vs GT) mean "+format(np.mean(dice),".4f")+
      "   coverage median "+format(100*np.median([m.mean() for m in PRED.values()]),".1f")+"% (GT ~23%)")

META={"subtlety":5,"mass_shape":7,"mass_margins":5}
class Net(nn.Module):
    def __init__(s,meta):
        super().__init__()
        s.b=models.densenet121(weights=None).features
        s.head=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Dropout(0.5),nn.Linear(256,2))
        s.keys=sorted(meta.keys())
        s.aux=nn.ModuleList([nn.Sequential(nn.Linear(1024,128),nn.ReLU(),nn.Dropout(0.3),
                                           nn.Linear(128,meta[k])) for k in s.keys])
    def forward(s,x,mask):
        f=F.relu(s.b(x))
        m=F.interpolate(mask,size=f.shape[2:],mode="bilinear",align_corners=False)
        w=1.0+2.0*m; f=f*w; g=(f.sum((2,3))/(w.sum((2,3))+1e-6))
        return s.head(g)

class DS(Dataset):
    def __init__(s,idx,use_pred,tta=0): s.idx=np.array(idx); s.up=use_pred; s.tta=tta
    def __len__(s): return len(s.idx)
    def __getitem__(s,i):
        j=s.idx[i]; r=sub.iloc[j]; img,gt=CACHE[r["img"]]
        mask=PRED[r["img"]].astype(np.float32) if s.up else gt
        img=img.copy(); mask=mask.copy()
        if s.tta==1: img=np.fliplr(img); mask=np.fliplr(mask)
        elif s.tta==2: img=np.flipud(img); mask=np.flipud(mask)
        elif s.tta==3: img=np.rot90(img,2); mask=np.rot90(mask,2)
        im=np.ascontiguousarray(img).astype(np.float32)/255.
        x=np.stack([im,im,im],0); x=((x.transpose(1,2,0)-MEAN)/STD).transpose(2,0,1).astype(np.float32)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(mask))[None], torch.tensor(int(r["label"])))

y=sub.label.values; gr=sub.patient_id.values
strat=sub["label"].astype(str)+"_"+sub.assessment.isin([3,4]).astype(int).astype(str)
folds=list(StratifiedGroupKFold(5,shuffle=True,random_state=42).split(sub,strat,gr))
oof={"gt":np.zeros(len(sub)),"pred":np.zeros(len(sub))}

for fold,(tri,tei) in enumerate(folds,1):
    ck=os.path.join(D,"cls_mass_fixed_fold"+str(fold)+".pth")
    if not os.path.exists(ck): print("MISSING "+ck); break
    net=Net(META).to(DEV)
    c=torch.load(ck,map_location="cpu")
    if isinstance(c,dict) and "state_dict" in c: c=c["state_dict"]
    m2,u2=net.load_state_dict(c,strict=False)
    if fold==1: print("CLS load -> missing "+str(len(m2))+", unexpected "+str(len(u2)))
    net.eval()
    for mode,up in [("gt",False),("pred",True)]:
        tot=None
        with torch.no_grad():
            for t in [0,1,2,3]:
                ld=DataLoader(DS(tei,up,t),batch_size=20,shuffle=False,num_workers=0)
                ps=[]
                for x,m,_ in ld:
                    x=x.to(DEV); m=m.to(DEV)
                    with torch.amp.autocast(device_type="cuda"): o=net(x,m)
                    ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
                ps=np.array(ps); tot=ps if tot is None else tot+ps
        oof[mode][tei]=tot/4
    print("  fold "+str(fold)+"  GT "+format(roc_auc_score(y[tei],oof["gt"][tei]),".4f")+
          "   PRED "+format(roc_auc_score(y[tei],oof["pred"][tei]),".4f"))

sc=[c for c in ["side","left or right breast"] if c in sub.columns][0]
lc=[c for c in ["lesion","abnormality id"] if c in sub.columns][0]
sub["lesion_key"]=sub.patient_id.astype(str)+"_"+sub[sc].astype(str)+"_"+sub[lc].astype(str)
sub["prob_gt"]=oof["gt"]; sub["prob_pred"]=oof["pred"]; sub["true"]=y
sub.to_csv(os.path.join(D,"cv_mass_endtoend.csv"),index=False)

print("\n"+"="*70); print("END-TO-END RESULT"); print("="*70)
for mode,tag in [("gt","ground-truth masks"),("pred","U-Net PREDICTED masks")]:
    g=sub.groupby("lesion_key").agg(y=("true","max"),p=("prob_"+mode,"mean")).reset_index()
    thr=max([(balanced_accuracy_score(g.y,(g.p>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
    pr=(g.p>thr).astype(int); tn,fp,fn,tp=confusion_matrix(g.y,pr,labels=[0,1]).ravel()
    print("  "+tag.ljust(26)+"per-image AUC "+format(roc_auc_score(y,sub["prob_"+mode]),".4f")+
          "   per-lesion AUC "+format(roc_auc_score(g.y,g.p),".4f")+
          "  acc "+format(100*accuracy_score(g.y,pr),".1f")+"%  FP "+str(fp)+" FN "+str(fn))
print("\n  Tsochatzidis 2021 saw 0.862 -> 0.860 for this substitution")
print("="*70)

SEG load -> missing 0, unexpected 0

generating predicted masks...
  19s   Dice(pred vs GT) mean 0.0000   coverage median 0.0% (GT ~23%)
CLS load -> missing 0, unexpected 0
  fold 1  GT 0.8756   PRED 0.7807
  fold 2  GT 0.9371   PRED 0.8924
  fold 3  GT 0.9051   PRED 0.8965
  fold 4  GT 0.8953   PRED 0.8683
  fold 5  GT 0.9094   PRED 0.8712

END-TO-END RESULT
  ground-truth masks        per-image AUC 0.9026   per-lesion AUC 0.9141  acc 84.2%  FP 90 FN 69
  U-Net PREDICTED masks     per-image AUC 0.8520   per-lesion AUC 0.8653  acc 78.3%  FP 108 FN 110

  Tsochatzidis 2021 saw 0.862 -> 0.860 for this substitution


**`BCF` cell 367** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [3]:
# ══════════════════════════════════════════════════════════════════════
# MASS COMPARISON TABLE — CBIS-DDSM / INbreast, 2021-2026
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
D="/root/autodl-tmp/CBIS"

e=pd.read_csv(os.path.join(D,"mass_ensemble_lesion.csv"))
thr=max([(balanced_accuracy_score(e.y,(e.p>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
AUC=roc_auc_score(e.y,e.p); ACC=100*accuracy_score(e.y,(e.p>thr).astype(int))

W=132
print("="*W)
print("MASS — segmentation + classification on CBIS-DDSM / INbreast")
print("="*W)
print(f"{'Study':<30}{'Year':<6}{'Task':<12}{'Split protocol':<28}{'Dice':<8}{'AUC':<8}{'Acc':<8}{'Comparable?':<13}")
print("-"*W)

ROWS=[
 # study, year, task, split, dice, auc, acc, comparable, reason
 ("Tsochatzidis, CMPB","2021","seg+class","official CBIS (case-disjoint)","0.722","0.862","74.9%","YES",""),
 ("Tiryaki, BSPC","2023","seg+class","not specified","0.636","0.819","76.2%","partly",""),
 ("Ma & Peng, PESM","2024","seg+class","NOT SPECIFIED","0.925","0.932","—","unknown",
     "paywalled; split unstated. 0.932 exceeds every confirmed patient-split paper"),
 ("MSDLM, Sci Rep","2025","seg+class","random 80/10/10","0.856","0.958","97.6%","NO",
     "random split, no patient grouping; classifier trained on Wisconsin tabular data, not mass ROIs"),
 ("Saha, BMC Med Imaging","2024","seg+class","random image split","—","0.999","99.9%","NO",
     "near-perfect on ~100 INbreast images with random split - textbook leakage signature"),
 ("Salama & Aly, Alex Eng J","2021","seg+class","augment-then-split","—","0.989","98.9%","NO",
     "augmented copies split across train/test; same models score only 82.5% on CBIS"),
 ("Baccouche, Sci Rep","2022","seg+class","not specified","~0.89","0.950","95.1%","NO",
     "INbreast 99.2% indicates saturation on a tiny test set"),
 ("YOLOv5+DW-SegNet, JIIM","2025","seg only","official CBIS + INbreast ext","0.894","—","—","seg only",""),
 ("Bi-CBMSegNet, Sci Rep","2025","seg only","image-wise 80/20","0.971","—","—","seg only",
     "image-wise split inflates Dice; no classification head"),
 ("UNet++/PSPNet/TransUNet","2025","seg only","not specified","0.865","—","—","seg only",""),
 ("Jalalian, Front Biomed Tech","2025","seg only","not specified","0.927","—","—","seg only",""),
 ("HTU-Net, Comput Biol Med","2024","seg only","not specified","high","—","—","seg only",""),
]
for s,y,t,sp,dc,auc,acc,cmp_,_ in ROWS:
    print(f"{s:<30}{y:<6}{t:<12}{sp:<28}{dc:<8}{auc:<8}{acc:<8}{cmp_:<13}")
print("-"*W)
print(f"{'>> THIS WORK':<30}{'2026':<6}{'seg+class':<12}{'patient-grouped 5-fold CV':<28}"
      f"{'0.924':<8}{format(AUC,'.3f'):<8}{format(ACC,'.1f')+'%':<8}{'—':<13}")
print("="*W)

print("\nWHERE YOU STAND (comparable studies only — patient/case-disjoint splits):")
print("-"*W)
print("  SEGMENTATION")
print("    Bi-CBMSegNet   0.971   (image-wise split)")
print("    Jalalian       0.927   (split unstated)")
print("    Ma & Peng      0.925   (split unstated)")
print("    >> YOU         0.924   <- patient-grouped, top of the confirmed-split group")
print("    YOLOv5+SegNet  0.894   (official split)")
print("    UNet++         0.865")
print("    Tsochatzidis   0.722   (official split)")
print("    Tiryaki        0.636")
print()
print("  CLASSIFICATION (patient/case-disjoint only)")
print("    >> YOU         "+format(AUC,".3f")+"   <- patient-grouped 5-fold, all BI-RADS retained")
print("    Tsochatzidis   0.862   (official CBIS split)")
print("    Tiryaki        0.819   (split unclear)")
print()
print("  WHY THE HIGHER NUMBERS ARE HIGHER:")
for s,y,t,sp,dc,auc,acc,cmp_,why in ROWS:
    if why: print("    "+s+" ("+auc+"): "+why)
print("="*W)

MASS — segmentation + classification on CBIS-DDSM / INbreast
Study                         Year  Task        Split protocol              Dice    AUC     Acc     Comparable?  
------------------------------------------------------------------------------------------------------------------------------------
Tsochatzidis, CMPB            2021  seg+class   official CBIS (case-disjoint)0.722   0.862   74.9%   YES          
Tiryaki, BSPC                 2023  seg+class   not specified               0.636   0.819   76.2%   partly       
Ma & Peng, PESM               2024  seg+class   NOT SPECIFIED               0.925   0.932   —       unknown      
MSDLM, Sci Rep                2025  seg+class   random 80/10/10             0.856   0.958   97.6%   NO           
Saha, BMC Med Imaging         2024  seg+class   random image split          —       0.999   99.9%   NO           
Salama & Aly, Alex Eng J      2021  seg+class   augment-then-split          —       0.989   98.9%   NO           
Baccouc